<a href="https://colab.research.google.com/github/amzad-786githumb/SPP_GAN_Research/blob/main/22_Reproducibility_Manifest_%26_Final_Audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# NOTEBOOK 22 — REPRODUCIBILITY, MANIFEST & FINAL AUDIT
# SECTION 1 — HEADER & SCOPE
# ============================================================

# ------------------------------------------------------------
# 1. IMPORTS
# ------------------------------------------------------------

from pathlib import Path
from datetime import datetime, timezone
import os
import sys
import json
import hashlib
import random
import gc
import warnings
import platform
import importlib.util

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


# ------------------------------------------------------------
# 2. GOOGLE DRIVE
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/SPP_GAN_Research"
)

if not PROJECT_ROOT.is_dir():

    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except ImportError:
        pass

assert PROJECT_ROOT.is_dir(), (
    f"Project root not found: {PROJECT_ROOT}. "
    "Mount Google Drive and verify the project directory."
)


# ------------------------------------------------------------
# 3. NOTEBOOK IDENTITY
# ------------------------------------------------------------

NOTEBOOK_ID = "notebook_22"

NOTEBOOK_NAME = (
    "Reproducibility, Manifest & Final Audit"
)

NOTEBOOK_ROOT = (
    PROJECT_ROOT
    / "results"
    / "notebooks"
    / NOTEBOOK_ID
)

TABLES_DIR = NOTEBOOK_ROOT / "tables"
RESULTS_DIR = NOTEBOOK_ROOT / "results"
METADATA_DIR = NOTEBOOK_ROOT / "metadata"
VALIDATION_DIR = NOTEBOOK_ROOT / "validation"
MANIFESTS_DIR = NOTEBOOK_ROOT / "manifests"
REPORTS_DIR = NOTEBOOK_ROOT / "reports"

for directory in [
    NOTEBOOK_ROOT,
    TABLES_DIR,
    RESULTS_DIR,
    METADATA_DIR,
    VALIDATION_DIR,
    MANIFESTS_DIR,
    REPORTS_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ------------------------------------------------------------
# 4. RESEARCH EXPERIMENT CONFIGURATION
# ------------------------------------------------------------

MASTER_SEED = 2025

REPETITION_SEEDS = [
    3026,
    3027,
    3028,
    3029,
    3030
]

EXPECTED_REPETITIONS = 5

EXPECTED_DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us"
]

EXPECTED_MODELS = [
    "statistical_baseline",
    "tvae",
    "ctgan",
    "dp_ctgan",
    "sppgan"
]

EXPECTED_ABLATION_VARIANTS = [
    "gan_only",
    "statistical_guidance_gan",
    "gan_dp",
    "full_sppgan"
]

EXPECTED_SPLIT = {
    "train": 0.70,
    "validation": 0.15,
    "test": 0.15
}

EXPECTED_EXPERIMENTS = (
    len(EXPECTED_DATASETS)
    * EXPECTED_REPETITIONS
)

EXPECTED_ABLATION_RECORDS = (
    len(EXPECTED_DATASETS)
    * len(EXPECTED_ABLATION_VARIANTS)
    * EXPECTED_REPETITIONS
)

AUDIT_TIMESTAMP = datetime.now(
    timezone.utc
).isoformat()


# ------------------------------------------------------------
# 5. ENVIRONMENT METADATA
# ------------------------------------------------------------

ENVIRONMENT_METADATA = {
    "python_version": sys.version,
    "python_executable": sys.executable,
    "platform": platform.platform(),
    "processor": platform.processor(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "project_root": str(PROJECT_ROOT),
    "notebook_root": str(NOTEBOOK_ROOT)
}


# ------------------------------------------------------------
# 6. SCOPE METADATA
# ------------------------------------------------------------

NOTEBOOK_SCOPE = {
    "notebook_id": NOTEBOOK_ID,
    "notebook_name": NOTEBOOK_NAME,
    "project_root": str(PROJECT_ROOT),
    "notebook_root": str(NOTEBOOK_ROOT),
    "master_seed": MASTER_SEED,
    "repetition_seeds": REPETITION_SEEDS,
    "expected_repetitions": EXPECTED_REPETITIONS,
    "expected_datasets": EXPECTED_DATASETS,
    "expected_models": EXPECTED_MODELS,
    "expected_ablation_variants": EXPECTED_ABLATION_VARIANTS,
    "expected_split": EXPECTED_SPLIT,
    "expected_dataset_repetition_combinations": EXPECTED_EXPERIMENTS,
    "expected_ablation_records": EXPECTED_ABLATION_RECORDS,
    "environment": ENVIRONMENT_METADATA,
    "timestamp_utc": AUDIT_TIMESTAMP
}


# ------------------------------------------------------------
# 7. SAVE SCOPE METADATA
# ------------------------------------------------------------

SCOPE_METADATA_PATH = (
    METADATA_DIR /
    "notebook_22_scope_metadata.json"
)

with open(
    SCOPE_METADATA_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        NOTEBOOK_SCOPE,
        f,
        indent=4,
        default=str
    )


# ------------------------------------------------------------
# 8. SECTION VALIDATION
# ------------------------------------------------------------

section_01_checks = [
    {
        "check": "Project root exists",
        "passed": PROJECT_ROOT.is_dir()
    },
    {
        "check": "Notebook root exists",
        "passed": NOTEBOOK_ROOT.is_dir()
    },
    {
        "check": "All output directories exist",
        "passed": all(
            directory.is_dir()
            for directory in [
                TABLES_DIR,
                RESULTS_DIR,
                METADATA_DIR,
                VALIDATION_DIR,
                MANIFESTS_DIR,
                REPORTS_DIR
            ]
        )
    },
    {
        "check": "Five repetition seeds configured",
        "passed": (
            len(REPETITION_SEEDS)
            == EXPECTED_REPETITIONS
        )
    },
    {
        "check": "Repetition seeds match methodology",
        "passed": (
            REPETITION_SEEDS
            == [3026, 3027, 3028, 3029, 3030]
        )
    },
    {
        "check": "Data split totals 1.0",
        "passed": (
            abs(sum(EXPECTED_SPLIT.values()) - 1.0)
            < 1e-12
        )
    },
    {
        "check": "Scope metadata saved",
        "passed": SCOPE_METADATA_PATH.is_file()
    }
]

section_01_validation = pd.DataFrame(
    section_01_checks
)

SECTION_01_VALIDATION_PATH = (
    VALIDATION_DIR /
    "section_01_validation.csv"
)

section_01_validation.to_csv(
    SECTION_01_VALIDATION_PATH,
    index=False
)

section_01_passed = int(
    section_01_validation["passed"].sum()
)

section_01_failed = int(
    (~section_01_validation["passed"]).sum()
)

SECTION_01_STATUS = (
    "PASS"
    if section_01_failed == 0
    else "FAIL"
)


# ------------------------------------------------------------
# 9. EXECUTION SUMMARY
# ------------------------------------------------------------

print("=" * 75)
print(NOTEBOOK_NAME.upper())
print("=" * 75)

print("Project root:", PROJECT_ROOT)
print("Notebook root:", NOTEBOOK_ROOT)

print("Datasets:", len(EXPECTED_DATASETS))
print("Models:", len(EXPECTED_MODELS))
print("Repetitions:", EXPECTED_REPETITIONS)

print(
    "Expected dataset-repetition combinations:",
    EXPECTED_EXPERIMENTS
)

print(
    "Expected ablation records:",
    EXPECTED_ABLATION_RECORDS
)

print("Master seed:", MASTER_SEED)
print("Timestamp UTC:", AUDIT_TIMESTAMP)

print("-" * 75)

print(
    "Validation passed:",
    section_01_passed
)

print(
    "Validation failed:",
    section_01_failed
)

print("Scope metadata:", SCOPE_METADATA_PATH)

print(
    "Validation report:",
    SECTION_01_VALIDATION_PATH
)

print("SECTION 1 STATUS:", SECTION_01_STATUS)

assert SECTION_01_STATUS == "PASS", (
    "Section 1 validation failed."
)


Mounted at /content/drive
REPRODUCIBILITY, MANIFEST & FINAL AUDIT
Project root: /content/drive/MyDrive/SPP_GAN_Research
Notebook root: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_22
Datasets: 3
Models: 5
Repetitions: 5
Expected dataset-repetition combinations: 15
Expected ablation records: 60
Master seed: 2025
Timestamp UTC: 2026-11-19T05:14:38.160025+00:00
---------------------------------------------------------------------------
Validation passed: 7
Validation failed: 0
Scope metadata: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_22/metadata/notebook_22_scope_metadata.json
Validation report: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_22/validation/section_01_validation.csv
SECTION 1 STATUS: PASS


In [2]:
# ==============================================================================
# NOTEBOOK 22 — SECTION 2: LOAD PROJECT CONFIGURATION
# ==============================================================================
# PURPOSE
# -------
# Load and independently verify the authoritative configuration generated by
# Notebook 00. Preserve the original Notebook 00 artifacts without modification.
#
# SOURCE OF TRUTH
# --------------
# /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_00/
#
# NOTEBOOK 00 ARTIFACT CONTRACT
# ----------------------------
# config/config files:
#   experiment_config.json
#   dataset_registry.json
#   model_registry.json
#   evaluation_config.json
#   privacy_config.json
#   sppgan_config.json
#
# environment/environment.json
# manifest/notebook_00_manifest.json
# manifest/configuration_fingerprint.json
#
# IMPORTANT
# ---------
# Missing original artifacts are NOT reconstructed or silently replaced.
# A reconstructed configuration must never be represented as an original
# persisted Notebook 00 artifact.
# ==============================================================================

import os
import json
import hashlib
import platform
import sys
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

print("=" * 100)
print("SECTION 2 — LOAD PROJECT CONFIGURATION")
print("=" * 100)

# ==============================================================================
# 2.1 — GOOGLE DRIVE VERIFICATION
# ==============================================================================

DRIVE_ROOT = Path("/content/drive")
MYDRIVE_ROOT = DRIVE_ROOT / "MyDrive"

if not MYDRIVE_ROOT.is_dir():
    raise FileNotFoundError(
        "Google Drive MyDrive is not accessible. "
        "Mount Google Drive before running Notebook 22."
    )

print("Drive mount:", DRIVE_ROOT)
print("MyDrive:", MYDRIVE_ROOT)

# ==============================================================================
# 2.2 — CANONICAL PROJECT PATHS
# ==============================================================================

PROJECT_ROOT = MYDRIVE_ROOT / "SPP_GAN_Research"

if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(
        f"Canonical project root does not exist:\n{PROJECT_ROOT}"
    )

NOTEBOOK_RESULTS_ROOT = PROJECT_ROOT / "results" / "notebooks"

NB00_ROOT = NOTEBOOK_RESULTS_ROOT / "notebook_00"
NB00_CONFIG_DIR = NB00_ROOT / "config"
NB00_ENVIRONMENT_DIR = NB00_ROOT / "environment"
NB00_MANIFEST_DIR = NB00_ROOT / "manifest"

NB22_ROOT = NOTEBOOK_RESULTS_ROOT / "notebook_22"

NB22_METADATA_DIR = NB22_ROOT / "metadata"
NB22_VALIDATION_DIR = NB22_ROOT / "validation"
NB22_LOGS_DIR = NB22_ROOT / "logs"

for directory in [
    NB22_METADATA_DIR,
    NB22_VALIDATION_DIR,
    NB22_LOGS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("\nCanonical project root:", PROJECT_ROOT)
print("Notebook 00 root:", NB00_ROOT)
print("Notebook 22 root:", NB22_ROOT)

# ==============================================================================
# 2.3 — NOTEBOOK 00 ARTIFACT REGISTRY
# ==============================================================================
# These keys and paths reproduce Notebook 00 Section 00.14 exactly.

ARTIFACT_PATHS = {
    "config_json":
        NB00_CONFIG_DIR / "experiment_config.json",

    "dataset_registry_json":
        NB00_CONFIG_DIR / "dataset_registry.json",

    "model_registry_json":
        NB00_CONFIG_DIR / "model_registry.json",

    "evaluation_config_json":
        NB00_CONFIG_DIR / "evaluation_config.json",

    "privacy_config_json":
        NB00_CONFIG_DIR / "privacy_config.json",

    "sppgan_config_json":
        NB00_CONFIG_DIR / "sppgan_config.json",

    "environment_json":
        NB00_ENVIRONMENT_DIR / "environment.json",

    "manifest_json":
        NB00_MANIFEST_DIR / "notebook_00_manifest.json",
}

CONFIGURATION_FINGERPRINT_PATH = (
    NB00_MANIFEST_DIR / "configuration_fingerprint.json"
)

# ==============================================================================
# 2.4 — VERIFY ORIGINAL ARTIFACT AVAILABILITY
# ==============================================================================

ARTIFACT_STATUS = {}

for artifact_name, artifact_path in ARTIFACT_PATHS.items():

    exists = artifact_path.exists()
    is_file = artifact_path.is_file() if exists else False
    size_bytes = artifact_path.stat().st_size if is_file else 0

    ARTIFACT_STATUS[artifact_name] = {
        "path": str(artifact_path),
        "exists": exists,
        "is_file": is_file,
        "size_bytes": size_bytes,
        "non_empty": size_bytes > 0,
    }

ARTIFACT_STATUS["configuration_fingerprint_json"] = {
    "path": str(CONFIGURATION_FINGERPRINT_PATH),
    "exists": CONFIGURATION_FINGERPRINT_PATH.exists(),
    "is_file": CONFIGURATION_FINGERPRINT_PATH.is_file(),
    "size_bytes": (
        CONFIGURATION_FINGERPRINT_PATH.stat().st_size
        if CONFIGURATION_FINGERPRINT_PATH.is_file()
        else 0
    ),
    "non_empty": (
        CONFIGURATION_FINGERPRINT_PATH.is_file()
        and CONFIGURATION_FINGERPRINT_PATH.stat().st_size > 0
    ),
}

print("\nNOTEBOOK 00 ORIGINAL ARTIFACT STATUS")
print("-" * 100)

for name, status in ARTIFACT_STATUS.items():
    print(
        f"{'FOUND' if status['exists'] else 'MISSING':<8} "
        f"{name:<34} "
        f"{status['path']}"
    )

# ==============================================================================
# 2.5 — LOAD JSON FILES SAFELY
# ==============================================================================

def load_json_artifact(path):
    path = Path(path)

    if not path.is_file():
        raise FileNotFoundError(str(path))

    if path.stat().st_size == 0:
        raise ValueError(f"Empty JSON artifact: {path}")

    with open(path, "r", encoding="utf-8") as file:
        data = json.load(file)

    if not isinstance(data, dict):
        raise TypeError(
            f"Expected a JSON object in {path}; "
            f"received {type(data).__name__}."
        )

    return data


CONFIGURATION_OBJECTS = {}
LOAD_ERRORS = {}

for artifact_name, artifact_path in ARTIFACT_PATHS.items():

    try:
        CONFIGURATION_OBJECTS[artifact_name] = (
            load_json_artifact(artifact_path)
        )

    except Exception as exc:
        LOAD_ERRORS[artifact_name] = (
            f"{type(exc).__name__}: {exc}"
        )

# ==============================================================================
# 2.6 — EXPOSE NOTEBOOK 00 CONFIGURATION OBJECTS
# ==============================================================================

EXPERIMENT_CONFIG = CONFIGURATION_OBJECTS.get("config_json")
DATASET_REGISTRY = CONFIGURATION_OBJECTS.get("dataset_registry_json")
MODEL_REGISTRY = CONFIGURATION_OBJECTS.get("model_registry_json")
EVALUATION_CONFIG = CONFIGURATION_OBJECTS.get("evaluation_config_json")
PRIVACY_CONFIG = CONFIGURATION_OBJECTS.get("privacy_config_json")
SPPGAN_CONFIG = CONFIGURATION_OBJECTS.get("sppgan_config_json")
ENVIRONMENT_METADATA = CONFIGURATION_OBJECTS.get("environment_json")
REPRODUCIBILITY_MANIFEST = CONFIGURATION_OBJECTS.get("manifest_json")

# ==============================================================================
# 2.7 — VALIDATE MANIFEST STRUCTURE AND CONFIGURATION SNAPSHOT
# ==============================================================================

CONFIG_SNAPSHOT = None

MANIFEST_STRUCTURE_PASS = False
CONFIG_SNAPSHOT_PASS = False
CONFIGURATION_LINKAGE_PASS = False

if isinstance(REPRODUCIBILITY_MANIFEST, dict):

    required_manifest_keys = {
        "manifest_version",
        "project",
        "environment",
        "configuration",
        "artifact_paths",
    }

    MANIFEST_STRUCTURE_PASS = required_manifest_keys.issubset(
        REPRODUCIBILITY_MANIFEST.keys()
    )

    if MANIFEST_STRUCTURE_PASS:

        CONFIG_SNAPSHOT = REPRODUCIBILITY_MANIFEST.get(
            "configuration"
        )

        required_snapshot_keys = {
            "project",
            "reproducibility",
            "experiment",
            "datasets",
            "models",
            "evaluation",
            "privacy",
            "sppgan",
            "paths",
        }

        CONFIG_SNAPSHOT_PASS = (
            isinstance(CONFIG_SNAPSHOT, dict)
            and required_snapshot_keys.issubset(
                CONFIG_SNAPSHOT.keys()
            )
        )

# ==============================================================================
# 2.8 — VERIFY CONFIGURATION FILES AGAINST MANIFEST SNAPSHOT
# ==============================================================================

SNAPSHOT_FILE_MAPPING = {
    "config_json": "experiment",
    "dataset_registry_json": "datasets",
    "model_registry_json": "models",
    "evaluation_config_json": "evaluation",
    "privacy_config_json": "privacy",
    "sppgan_config_json": "sppgan",
}

SNAPSHOT_FILE_CHECKS = {}

if CONFIG_SNAPSHOT_PASS:

    for artifact_name, snapshot_key in SNAPSHOT_FILE_MAPPING.items():

        loaded_object = CONFIGURATION_OBJECTS.get(artifact_name)

        SNAPSHOT_FILE_CHECKS[artifact_name] = (
            isinstance(loaded_object, dict)
            and loaded_object == CONFIG_SNAPSHOT.get(snapshot_key)
        )

    CONFIGURATION_LINKAGE_PASS = all(
        SNAPSHOT_FILE_CHECKS.values()
    )

# ==============================================================================
# 2.9 — VERIFY NOTEBOOK 00 CONFIGURATION FINGERPRINT
# ==============================================================================
# Notebook 00 canonical serialization:
# json.dumps(
#     data,
#     sort_keys=True,
#     default=str,
#     ensure_ascii=False,
#     separators=(",", ":")
# )

def canonical_json_string(data):
    return json.dumps(
        data,
        sort_keys=True,
        default=str,
        ensure_ascii=False,
        separators=(",", ":"),
    )


def calculate_sha256(data):
    canonical_bytes = canonical_json_string(data).encode("utf-8")
    return hashlib.sha256(canonical_bytes).hexdigest()


CONFIGURATION_FINGERPRINT = None
RECALCULATED_CONFIG_HASH = None
FINGERPRINT_HASH_MATCH_PASS = False
FINGERPRINT_STRUCTURE_PASS = False

if CONFIGURATION_FINGERPRINT_PATH.is_file():

    try:
        CONFIGURATION_FINGERPRINT = load_json_artifact(
            CONFIGURATION_FINGERPRINT_PATH
        )

        required_fingerprint_keys = {
            "fingerprint_version",
            "project",
            "fingerprinted_object",
            "serialization",
            "algorithm",
            "hash",
            "created_utc",
        }

        FINGERPRINT_STRUCTURE_PASS = (
            required_fingerprint_keys.issubset(
                CONFIGURATION_FINGERPRINT.keys()
            )
        )

        if (
            FINGERPRINT_STRUCTURE_PASS
            and CONFIG_SNAPSHOT_PASS
        ):

            RECALCULATED_CONFIG_HASH = calculate_sha256(
                CONFIG_SNAPSHOT
            )

            FINGERPRINT_HASH_MATCH_PASS = (
                CONFIGURATION_FINGERPRINT.get("algorithm") == "SHA256"
                and CONFIGURATION_FINGERPRINT.get(
                    "fingerprinted_object"
                ) == "CONFIG_SNAPSHOT"
                and CONFIGURATION_FINGERPRINT.get("hash")
                == RECALCULATED_CONFIG_HASH
            )

    except Exception as exc:
        LOAD_ERRORS["configuration_fingerprint_json"] = (
            f"{type(exc).__name__}: {exc}"
        )


# ==============================================================================
# 2.10 — VALIDATE PROJECT AND EXPERIMENT CONFIGURATION
# ==============================================================================

EXPECTED_DATASETS = {
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
}

EXPECTED_MODELS = {
    "statistical",
    "tvae",
    "ctgan",
    "dp_ctgan",
    "spp_gan",
}

PROJECT_METADATA_PASS = False
DATASET_REGISTRY_PASS = False
MODEL_REGISTRY_PASS = False
MASTER_SEED_PASS = False
REPETITION_POLICY_PASS = False
DATA_SPLIT_PASS = False

if CONFIG_SNAPSHOT_PASS:

    project_metadata = CONFIG_SNAPSHOT.get("project", {})
    reproducibility = CONFIG_SNAPSHOT.get("reproducibility", {})
    experiment = CONFIG_SNAPSHOT.get("experiment", {})

    # --------------------------------------------------------------------------
    # Project metadata
    # --------------------------------------------------------------------------

    PROJECT_METADATA_PASS = (
        project_metadata.get("notebook_id") == "00"
        and project_metadata.get("project_name")
        == "SPP-GAN Research Project"
        and project_metadata.get("project_version") == "1.0"
    )

    # --------------------------------------------------------------------------
    # Dataset registry
    # --------------------------------------------------------------------------

    datasets = CONFIG_SNAPSHOT.get("datasets", {})

    DATASET_REGISTRY_PASS = (
        isinstance(datasets, dict)
        and EXPECTED_DATASETS.issubset(set(datasets.keys()))
    )

    # --------------------------------------------------------------------------
    # Model registry
    # --------------------------------------------------------------------------

    models = CONFIG_SNAPSHOT.get("models", {})

    MODEL_REGISTRY_PASS = (
        isinstance(models, dict)
        and EXPECTED_MODELS.issubset(set(models.keys()))
    )

    # --------------------------------------------------------------------------
    # Master seed — authoritative reproducibility snapshot
    # --------------------------------------------------------------------------

    master_seed = reproducibility.get("master_seed")

    MASTER_SEED_PASS = (
        isinstance(master_seed, int)
        and not isinstance(master_seed, bool)
        and master_seed == 2025
    )

    # --------------------------------------------------------------------------
    # Repetition policy — authoritative experimental configuration
    # --------------------------------------------------------------------------

    seed_policy = experiment.get("seed_policy", {})

    repetition_seeds = seed_policy.get(
        "repetition_seeds", {}
    )

    expected_repetition_seeds = {
        repetition: 2025 + 1000 + repetition
        for repetition in range(1, 6)
    }

    normalized_repetition_seeds = {}

    if isinstance(repetition_seeds, dict):
        try:
            normalized_repetition_seeds = {
                int(key): int(value)
                for key, value in repetition_seeds.items()
            }
        except (TypeError, ValueError):
            normalized_repetition_seeds = {}

    REPETITION_POLICY_PASS = (
        isinstance(seed_policy, dict)
        and seed_policy.get("master_seed") == 2025
        and seed_policy.get("repetitions") == 5
        and seed_policy.get("deterministic") is True
        and seed_policy.get("repetition_seed_offset") == 1000
        and normalized_repetition_seeds
        == expected_repetition_seeds
    )

    # --------------------------------------------------------------------------
    # Data split configuration
    # --------------------------------------------------------------------------

    split_config = experiment.get("data_split", {})

    DATA_SPLIT_PASS = (
        isinstance(split_config, dict)
        and split_config.get("train_fraction") == 0.70
        and split_config.get("validation_fraction") == 0.15
        and split_config.get("test_fraction") == 0.15
        and abs(
            split_config.get("train_fraction", 0)
            + split_config.get("validation_fraction", 0)
            + split_config.get("test_fraction", 0)
            - 1.0
        ) < 1e-12
    )

# ==============================================================================
# REPORT
# ==============================================================================

print("\nEXPERIMENT CONFIGURATION VALIDATION")
print("-" * 100)

print("Project metadata       :", "PASS" if PROJECT_METADATA_PASS else "FAIL")
print("Dataset registry       :", "PASS" if DATASET_REGISTRY_PASS else "FAIL")
print("Model registry         :", "PASS" if MODEL_REGISTRY_PASS else "FAIL")
print("Master seed            :", "PASS" if MASTER_SEED_PASS else "FAIL")
print("Repetition seed policy :", "PASS" if REPETITION_POLICY_PASS else "FAIL")
print("Data split             :", "PASS" if DATA_SPLIT_PASS else "FAIL")

print("\nExpected repetition seeds:")
for repetition, seed in expected_repetition_seeds.items():
    actual = normalized_repetition_seeds.get(repetition)
    print(
        f"Repetition {repetition}: "
        f"expected={seed}, actual={actual}, "
        f"{'PASS' if actual == seed else 'FAIL'}"
    )

# ==============================================================================
# 2.11 — VERIFY MANIFEST ARTIFACT PATH LINKAGE
# ==============================================================================

MANIFEST_PATH_LINKAGE_PASS = False

if isinstance(REPRODUCIBILITY_MANIFEST, dict):

    manifest_paths = REPRODUCIBILITY_MANIFEST.get(
        "artifact_paths", {}
    )

    expected_paths = {
        key: str(value)
        for key, value in ARTIFACT_PATHS.items()
    }

    MANIFEST_PATH_LINKAGE_PASS = (
        isinstance(manifest_paths, dict)
        and manifest_paths == expected_paths
    )

# ==============================================================================
# 2.12 — CONFIGURATION HASHES FOR PROVENANCE
# ==============================================================================

ARTIFACT_SHA256 = {}

for artifact_name, artifact_path in ARTIFACT_PATHS.items():

    if artifact_path.is_file():

        digest = hashlib.sha256()

        with open(artifact_path, "rb") as file:
            for chunk in iter(
                lambda: file.read(1024 * 1024),
                b"",
            ):
                digest.update(chunk)

        ARTIFACT_SHA256[artifact_name] = digest.hexdigest()

# ==============================================================================
# 2.13 — SECTION 2 VALIDATION
# ==============================================================================

VALIDATION_CHECKS = {
    "project_root_exists":
        PROJECT_ROOT.is_dir(),

    "notebook_00_root_exists":
        NB00_ROOT.is_dir(),

    "all_eight_original_artifacts_present":
        all(
            status["is_file"] and status["non_empty"]
            for name, status in ARTIFACT_STATUS.items()
            if name != "configuration_fingerprint_json"
        ),

    "all_configuration_files_loaded":
        all(
            name in CONFIGURATION_OBJECTS
            for name in ARTIFACT_PATHS
        ),

    "manifest_structure_valid":
        MANIFEST_STRUCTURE_PASS,

    "configuration_snapshot_valid":
        CONFIG_SNAPSHOT_PASS,

    "configuration_files_match_snapshot":
        CONFIGURATION_LINKAGE_PASS,

    "fingerprint_file_present":
        CONFIGURATION_FINGERPRINT_PATH.is_file(),

    "fingerprint_structure_valid":
        FINGERPRINT_STRUCTURE_PASS,

    "fingerprint_matches_snapshot":
        FINGERPRINT_HASH_MATCH_PASS,

    "project_metadata_valid":
        PROJECT_METADATA_PASS,

    "dataset_registry_valid":
        DATASET_REGISTRY_PASS,

    "model_registry_valid":
        MODEL_REGISTRY_PASS,

    "master_seed_valid":
        MASTER_SEED_PASS,

    "repetition_policy_valid":
        REPETITION_POLICY_PASS,

    "data_split_valid":
        DATA_SPLIT_PASS,

    "manifest_artifact_paths_match":
        MANIFEST_PATH_LINKAGE_PASS,
}

VALIDATION_DF = pd.DataFrame([
    {
        "check": name,
        "status": "PASS" if passed else "FAIL",
    }
    for name, passed in VALIDATION_CHECKS.items()
])

PASSED_CHECKS = int(
    VALIDATION_DF["status"].eq("PASS").sum()
)

FAILED_CHECKS = int(
    VALIDATION_DF["status"].eq("FAIL").sum()
)

SECTION_02_PASS = FAILED_CHECKS == 0

# ==============================================================================
# 2.14 — SAVE NOTEBOOK 22 AUDIT METADATA
# ==============================================================================

SECTION_02_METADATA = {
    "notebook_id": "22",
    "section": "02_load_project_configuration",
    "execution_timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "notebook_00_root": str(NB00_ROOT),
    "artifact_paths": {
        key: str(value)
        for key, value in ARTIFACT_PATHS.items()
    },
    "artifact_status": ARTIFACT_STATUS,
    "load_errors": LOAD_ERRORS,
    "validation_checks": VALIDATION_CHECKS,
    "passed_checks": PASSED_CHECKS,
    "failed_checks": FAILED_CHECKS,
    "section_status": (
        "PASS" if SECTION_02_PASS else "FAIL"
    ),
    "configuration_fingerprint": CONFIGURATION_FINGERPRINT,
    "recalculated_configuration_hash": RECALCULATED_CONFIG_HASH,
    "artifact_sha256": ARTIFACT_SHA256,
    "python_version": platform.python_version(),
}

METADATA_PATH = (
    NB22_METADATA_DIR
    / "section_02_configuration_audit.json"
)

VALIDATION_PATH = (
    NB22_VALIDATION_DIR
    / "section_02_validation.csv"
)

with open(METADATA_PATH, "w", encoding="utf-8") as file:
    json.dump(
        SECTION_02_METADATA,
        file,
        indent=4,
        ensure_ascii=False,
        default=str,
    )

VALIDATION_DF.to_csv(
    VALIDATION_PATH,
    index=False,
)

# ==============================================================================
# 2.15 — FINAL REPORT
# ==============================================================================

print("\n" + "=" * 100)
print("SECTION 2 — CONFIGURATION VALIDATION REPORT")
print("=" * 100)

print(VALIDATION_DF.to_string(index=False))

print("\nConfiguration summary")
print("-" * 100)
print("Experiment configuration loaded :", EXPERIMENT_CONFIG is not None)
print("Dataset registry loaded         :", DATASET_REGISTRY is not None)
print("Model registry loaded           :", MODEL_REGISTRY is not None)
print("Evaluation configuration loaded:", EVALUATION_CONFIG is not None)
print("Privacy configuration loaded    :", PRIVACY_CONFIG is not None)
print("SPP-GAN configuration loaded    :", SPPGAN_CONFIG is not None)
print("Environment metadata loaded     :", ENVIRONMENT_METADATA is not None)
print("Reproducibility manifest loaded :", REPRODUCIBILITY_MANIFEST is not None)
print("Fingerprint verified            :", FINGERPRINT_HASH_MATCH_PASS)

print("\nValidation totals")
print("-" * 100)
print("Total checks :", len(VALIDATION_CHECKS))
print("Passed       :", PASSED_CHECKS)
print("Failed       :", FAILED_CHECKS)

print("\nAudit files")
print("-" * 100)
print("Metadata   :", METADATA_PATH)
print("Validation :", VALIDATION_PATH)

if SECTION_02_PASS:
    print("\nSECTION 2 STATUS: PASS")
    print("Original Notebook 00 configuration loaded and verified.")
else:
    print("\nSECTION 2 STATUS: FAIL")
    print("Original Notebook 00 configuration is incomplete or unverifiable.")
    print("Do not freeze Section 2.")
    print("Do not treat missing or reconstructed files as original artifacts.")

print("=" * 100)


SECTION 2 — LOAD PROJECT CONFIGURATION
Drive mount: /content/drive
MyDrive: /content/drive/MyDrive

Canonical project root: /content/drive/MyDrive/SPP_GAN_Research
Notebook 00 root: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_00
Notebook 22 root: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_22

NOTEBOOK 00 ORIGINAL ARTIFACT STATUS
----------------------------------------------------------------------------------------------------
FOUND    config_json                        /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_00/config/experiment_config.json
FOUND    dataset_registry_json              /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_00/config/dataset_registry.json
FOUND    model_registry_json                /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_00/config/model_registry.json
FOUND    evaluation_config_json             /content/drive/MyDrive/SPP_GAN_Research/results/n

In [3]:
# ============================================================
# SECTION 3 — VERIFY NOTEBOOK 00 ARTIFACTS
# ============================================================

from pathlib import Path
import pandas as pd
import json

NOTEBOOK_00 = PROJECT_ROOT / "results/notebooks/notebook_00"

# ------------------------------------------------------------
# 1. DEFINE REQUIRED NOTEBOOK 00 ARTIFACTS
# ------------------------------------------------------------

REQUIRED_NOTEBOOK_00_ARTIFACTS = {
    "config_json": (
        "config/experiment_config.json"
    ),
    "dataset_registry_json": (
        "config/dataset_registry.json"
    ),
    "model_registry_json": (
        "config/model_registry.json"
    ),
    "evaluation_config_json": (
        "config/evaluation_config.json"
    ),
    "privacy_config_json": (
        "config/privacy_config.json"
    ),
    "sppgan_config_json": (
        "config/sppgan_config.json"
    ),
    "environment_json": (
        "environment/environment.json"
    ),
    "manifest_json": (
        "manifest/notebook_00_manifest.json"
    ),
    "configuration_fingerprint_json": (
        "manifest/configuration_fingerprint.json"
    ),
}

# ------------------------------------------------------------
# 2. INSPECT REQUIRED ARTIFACTS
# ------------------------------------------------------------

notebook_00_records = []

for artifact_name, relative_path in (
    REQUIRED_NOTEBOOK_00_ARTIFACTS.items()
):
    artifact_path = NOTEBOOK_00 / relative_path

    if not artifact_path.exists():
        state = "MISSING"
        size_bytes = 0

    elif not artifact_path.is_file():
        state = "NOT_A_FILE"
        size_bytes = 0

    else:
        size_bytes = artifact_path.stat().st_size
        state = (
            "PRESENT"
            if size_bytes > 0
            else "EMPTY"
        )

    notebook_00_records.append({
        "notebook": "notebook_00",
        "artifact_name": artifact_name,
        "relative_path": relative_path,
        "path": str(artifact_path),
        "file_state": state,
        "size_bytes": size_bytes,
        "required": True,
    })

notebook_00_audit = pd.DataFrame(
    notebook_00_records
)

# ------------------------------------------------------------
# 3. VERIFY JSON ARTIFACT STRUCTURE
# ------------------------------------------------------------

json_validation_records = []

for record in notebook_00_records:

    artifact_path = Path(record["path"])

    if record["file_state"] != "PRESENT":
        json_status = "NOT_CHECKED"
        error_message = record["file_state"]

    else:
        try:
            with open(
                artifact_path,
                "r",
                encoding="utf-8"
            ) as file:
                json.load(file)

            json_status = "VALID_JSON"
            error_message = ""

        except Exception as exc:
            json_status = "INVALID_JSON"
            error_message = str(exc)

    json_validation_records.append({
        "artifact_name": record["artifact_name"],
        "path": str(artifact_path),
        "json_status": json_status,
        "error_message": error_message,
    })

notebook_00_json_audit = pd.DataFrame(
    json_validation_records
)

# ------------------------------------------------------------
# 4. DISCOVER ADDITIONAL NOTEBOOK 00 ARTIFACTS
# ------------------------------------------------------------

DISCOVERED_PATTERNS = [
    "*.json",
    "*.yaml",
    "*.yml",
    "*.csv",
]

discovered_files = []

if NOTEBOOK_00.is_dir():
    for pattern in DISCOVERED_PATTERNS:
        discovered_files.extend(
            NOTEBOOK_00.rglob(pattern)
        )

discovered_files = sorted(
    set(
        path for path in discovered_files
        if path.is_file()
    )
)

# ------------------------------------------------------------
# 5. VALIDATE REQUIRED ARTIFACT COVERAGE
# ------------------------------------------------------------

required_artifacts_present = (
    notebook_00_audit["file_state"]
    == "PRESENT"
).sum()

required_artifacts_total = len(
    REQUIRED_NOTEBOOK_00_ARTIFACTS
)

valid_json_count = (
    notebook_00_json_audit["json_status"]
    == "VALID_JSON"
).sum()

all_required_present = (
    required_artifacts_present
    == required_artifacts_total
)

all_required_json_valid = (
    valid_json_count
    == required_artifacts_total
)

notebook_00_root_valid = (
    NOTEBOOK_00.is_dir()
)

# ------------------------------------------------------------
# 6. SAVE AUDIT REPORTS
# ------------------------------------------------------------

TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True
)

notebook_00_audit.to_csv(
    TABLES_DIR / "notebook_00_artifact_audit.csv",
    index=False
)

notebook_00_json_audit.to_csv(
    TABLES_DIR / "notebook_00_json_validation.csv",
    index=False
)

# ------------------------------------------------------------
# 7. SECTION VALIDATION
# ------------------------------------------------------------

section_03_checks = [
    {
        "check": "notebook_00_root_exists",
        "status": (
            "PASS" if notebook_00_root_valid
            else "FAIL"
        ),
    },
    {
        "check": "all_required_artifacts_present",
        "status": (
            "PASS" if all_required_present
            else "FAIL"
        ),
    },
    {
        "check": "all_required_json_files_valid",
        "status": (
            "PASS" if all_required_json_valid
            else "FAIL"
        ),
    },
    {
        "check": "required_artifact_count",
        "status": (
            "PASS"
            if required_artifacts_total == 9
            else "FAIL"
        ),
    },
    {
        "check": "artifact_audit_saved",
        "status": (
            "PASS"
            if (
                TABLES_DIR
                / "notebook_00_artifact_audit.csv"
            ).is_file()
            else "FAIL"
        ),
    },
    {
        "check": "json_validation_saved",
        "status": (
            "PASS"
            if (
                TABLES_DIR
                / "notebook_00_json_validation.csv"
            ).is_file()
            else "FAIL"
        ),
    },
]

section_03_validation = pd.DataFrame(
    section_03_checks
)

VALIDATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

section_03_validation.to_csv(
    VALIDATION_DIR / "section_03_validation.csv",
    index=False
)

# ------------------------------------------------------------
# 8. EXECUTION SUMMARY
# ------------------------------------------------------------

passed_checks = (
    section_03_validation["status"] == "PASS"
).sum()

failed_checks = (
    section_03_validation["status"] == "FAIL"
).sum()

section_03_status = (
    "PASS"
    if failed_checks == 0
    else "FAIL"
)

print("=" * 100)
print("SECTION 3 — NOTEBOOK 00 ARTIFACT VERIFICATION")
print("=" * 100)

print("Notebook 00 root:", NOTEBOOK_00)
print("Required artifacts:", required_artifacts_total)
print("Required artifacts present:", required_artifacts_present)
print("Valid JSON artifacts:", valid_json_count)
print("Discovered structured files:", len(discovered_files))

print("\nREQUIRED ARTIFACT STATUS")
print("-" * 100)

print(
    notebook_00_audit[
        ["artifact_name", "file_state", "size_bytes"]
    ].to_string(index=False)
)

print("\nJSON VALIDATION")
print("-" * 100)

print(
    notebook_00_json_audit[
        ["artifact_name", "json_status"]
    ].to_string(index=False)
)

print("\nSECTION VALIDATION")
print("-" * 100)

print(section_03_validation.to_string(index=False))

print("\nPassed checks:", passed_checks)
print("Failed checks:", failed_checks)

print("\nSECTION 3 STATUS:", section_03_status)

print("=" * 100)


SECTION 3 — NOTEBOOK 00 ARTIFACT VERIFICATION
Notebook 00 root: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_00
Required artifacts: 9
Required artifacts present: 9
Valid JSON artifacts: 9
Discovered structured files: 9

REQUIRED ARTIFACT STATUS
----------------------------------------------------------------------------------------------------
                 artifact_name file_state  size_bytes
                   config_json    PRESENT        2262
         dataset_registry_json    PRESENT        1903
           model_registry_json    PRESENT        1516
        evaluation_config_json    PRESENT        3485
           privacy_config_json    PRESENT         519
            sppgan_config_json    PRESENT        1951
              environment_json    PRESENT        2205
                 manifest_json    PRESENT       20136
configuration_fingerprint_json    PRESENT         533

JSON VALIDATION
---------------------------------------------------------------------------

In [4]:
# ============================================================
# SECTION 4 — VERIFY NOTEBOOK 01 ARTIFACTS
# ============================================================

from pathlib import Path
import pandas as pd
import json
import hashlib

# ------------------------------------------------------------
# 1. DEFINE NOTEBOOK PATHS
# ------------------------------------------------------------

NOTEBOOK_01 = PROJECT_ROOT / "results" / "notebooks" / "notebook_01"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

NOTEBOOK_01_PATTERNS = [
    "*.csv",
    "*.json",
    "*.parquet",
    "*.pkl",
]

# ------------------------------------------------------------
# 2. DISCOVER ARTIFACTS
# ------------------------------------------------------------

notebook_01_files = []

if NOTEBOOK_01.is_dir():
    for pattern in NOTEBOOK_01_PATTERNS:
        notebook_01_files.extend(
            NOTEBOOK_01.rglob(pattern)
        )

notebook_01_files = sorted(
    set(
        path for path in notebook_01_files
        if path.is_file()
    )
)

# ------------------------------------------------------------
# 3. CREATE ARTIFACT INVENTORY
# ------------------------------------------------------------

notebook_01_records = []

for path in notebook_01_files:

    size_bytes = path.stat().st_size

    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    notebook_01_records.append({
        "notebook": "notebook_01",
        "artifact_name": path.name,
        "relative_path": str(
            path.relative_to(NOTEBOOK_01)
        ),
        "path": str(path),
        "extension": path.suffix.lower(),
        "file_state": (
            "PRESENT" if size_bytes > 0 else "EMPTY"
        ),
        "size_bytes": size_bytes,
        "sha256": digest.hexdigest(),
    })

notebook_01_audit = pd.DataFrame(
    notebook_01_records
)

# ------------------------------------------------------------
# 4. VALIDATE ARTIFACT CONTENTS
# ------------------------------------------------------------

notebook_01_content_checks = []

for path in notebook_01_files:

    extension = path.suffix.lower()
    status = "PASS"
    details = ""

    try:

        if path.stat().st_size == 0:
            status = "FAIL"
            details = "Empty file"

        elif extension == ".json":

            with open(
                path,
                "r",
                encoding="utf-8"
            ) as file:
                json.load(file)

            details = "Valid JSON"

        elif extension == ".csv":

            sample = pd.read_csv(
                path,
                nrows=5
            )

            if len(sample.columns) == 0:
                status = "FAIL"
                details = "No columns detected"
            else:
                details = (
                    f"Readable CSV; "
                    f"{len(sample.columns)} columns detected"
                )

        elif extension == ".parquet":

            import pyarrow.parquet as pq

            parquet_file = pq.ParquetFile(path)
            metadata = parquet_file.metadata
            schema = parquet_file.schema

            if len(schema.names) == 0:
                status = "FAIL"
                details = "No columns detected"
            else:
                details = (
                    f"Readable Parquet; "
                    f"{len(schema.names)} columns; "
                    f"{metadata.num_rows} rows"
                )

        elif extension == ".pkl":

            # Pickle is not deserialized because loading
            # untrusted pickle files can execute code.
            details = (
                "Non-empty pickle file; "
                "content not deserialized"
            )

        else:
            status = "FAIL"
            details = "Unsupported file extension"

    except Exception as exc:

        status = "FAIL"
        details = (
            f"{type(exc).__name__}: {str(exc)[:300]}"
        )

    notebook_01_content_checks.append({
        "artifact_name": path.name,
        "path": str(path),
        "extension": extension,
        "content_status": status,
        "details": details,
    })

notebook_01_content_audit = pd.DataFrame(
    notebook_01_content_checks
)

# ------------------------------------------------------------
# 5. CALCULATE ARTIFACT COVERAGE
# ------------------------------------------------------------

notebook_01_root_exists = NOTEBOOK_01.is_dir()

discovered_count = len(notebook_01_files)

nonempty_count = (
    int(
        notebook_01_audit["file_state"].eq("PRESENT").sum()
    )
    if not notebook_01_audit.empty
    else 0
)

content_pass_count = (
    int(
        notebook_01_content_audit[
            "content_status"
        ].eq("PASS").sum()
    )
    if not notebook_01_content_audit.empty
    else 0
)

all_artifacts_nonempty = (
    discovered_count > 0
    and nonempty_count == discovered_count
)

all_content_checks_pass = (
    discovered_count > 0
    and content_pass_count == discovered_count
)

# ------------------------------------------------------------
# 6. SAVE AUDIT REPORTS
# ------------------------------------------------------------

ARTIFACT_AUDIT_PATH = (
    TABLES_DIR / "notebook_01_artifact_audit.csv"
)

CONTENT_AUDIT_PATH = (
    TABLES_DIR / "notebook_01_content_validation.csv"
)

VALIDATION_PATH = (
    VALIDATION_DIR / "section_04_validation.csv"
)

notebook_01_audit.to_csv(
    ARTIFACT_AUDIT_PATH,
    index=False
)

notebook_01_content_audit.to_csv(
    CONTENT_AUDIT_PATH,
    index=False
)

# ------------------------------------------------------------
# 7. SECTION VALIDATION
# ------------------------------------------------------------

section_04_checks = [
    {
        "check": "notebook_01_root_exists",
        "status": (
            "PASS" if notebook_01_root_exists else "FAIL"
        ),
    },
    {
        "check": "artifacts_discovered",
        "status": (
            "PASS" if discovered_count > 0 else "FAIL"
        ),
    },
    {
        "check": "all_artifacts_nonempty",
        "status": (
            "PASS" if all_artifacts_nonempty else "FAIL"
        ),
    },
    {
        "check": "all_artifact_content_checks_pass",
        "status": (
            "PASS" if all_content_checks_pass else "FAIL"
        ),
    },
    {
        "check": "artifact_audit_saved",
        "status": (
            "PASS" if ARTIFACT_AUDIT_PATH.is_file() else "FAIL"
        ),
    },
    {
        "check": "content_validation_saved",
        "status": (
            "PASS" if CONTENT_AUDIT_PATH.is_file() else "FAIL"
        ),
    },
]

section_04_validation = pd.DataFrame(
    section_04_checks
)

section_04_validation.to_csv(
    VALIDATION_PATH,
    index=False
)

# ------------------------------------------------------------
# 8. FINAL SECTION STATUS
# ------------------------------------------------------------

passed_checks = int(
    section_04_validation["status"].eq("PASS").sum()
)

failed_checks = int(
    section_04_validation["status"].eq("FAIL").sum()
)

section_04_status = (
    "PASS" if failed_checks == 0 else "FAIL"
)

# ------------------------------------------------------------
# 9. EXECUTION REPORT
# ------------------------------------------------------------

print("=" * 100)
print("SECTION 4 — NOTEBOOK 01 ARTIFACT VERIFICATION")
print("=" * 100)

print("Notebook 01 root:", NOTEBOOK_01)
print("Directory exists:", notebook_01_root_exists)
print("Discovered artifacts:", discovered_count)
print("Non-empty artifacts:", nonempty_count)
print("Content checks passed:", content_pass_count)

print("\nARTIFACT INVENTORY")
print("-" * 100)

if not notebook_01_audit.empty:
    print(
        notebook_01_audit[
            [
                "artifact_name",
                "extension",
                "file_state",
                "size_bytes",
            ]
        ].to_string(index=False)
    )
else:
    print("No supported artifacts discovered.")

print("\nCONTENT VALIDATION")
print("-" * 100)

if not notebook_01_content_audit.empty:
    print(
        notebook_01_content_audit[
            [
                "artifact_name",
                "content_status",
                "details",
            ]
        ].to_string(index=False)
    )
else:
    print("No content checks performed.")

print("\nSECTION VALIDATION")
print("-" * 100)

print(section_04_validation.to_string(index=False))

print("\nPassed checks:", passed_checks)
print("Failed checks:", failed_checks)

print("\nAudit files:")
print("Artifact inventory:", ARTIFACT_AUDIT_PATH)
print("Content validation:", CONTENT_AUDIT_PATH)
print("Section validation:", VALIDATION_PATH)

print("\nSECTION 4 STATUS:", section_04_status)
print("=" * 100)


SECTION 4 — NOTEBOOK 01 ARTIFACT VERIFICATION
Notebook 01 root: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_01
Directory exists: True
Discovered artifacts: 22
Non-empty artifacts: 22
Content checks passed: 22

ARTIFACT INVENTORY
----------------------------------------------------------------------------------------------------
                           artifact_name extension file_state  size_bytes
      adult_income_feature_inventory.csv      .csv    PRESENT         965
    bank_marketing_feature_inventory.csv      .csv    PRESENT         991
    diabetes_130us_feature_inventory.csv      .csv    PRESENT        3148
               notebook_01_manifest.json     .json    PRESENT        3073
      raw_dataset_download_manifest.json     .json    PRESENT        2164
      raw_file_fingerprint_manifest.json     .json    PRESENT        1373
        adult_income_raw_missingness.csv      .csv    PRESENT         886
      bank_marketing_raw_missingness.csv      .csv    P

In [5]:
# ============================================================
# SECTION 5 — VERIFY NOTEBOOK 02 ARTIFACTS
# ============================================================

from pathlib import Path
import pandas as pd
import json

# ------------------------------------------------------------
# 1. CANONICAL NOTEBOOK 02 DATA ROOT
# ------------------------------------------------------------

NOTEBOOK_02_DATA = (
    PROJECT_ROOT / "data/processed/notebook_02"
)

# ------------------------------------------------------------
# 2. DEFINE EXPECTED NOTEBOOK 02 ARTIFACTS
# ------------------------------------------------------------

EXPECTED_NOTEBOOK_02 = {
    "adult_income_train": {
        "path": "native/adult_income/train.csv",
        "expected_rows": 34189,
    },
    "bank_marketing_train": {
        "path": "native/bank_marketing/train.csv",
        "expected_rows": 31647,
    },
    "diabetes_130us_train": {
        "path": "native/diabetes_130us/train.csv",
        "expected_rows": 71236,
    },
    "native_dataset_manifest": {
        "path": "native/native_dataset_manifest.csv",
        "expected_rows": None,
    },
    "preprocessor_manifest": {
        "path": "schemas/preprocessor_manifest.csv",
        "expected_rows": None,
    },
}

# ------------------------------------------------------------
# 3. VERIFY ROOT DIRECTORY
# ------------------------------------------------------------

notebook_02_root_exists = NOTEBOOK_02_DATA.is_dir()

# ------------------------------------------------------------
# 4. DISCOVER ALL SAVED ARTIFACTS
# ------------------------------------------------------------

notebook_02_files = (
    sorted(
        path for path in NOTEBOOK_02_DATA.rglob("*")
        if path.is_file()
    )
    if notebook_02_root_exists
    else []
)

# ------------------------------------------------------------
# 5. VERIFY EXPECTED ARTIFACTS
# ------------------------------------------------------------

notebook_02_records = []

for artifact_name, specification in (
    EXPECTED_NOTEBOOK_02.items()
):

    artifact_path = (
        NOTEBOOK_02_DATA / specification["path"]
    )

    expected_rows = specification["expected_rows"]

    actual_rows = None
    actual_columns = None
    row_count_status = "NOT_APPLICABLE"
    content_status = "NOT_CHECKED"
    error_message = ""

    if not artifact_path.exists():

        file_state = "MISSING"
        size_bytes = 0
        content_status = "FAIL"
        error_message = "Expected artifact not found"

    elif not artifact_path.is_file():

        file_state = "NOT_A_FILE"
        size_bytes = 0
        content_status = "FAIL"
        error_message = "Expected path is not a file"

    else:

        size_bytes = artifact_path.stat().st_size

        if size_bytes == 0:

            file_state = "EMPTY"
            content_status = "FAIL"
            error_message = "Artifact is empty"

        else:

            file_state = "PRESENT"

            try:

                # All five expected artifacts are CSV files.
                dataframe = pd.read_csv(
                    artifact_path,
                    low_memory=False
                )

                actual_rows = len(dataframe)
                actual_columns = len(dataframe.columns)

                if actual_columns == 0:
                    content_status = "FAIL"
                    error_message = "No columns detected"

                else:
                    content_status = "PASS"

                if expected_rows is not None:

                    row_count_status = (
                        "PASS"
                        if actual_rows == expected_rows
                        else "FAIL"
                    )

                else:
                    row_count_status = "NOT_APPLICABLE"

            except Exception as exc:

                content_status = "FAIL"
                error_message = (
                    f"{type(exc).__name__}: "
                    f"{str(exc)[:300]}"
                )

    notebook_02_records.append({
        "notebook": "notebook_02",
        "artifact_name": artifact_name,
        "relative_path": specification["path"],
        "path": str(artifact_path),
        "file_state": file_state,
        "size_bytes": size_bytes,
        "expected_rows": expected_rows,
        "actual_rows": actual_rows,
        "actual_columns": actual_columns,
        "content_status": content_status,
        "row_count_status": row_count_status,
        "error_message": error_message,
    })

notebook_02_audit = pd.DataFrame(
    notebook_02_records
)

# ------------------------------------------------------------
# 6. CREATE ADDITIONAL FILE INVENTORY
# ------------------------------------------------------------

notebook_02_inventory = []

for path in notebook_02_files:

    notebook_02_inventory.append({
        "artifact_name": path.name,
        "relative_path": str(
            path.relative_to(NOTEBOOK_02_DATA)
        ),
        "extension": path.suffix.lower(),
        "size_bytes": path.stat().st_size,
        "file_state": (
            "PRESENT"
            if path.stat().st_size > 0
            else "EMPTY"
        ),
    })

notebook_02_inventory_df = pd.DataFrame(
    notebook_02_inventory
)

# ------------------------------------------------------------
# 7. CALCULATE VALIDATION RESULTS
# ------------------------------------------------------------

expected_count = len(EXPECTED_NOTEBOOK_02)

present_count = (
    (notebook_02_audit["file_state"] == "PRESENT").sum()
)

content_pass_count = (
    (notebook_02_audit["content_status"] == "PASS").sum()
)

row_checks = notebook_02_audit[
    notebook_02_audit["expected_rows"].notna()
]

row_count_pass_count = (
    (row_checks["row_count_status"] == "PASS").sum()
)

all_expected_present = (
    present_count == expected_count
)

all_content_valid = (
    content_pass_count == expected_count
)

all_expected_rows_match = (
    row_count_pass_count == len(row_checks)
    and len(row_checks) == 3
)

# ------------------------------------------------------------
# 8. SAVE AUDIT REPORTS
# ------------------------------------------------------------

TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True
)

notebook_02_audit.to_csv(
    TABLES_DIR / "notebook_02_artifact_audit.csv",
    index=False
)

notebook_02_inventory_df.to_csv(
    TABLES_DIR / "notebook_02_complete_inventory.csv",
    index=False
)

# ------------------------------------------------------------
# 9. SECTION VALIDATION
# ------------------------------------------------------------

section_05_checks = [
    {
        "check": "notebook_02_data_root_exists",
        "status": (
            "PASS" if notebook_02_root_exists
            else "FAIL"
        ),
    },
    {
        "check": "notebook_02_artifacts_discovered",
        "status": (
            "PASS" if len(notebook_02_files) > 0
            else "FAIL"
        ),
    },
    {
        "check": "all_expected_artifacts_present",
        "status": (
            "PASS" if all_expected_present
            else "FAIL"
        ),
    },
    {
        "check": "all_expected_csv_files_readable",
        "status": (
            "PASS" if all_content_valid
            else "FAIL"
        ),
    },
    {
        "check": "training_row_counts_valid",
        "status": (
            "PASS" if all_expected_rows_match
            else "FAIL"
        ),
    },
    {
        "check": "artifact_audit_saved",
        "status": (
            "PASS"
            if (
                TABLES_DIR
                / "notebook_02_artifact_audit.csv"
            ).is_file()
            else "FAIL"
        ),
    },
    {
        "check": "complete_inventory_saved",
        "status": (
            "PASS"
            if (
                TABLES_DIR
                / "notebook_02_complete_inventory.csv"
            ).is_file()
            else "FAIL"
        ),
    },
]

section_05_validation = pd.DataFrame(
    section_05_checks
)

VALIDATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

section_05_validation.to_csv(
    VALIDATION_DIR / "section_05_validation.csv",
    index=False
)

# ------------------------------------------------------------
# 10. FINAL REPORT
# ------------------------------------------------------------

passed_checks = (
    section_05_validation["status"] == "PASS"
).sum()

failed_checks = (
    section_05_validation["status"] == "FAIL"
).sum()

section_05_status = (
    "PASS"
    if failed_checks == 0
    else "FAIL"
)

print("=" * 100)
print("SECTION 5 — NOTEBOOK 02 ARTIFACT VERIFICATION")
print("=" * 100)

print("Canonical data root:", NOTEBOOK_02_DATA)
print("Directory exists:", notebook_02_root_exists)
print("Discovered files:", len(notebook_02_files))

print("\nEXPECTED ARTIFACT VERIFICATION")
print("-" * 100)

print(
    notebook_02_audit[
        [
            "artifact_name",
            "file_state",
            "expected_rows",
            "actual_rows",
            "actual_columns",
            "content_status",
            "row_count_status",
        ]
    ].to_string(index=False)
)

print("\nARTIFACT SUMMARY")
print("-" * 100)

print("Expected artifacts:", expected_count)
print("Artifacts present:", present_count)
print("Readable CSV artifacts:", content_pass_count)
print("Training row counts matched:", row_count_pass_count)

print("\nSECTION VALIDATION")
print("-" * 100)

print(section_05_validation.to_string(index=False))

print("\nPassed checks:", passed_checks)
print("Failed checks:", failed_checks)

print("\nSECTION 5 STATUS:", section_05_status)

print("=" * 100)


SECTION 5 — NOTEBOOK 02 ARTIFACT VERIFICATION
Canonical data root: /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02
Directory exists: True
Discovered files: 38

EXPECTED ARTIFACT VERIFICATION
----------------------------------------------------------------------------------------------------
          artifact_name file_state  expected_rows  actual_rows  actual_columns content_status row_count_status
     adult_income_train    PRESENT        34189.0        34189              16           PASS             PASS
   bank_marketing_train    PRESENT        31647.0        31647              18           PASS             PASS
   diabetes_130us_train    PRESENT        71236.0        71236              49           PASS             PASS
native_dataset_manifest    PRESENT            NaN            9              19           PASS   NOT_APPLICABLE
  preprocessor_manifest    PRESENT            NaN            3              21           PASS   NOT_APPLICABLE

ARTIFACT SUMMARY
-----

In [6]:
# ============================================================
# SECTION 6 — VERIFY NOTEBOOK 03 ARTIFACTS
# ============================================================

from pathlib import Path
import pandas as pd
import json

# ------------------------------------------------------------
# 1. CANONICAL NOTEBOOK 03 ROOT
# ------------------------------------------------------------

NOTEBOOK_03 = (
    PROJECT_ROOT / "data/processed/notebook_03"
)

# ------------------------------------------------------------
# 2. DEFINE REQUIRED ARTIFACTS
# ------------------------------------------------------------

EXPECTED_NOTEBOOK_03 = {
    "adult_income_guidance": {
        "path": "guidance/adult_income_spp_gan_statistical_guidance.json",
        "format": "json",
    },
    "bank_marketing_guidance": {
        "path": "guidance/bank_marketing_spp_gan_statistical_guidance.json",
        "format": "json",
    },
    "diabetes_130us_guidance": {
        "path": "guidance/diabetes_130us_spp_gan_statistical_guidance.json",
        "format": "json",
    },
    "adult_income_reference": {
        "path": "reference/adult_income_spp_gan_statistical_reference.json",
        "format": "json",
    },
    "bank_marketing_reference": {
        "path": "reference/bank_marketing_spp_gan_statistical_reference.json",
        "format": "json",
    },
    "diabetes_130us_reference": {
        "path": "reference/diabetes_130us_spp_gan_statistical_reference.json",
        "format": "json",
    },
    "dataset_characterization": {
        "path": "reports/dataset_characterization.csv",
        "format": "csv",
    },
    "feature_type_characterization": {
        "path": "reports/feature_type_characterization.csv",
        "format": "csv",
    },
    "notebook_03_final_status": {
        "path": "reports/notebook_03_final_status.csv",
        "format": "csv",
    },
    "statistical_coverage_summary": {
        "path": "reports/statistical_coverage_summary.csv",
        "format": "csv",
    },
    "completion_metadata": {
        "path": "schemas/notebook_03_completion_metadata.json",
        "format": "json",
    },
    "completion_summary": {
        "path": "schemas/notebook_03_completion_summary.json",
        "format": "json",
    },
    "final_verification": {
        "path": "validation/notebook_03_final_verification.csv",
        "format": "csv",
    },
}

EXPECTED_DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

# ------------------------------------------------------------
# 3. VERIFY ROOT DIRECTORY
# ------------------------------------------------------------

notebook_03_root_exists = NOTEBOOK_03.is_dir()

# ------------------------------------------------------------
# 4. DISCOVER ALL NOTEBOOK 03 FILES
# ------------------------------------------------------------

notebook_03_files = (
    sorted(
        path for path in NOTEBOOK_03.rglob("*")
        if path.is_file()
    )
    if notebook_03_root_exists
    else []
)

# ------------------------------------------------------------
# 5. VERIFY REQUIRED ARTIFACTS
# ------------------------------------------------------------

notebook_03_records = []

for artifact_name, specification in (
    EXPECTED_NOTEBOOK_03.items()
):

    artifact_path = (
        NOTEBOOK_03 / specification["path"]
    )

    expected_format = specification["format"]

    actual_rows = None
    actual_columns = None
    content_status = "NOT_CHECKED"
    error_message = ""

    if not artifact_path.exists():

        file_state = "MISSING"
        size_bytes = 0
        content_status = "FAIL"
        error_message = "Expected artifact not found"

    elif not artifact_path.is_file():

        file_state = "NOT_A_FILE"
        size_bytes = 0
        content_status = "FAIL"
        error_message = "Expected path is not a file"

    else:

        size_bytes = artifact_path.stat().st_size

        if size_bytes == 0:

            file_state = "EMPTY"
            content_status = "FAIL"
            error_message = "Artifact is empty"

        else:

            file_state = "PRESENT"

            try:

                if expected_format == "json":

                    with open(
                        artifact_path,
                        "r",
                        encoding="utf-8"
                    ) as file:
                        json.load(file)

                    content_status = "PASS"

                elif expected_format == "csv":

                    dataframe = pd.read_csv(
                        artifact_path,
                        low_memory=False
                    )

                    actual_rows = len(dataframe)
                    actual_columns = len(dataframe.columns)

                    content_status = (
                        "PASS"
                        if actual_columns > 0
                        else "FAIL"
                    )

                    if actual_columns == 0:
                        error_message = (
                            "CSV contains no columns"
                        )

            except Exception as exc:

                content_status = "FAIL"
                error_message = (
                    f"{type(exc).__name__}: "
                    f"{str(exc)[:300]}"
                )

    notebook_03_records.append({
        "notebook": "notebook_03",
        "artifact_name": artifact_name,
        "relative_path": specification["path"],
        "path": str(artifact_path),
        "expected_format": expected_format,
        "file_state": file_state,
        "size_bytes": size_bytes,
        "actual_rows": actual_rows,
        "actual_columns": actual_columns,
        "content_status": content_status,
        "error_message": error_message,
    })

notebook_03_audit = pd.DataFrame(
    notebook_03_records
)

# ------------------------------------------------------------
# 6. CHECK DATASET GUIDANCE AND REFERENCE COVERAGE
# ------------------------------------------------------------

dataset_coverage_records = []

for dataset in EXPECTED_DATASETS:

    guidance_path = (
        NOTEBOOK_03
        / "guidance"
        / f"{dataset}_spp_gan_statistical_guidance.json"
    )

    reference_path = (
        NOTEBOOK_03
        / "reference"
        / f"{dataset}_spp_gan_statistical_reference.json"
    )

    dataset_coverage_records.append({
        "dataset": dataset,
        "guidance_present": guidance_path.is_file(),
        "reference_present": reference_path.is_file(),
        "guidance_nonempty": (
            guidance_path.is_file()
            and guidance_path.stat().st_size > 0
        ),
        "reference_nonempty": (
            reference_path.is_file()
            and reference_path.stat().st_size > 0
        ),
    })

notebook_03_dataset_coverage = pd.DataFrame(
    dataset_coverage_records
)

dataset_coverage_pass = (
    notebook_03_dataset_coverage[
        [
            "guidance_present",
            "reference_present",
            "guidance_nonempty",
            "reference_nonempty",
        ]
    ].all(axis=1).all()
)

# ------------------------------------------------------------
# 7. CALCULATE VALIDATION RESULTS
# ------------------------------------------------------------

expected_count = len(EXPECTED_NOTEBOOK_03)

present_count = (
    (notebook_03_audit["file_state"] == "PRESENT").sum()
)

content_pass_count = (
    (notebook_03_audit["content_status"] == "PASS").sum()
)

all_expected_present = (
    present_count == expected_count
)

all_content_valid = (
    content_pass_count == expected_count
)

artifacts_discovered = (
    len(notebook_03_files) > 0
)

# ------------------------------------------------------------
# 8. SAVE AUDIT REPORTS
# ------------------------------------------------------------

TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True
)

notebook_03_audit.to_csv(
    TABLES_DIR / "notebook_03_artifact_audit.csv",
    index=False
)

notebook_03_dataset_coverage.to_csv(
    TABLES_DIR / "notebook_03_dataset_coverage.csv",
    index=False
)

# ------------------------------------------------------------
# 9. SECTION VALIDATION
# ------------------------------------------------------------

section_06_checks = [
    {
        "check": "notebook_03_root_exists",
        "status": (
            "PASS" if notebook_03_root_exists
            else "FAIL"
        ),
    },
    {
        "check": "artifacts_discovered",
        "status": (
            "PASS" if artifacts_discovered
            else "FAIL"
        ),
    },
    {
        "check": "all_required_artifacts_present",
        "status": (
            "PASS" if all_expected_present
            else "FAIL"
        ),
    },
    {
        "check": "all_required_artifacts_readable",
        "status": (
            "PASS" if all_content_valid
            else "FAIL"
        ),
    },
    {
        "check": "three_dataset_guidance_coverage",
        "status": (
            "PASS" if dataset_coverage_pass
            else "FAIL"
        ),
    },
    {
        "check": "artifact_audit_saved",
        "status": (
            "PASS"
            if (
                TABLES_DIR
                / "notebook_03_artifact_audit.csv"
            ).is_file()
            else "FAIL"
        ),
    },
    {
        "check": "dataset_coverage_saved",
        "status": (
            "PASS"
            if (
                TABLES_DIR
                / "notebook_03_dataset_coverage.csv"
            ).is_file()
            else "FAIL"
        ),
    },
]

section_06_validation = pd.DataFrame(
    section_06_checks
)

VALIDATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

section_06_validation.to_csv(
    VALIDATION_DIR / "section_06_validation.csv",
    index=False
)

# ------------------------------------------------------------
# 10. FINAL REPORT
# ------------------------------------------------------------

passed_checks = (
    section_06_validation["status"] == "PASS"
).sum()

failed_checks = (
    section_06_validation["status"] == "FAIL"
).sum()

section_06_status = (
    "PASS"
    if failed_checks == 0
    else "FAIL"
)

print("=" * 100)
print("SECTION 6 — NOTEBOOK 03 ARTIFACT VERIFICATION")
print("=" * 100)

print("Canonical root:", NOTEBOOK_03)
print("Directory exists:", notebook_03_root_exists)
print("Discovered files:", len(notebook_03_files))

print("\nEXPECTED ARTIFACT VERIFICATION")
print("-" * 100)

print(
    notebook_03_audit[
        [
            "artifact_name",
            "file_state",
            "expected_format",
            "content_status",
            "actual_rows",
            "actual_columns",
        ]
    ].to_string(index=False)
)

print("\nDATASET GUIDANCE COVERAGE")
print("-" * 100)

print(
    notebook_03_dataset_coverage.to_string(
        index=False
    )
)

print("\nARTIFACT SUMMARY")
print("-" * 100)

print("Expected artifacts:", expected_count)
print("Artifacts present:", present_count)
print("Artifacts readable:", content_pass_count)

print("\nSECTION VALIDATION")
print("-" * 100)

print(
    section_06_validation.to_string(
        index=False
    )
)

print("\nPassed checks:", passed_checks)
print("Failed checks:", failed_checks)

print("\nSECTION 6 STATUS:", section_06_status)

print("=" * 100)


SECTION 6 — NOTEBOOK 03 ARTIFACT VERIFICATION
Canonical root: /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_03
Directory exists: True
Discovered files: 35

EXPECTED ARTIFACT VERIFICATION
----------------------------------------------------------------------------------------------------
                artifact_name file_state expected_format content_status  actual_rows  actual_columns
        adult_income_guidance    PRESENT            json           PASS          NaN             NaN
      bank_marketing_guidance    PRESENT            json           PASS          NaN             NaN
      diabetes_130us_guidance    PRESENT            json           PASS          NaN             NaN
       adult_income_reference    PRESENT            json           PASS          NaN             NaN
     bank_marketing_reference    PRESENT            json           PASS          NaN             NaN
     diabetes_130us_reference    PRESENT            json           PASS          NaN    

In [7]:
# =============================================================================
# SECTION 7 — VERIFY BASELINE ARTIFACTS
# =============================================================================

from pathlib import Path
import json
import pandas as pd

print("=" * 100)
print("SECTION 7 — VERIFY BASELINE ARTIFACTS")
print("=" * 100)

BASELINE_ROOTS = {
    "notebook_04": (
        PROJECT_ROOT / "data" / "processed" / "notebook_04"
    ),
    "notebook_05": (
        PROJECT_ROOT / "results" / "notebooks" / "notebook_05"
    ),
    "notebook_06": (
        PROJECT_ROOT / "results" / "notebooks" / "notebook_06"
    )
}

SUPPORTED_EXTENSIONS = {
    ".csv", ".json", ".pkl", ".pt", ".pth", ".parquet"
}

artifact_records = []
content_records = []
directory_records = []

for notebook_name, root in BASELINE_ROOTS.items():

    exists = root.exists() and root.is_dir()

    directory_records.append({
        "notebook": notebook_name,
        "directory": str(root),
        "exists": exists
    })

    if not exists:
        print(f"WARNING: Missing directory — {notebook_name}")
        continue

    files = sorted(
        path for path in root.rglob("*")
        if path.is_file()
        and path.suffix.lower() in SUPPORTED_EXTENSIONS
    )

    for path in files:

        size = path.stat().st_size
        suffix = path.suffix.lower()
        relative_path = str(path.relative_to(root))

        artifact_records.append({
            "notebook": notebook_name,
            "relative_path": relative_path,
            "file_name": path.name,
            "extension": suffix,
            "size_bytes": size,
            "non_empty": size > 0
        })

        readable = None
        rows = None
        columns = None
        error = None

        try:
            if size == 0:
                raise ValueError("Empty file")

            if suffix == ".csv":
                df = pd.read_csv(path)
                readable = True
                rows = len(df)
                columns = len(df.columns)

            elif suffix == ".json":
                with open(path, "r", encoding="utf-8") as f:
                    json.load(f)
                readable = True

            elif suffix == ".parquet":
                df = pd.read_parquet(path)
                readable = True
                rows = len(df)
                columns = len(df.columns)

            else:
                # Binary artifacts are inventoried only.
                # Do not deserialize untrusted pickle/checkpoint files.
                readable = "inventory_only"

        except Exception as exc:
            readable = False
            error = str(exc)

        content_records.append({
            "notebook": notebook_name,
            "relative_path": relative_path,
            "extension": suffix,
            "readable": readable,
            "rows": rows,
            "columns": columns,
            "error": error
        })

artifact_df = pd.DataFrame(artifact_records)
content_df = pd.DataFrame(content_records)
directory_df = pd.DataFrame(directory_records)

# -------------------------------------------------------------------------
# SAVE REPORTS
# -------------------------------------------------------------------------

SECTION_07_TABLES = (
    PROJECT_ROOT / "results" / "notebooks" / "notebook_22" / "tables"
)

SECTION_07_VALIDATION = (
    PROJECT_ROOT / "results" / "notebooks" / "notebook_22" / "validation"
)

SECTION_07_TABLES.mkdir(parents=True, exist_ok=True)
SECTION_07_VALIDATION.mkdir(parents=True, exist_ok=True)

artifact_df.to_csv(
    SECTION_07_TABLES / "baseline_artifact_audit.csv",
    index=False
)

content_df.to_csv(
    SECTION_07_TABLES / "baseline_content_validation.csv",
    index=False
)

directory_df.to_csv(
    SECTION_07_TABLES / "baseline_directory_audit.csv",
    index=False
)

# -------------------------------------------------------------------------
# VALIDATION
# -------------------------------------------------------------------------

expected_directories = len(BASELINE_ROOTS)
existing_directories = int(directory_df["exists"].sum())

total_artifacts = len(artifact_df)

non_empty_artifacts = (
    int(artifact_df["non_empty"].sum())
    if total_artifacts else 0
)

checkable = content_df[
    content_df["extension"].isin([".csv", ".json", ".parquet"])
]

readable_count = int((checkable["readable"] == True).sum())
unreadable_count = int((checkable["readable"] == False).sum())

checks = {
    "all_baseline_directories_exist":
        existing_directories == expected_directories,

    "baseline_artifacts_discovered":
        total_artifacts > 0,

    "all_discovered_artifacts_nonempty":
        total_artifacts > 0
        and non_empty_artifacts == total_artifacts,

    "all_checkable_files_readable":
        unreadable_count == 0,

    "notebook_04_artifacts_present":
        (
            artifact_df["notebook"].eq("notebook_04").any()
            if not artifact_df.empty else False
        ),

    "notebook_05_artifacts_present":
        (
            artifact_df["notebook"].eq("notebook_05").any()
            if not artifact_df.empty else False
        ),

    "notebook_06_artifacts_present":
        (
            artifact_df["notebook"].eq("notebook_06").any()
            if not artifact_df.empty else False
        )
}

validation_df = pd.DataFrame([
    {
        "check": name,
        "status": "PASS" if passed else "FAIL"
    }
    for name, passed in checks.items()
])

validation_df.to_csv(
    SECTION_07_VALIDATION / "section_07_validation.csv",
    index=False
)

# -------------------------------------------------------------------------
# SUMMARY
# -------------------------------------------------------------------------

print("\nBASELINE ARTIFACT SUMMARY")
print("-" * 100)

print("Expected directories:", expected_directories)
print("Existing directories:", existing_directories)
print("Discovered artifacts:", total_artifacts)
print("Non-empty artifacts:", non_empty_artifacts)
print("Readable CSV/JSON/Parquet:", readable_count)
print("Unreadable CSV/JSON/Parquet:", unreadable_count)

print("\nNOTEBOOK-WISE ARTIFACT COUNTS")
print("-" * 100)

if not artifact_df.empty:
    print(
        artifact_df.groupby("notebook")
        .size()
        .to_string()
    )

print("\nVALIDATION RESULTS")
print("-" * 100)
print(validation_df.to_string(index=False))

passed = int((validation_df["status"] == "PASS").sum())
total = len(validation_df)

print("\nChecks passed:", passed, "/", total)

section_status = (
    "PASS" if passed == total else "FAIL"
)

print("\nSECTION 7 STATUS:", section_status)

print("\nSaved reports:")
print(SECTION_07_TABLES)
print(SECTION_07_VALIDATION)

print("=" * 100)
print("SECTION 7 COMPLETE")
print("=" * 100)

SECTION 7 — VERIFY BASELINE ARTIFACTS

BASELINE ARTIFACT SUMMARY
----------------------------------------------------------------------------------------------------
Expected directories: 3
Existing directories: 3
Discovered artifacts: 161
Non-empty artifacts: 161
Readable CSV/JSON/Parquet: 113
Unreadable CSV/JSON/Parquet: 0

NOTEBOOK-WISE ARTIFACT COUNTS
----------------------------------------------------------------------------------------------------
notebook
notebook_04    106
notebook_05     28
notebook_06     27

VALIDATION RESULTS
----------------------------------------------------------------------------------------------------
                            check status
   all_baseline_directories_exist   PASS
    baseline_artifacts_discovered   PASS
all_discovered_artifacts_nonempty   PASS
     all_checkable_files_readable   PASS
    notebook_04_artifacts_present   PASS
    notebook_05_artifacts_present   PASS
    notebook_06_artifacts_present   PASS

Checks passed: 7 / 7

SEC

In [8]:
# =============================================================================
# NOTEBOOK 22 — SECTION 8
# VERIFY SPP-GAN ARTIFACTS — FINAL VERSION
# =============================================================================
# Scope:
# 1. Verify Notebook 12 training registry and metadata.
# 2. Verify all 15 final model files using actual registry hash/size fields.
# 3. Verify source checkpoints using training metadata and registry hashes.
# 4. Verify five repetitions and seeds for each dataset.
# 5. Verify dataset-specific architecture dimensions.
# 6. Audit Notebook 13 generation summary, seed plan and execution log.
# 7. Verify canonical synthetic CSV files and SHA-256 hashes.
# 8. Audit restoration evidence and differential privacy boundaries.
# 9. Inspect generator source availability.
# 10. Save a consolidated audit report.
#
# SAFETY:
# - No model retraining.
# - No synthetic data regeneration.
# - No original artifact modification or deletion.
# - Audit outputs are saved separately.
# =============================================================================


# =============================================================================
# 1. IMPORTS AND CONFIGURATION
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


PROJECT_ROOT = Path(
    "/content/drive/MyDrive/SPP_GAN_Research"
)

NOTEBOOK_09 = PROJECT_ROOT / "results/notebooks/notebook_09"
NOTEBOOK_10 = PROJECT_ROOT / "results/notebooks/notebook_10"
NOTEBOOK_11 = PROJECT_ROOT / "results/notebooks/notebook_11"
NOTEBOOK_12 = PROJECT_ROOT / "results/notebooks/notebook_12"
NOTEBOOK_13 = PROJECT_ROOT / "results/notebooks/notebook_13"


AUDIT_DIR = (
    PROJECT_ROOT
    / "results/notebooks/notebook_22/section_08_sppgan_audit_final"
)

AUDIT_DIR.mkdir(parents=True, exist_ok=True)


DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us"
]

EXPECTED_REPETITIONS = 5

EXPECTED_SEEDS = [
    3026, 3027, 3028, 3029, 3030
]

EXPECTED_MODEL_COUNT = 15

EXPECTED_ROWS = {
    "adult_income": 34189,
    "bank_marketing": 31647,
    "diabetes_130us": 71236
}

EXPECTED_COLUMNS = {
    "adult_income": 15,
    "bank_marketing": 17,
    "diabetes_130us": 48
}

EXPECTED_TARGETS = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted"
}

EXPECTED_TRANSFORMED_DIMS = {
    "adult_income": 105,
    "bank_marketing": 51,
    "diabetes_130us": 2329
}

EXPECTED_LATENT_DIM = 128


# Historical hashes from the previously verified canonical CSV files.
EXPECTED_SYNTHETIC_HASHES = {
    "adult_income":
        "8db2fff4a874767e2620a39e2081b71edd06226d6f206536de7d5e257c00c9cf",

    "bank_marketing":
        "ba2825bf7e372d64c5c5b14d70700305e4d88617532b6fcc419a0dc2e9e70936",

    "diabetes_130us":
        "d8ccfde666814d01e740e8cef0338639b73ef21b2d2a38b086103d99bcfd0f26"
}


# Actual Notebook 12 artifact paths.
MODEL_REGISTRY_PATH = (
    NOTEBOOK_12
    / "registries/notebook_12B_final_model_registry.csv"
)

TRAINING_METADATA_PATH = (
    NOTEBOOK_12
    / "registries/notebook_12B_training_metadata.csv"
)

TRAINING_MANIFEST_PATH = (
    NOTEBOOK_12
    / "metadata/notebook_12B_final_manifest.json"
)

TRAINING_METADATA_MANIFEST_PATH = (
    NOTEBOOK_12
    / "metadata/notebook_12B_training_metadata_manifest.json"
)

ARCHITECTURE_PATH = (
    NOTEBOOK_12
    / "metadata/notebook_08_architecture_snapshot.json"
)

TRAINING_CONFIG_PATH = (
    NOTEBOOK_12
    / "configuration/sppgan_training_configuration.json"
)


# Actual Notebook 13 generation evidence paths.
GENERATION_LOG_PATH = (
    NOTEBOOK_13
    / "logs/section_08_generation_log.csv"
)

GENERATION_SUMMARY_PATH = (
    NOTEBOOK_13
    / "logs/section_08_generation_summary.json"
)

GENERATION_SEED_PLAN_PATH = (
    NOTEBOOK_13
    / "metadata/generation_seed_plan.csv"
)

GENERATION_CONFIG_PATH = (
    NOTEBOOK_13
    / "configuration/sppgan_generation_configuration.json"
)

GENERATION_METADATA_PATH = (
    NOTEBOOK_13
    / "metadata/sppgan_synthetic_generation_metadata.json"
)


# Actual canonical synthetic outputs.
SYNTHETIC_PATHS = {
    dataset: (
        NOTEBOOK_13
        / "synthetic_datasets"
        / f"{dataset}_sppgan_synthetic.csv"
    )
    for dataset in DATASETS
}


SECTION_8_STARTED = datetime.now(
    timezone.utc
).isoformat()


print("=" * 100)
print("NOTEBOOK 22 — SECTION 8: VERIFY SPP-GAN ARTIFACTS")
print("=" * 100)

print("Project root:", PROJECT_ROOT)
print("Datasets:", DATASETS)
print("Expected repetitions:", EXPECTED_REPETITIONS)
print("Expected seeds:", EXPECTED_SEEDS)
print("Expected final models:", EXPECTED_MODEL_COUNT)
print("Audit directory:", AUDIT_DIR)


if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(
        f"Project root not found: {PROJECT_ROOT}"
    )

if not NOTEBOOK_12.is_dir():
    raise FileNotFoundError(
        f"Notebook 12 directory not found: {NOTEBOOK_12}"
    )

if not NOTEBOOK_13.is_dir():
    raise FileNotFoundError(
        f"Notebook 13 directory not found: {NOTEBOOK_13}"
    )


# =============================================================================
# 2. AUDIT UTILITIES
# =============================================================================

AUDIT_RESULTS = []


def record_check(
    category,
    name,
    status,
    details="",
    artifact_path=""
):
    AUDIT_RESULTS.append({
        "category": category,
        "check_name": name,
        "status": status,
        "details": json.dumps(
            details,
            default=str
        ) if isinstance(details, (dict, list)) else str(details),
        "artifact_path": str(artifact_path),
        "timestamp_utc": datetime.now(
            timezone.utc
        ).isoformat()
    })


def load_csv(path):
    path = Path(path)

    if not path.is_file() or path.stat().st_size == 0:
        return None

    try:
        return pd.read_csv(path, low_memory=False)

    except pd.errors.EmptyDataError:
        print("EMPTY CSV:", path)
        return None

    except Exception as exc:
        print("CSV ERROR:", path, exc)
        return None


def load_json(path):
    path = Path(path)

    if not path.is_file() or path.stat().st_size == 0:
        return None

    try:
        with open(path, "r", encoding="utf-8") as file:
            return json.load(file)

    except Exception as exc:
        print("JSON ERROR:", path, exc)
        return None


def save_json(path, data):
    with open(path, "w", encoding="utf-8") as file:
        json.dump(
            data,
            file,
            indent=2,
            default=str,
            ensure_ascii=False
        )


def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


def normalize(value):
    return re.sub(
        r"[^a-z0-9]+",
        "_",
        str(value).strip().lower()
    ).strip("_")


def find_column(df, candidates):
    if df is None:
        return None

    lookup = {
        normalize(col): col
        for col in df.columns
    }

    for candidate in candidates:
        key = normalize(candidate)

        if key in lookup:
            return lookup[key]

    return None


def safe_int(value):
    try:
        if pd.isna(value):
            return None

        number = float(value)

        return int(number) if number.is_integer() else None

    except (TypeError, ValueError):
        return None


def safe_float(value):
    try:
        if pd.isna(value):
            return None

        return float(value)

    except (TypeError, ValueError):
        return None


def resolve_path(value):
    if value is None or pd.isna(value):
        return None

    raw = Path(str(value).strip())

    candidates = [
        raw,
        PROJECT_ROOT / raw,
        NOTEBOOK_12 / raw
    ]

    for candidate in candidates:

        if candidate.is_file():
            return candidate.resolve()

    # A unique filename match is acceptable only when unambiguous.
    matches = list(NOTEBOOK_12.rglob(raw.name))

    if len(matches) == 1:
        return matches[0].resolve()

    return None


def recursive_values(obj, target_key):
    values = []

    if isinstance(obj, dict):

        for key, value in obj.items():

            if normalize(key) == normalize(target_key):
                values.append(value)

            values.extend(
                recursive_values(value, target_key)
            )

    elif isinstance(obj, list):

        for item in obj:
            values.extend(
                recursive_values(item, target_key)
            )

    return values


def extract_numeric_values(values):
    output = []

    for value in values:

        number = safe_float(value)

        if number is not None:
            output.append(number)

    return output


def dataset_scope(obj, dataset):
    """
    Return only explicitly dataset-keyed architecture evidence.
    Global values are not accepted as dataset-specific evidence.
    """

    matches = []

    if isinstance(obj, dict):

        for key, value in obj.items():

            if normalize(key) == normalize(dataset):
                matches.append(value)

            if isinstance(value, (dict, list)):
                matches.extend(
                    dataset_scope(value, dataset)
                )

    elif isinstance(obj, list):

        for item in obj:

            if isinstance(item, dict):

                name = (
                    item.get("dataset")
                    or item.get("dataset_name")
                )

                if (
                    name is not None
                    and normalize(name) == normalize(dataset)
                ):
                    matches.append(item)

                matches.extend(
                    dataset_scope(item, dataset)
                )

    return matches


def status_value(value):
    if value is None or pd.isna(value):
        return "UNCLASSIFIED"

    value = str(value).strip().upper()

    if value in {
        "PASS", "SUCCESS", "COMPLETED",
        "GENERATED", "TRAINING_COMPLETE"
    }:
        return "SUCCESS"

    if value in {
        "FAIL", "FAILED", "ERROR",
        "GENERATION_FAILED"
    }:
        return "FAILED"

    return "UNCLASSIFIED"


def write_csv(df, filename):
    df.to_csv(
        AUDIT_DIR / filename,
        index=False
    )


# =============================================================================
# 3. LOAD AUTHORITATIVE NOTEBOOK 12 ARTIFACTS
# =============================================================================

model_registry = load_csv(
    MODEL_REGISTRY_PATH
)

training_metadata = load_csv(
    TRAINING_METADATA_PATH
)

training_manifest = load_json(
    TRAINING_MANIFEST_PATH
)

training_metadata_manifest = load_json(
    TRAINING_METADATA_MANIFEST_PATH
)

architecture_snapshot = load_json(
    ARCHITECTURE_PATH
)

training_configuration = load_json(
    TRAINING_CONFIG_PATH
)


notebook12_artifacts = {
    "final_model_registry": MODEL_REGISTRY_PATH,
    "training_metadata": TRAINING_METADATA_PATH,
    "training_manifest": TRAINING_MANIFEST_PATH,
    "training_metadata_manifest": TRAINING_METADATA_MANIFEST_PATH,
    "architecture_snapshot": ARCHITECTURE_PATH,
    "training_configuration": TRAINING_CONFIG_PATH
}


for name, path in notebook12_artifacts.items():

    valid = (
        path.is_file()
        and path.stat().st_size > 0
    )

    record_check(
        "NOTEBOOK_12",
        name,
        "PASS" if valid else "FAIL",
        {"exists": valid},
        path
    )


if model_registry is None or training_metadata is None:
    raise RuntimeError(
        "Required Notebook 12 registry or training metadata could not be loaded."
    )


print("\nNOTEBOOK 12 SUMMARY")
print("-" * 100)

print("Final model registry rows:", len(model_registry))
print("Training metadata rows:", len(training_metadata))

print(
    "Training manifest status:",
    training_manifest.get("status", "NOT SPECIFIED")
    if isinstance(training_manifest, dict)
    else "NOT AVAILABLE"
)

print("\nFINAL MODEL REGISTRY COLUMNS")
print(model_registry.columns.tolist())

print("\nTRAINING METADATA COLUMNS")
print(training_metadata.columns.tolist())


# =============================================================================
# 4. VERIFY FINAL MODEL REGISTRY STRUCTURE
# =============================================================================

REGISTRY_COLUMNS = {
    "dataset": "dataset",
    "repetition": "repetition_id",
    "status": "status",
    "final_model_path": "final_model_path",
    "final_model_hash": "final_model_sha256",
    "final_model_size": "final_model_size_bytes",
    "source_checkpoint": "source_checkpoint",
    "source_checkpoint_hash": "source_checkpoint_sha256"
}

missing_registry_columns = [
    column
    for column in REGISTRY_COLUMNS.values()
    if column not in model_registry.columns
]


record_check(
    "NOTEBOOK_12",
    "actual_final_registry_schema",
    "PASS" if not missing_registry_columns else "FAIL",
    {
        "required_columns": list(REGISTRY_COLUMNS.values()),
        "missing_columns": missing_registry_columns
    },
    MODEL_REGISTRY_PATH
)


if missing_registry_columns:
    raise RuntimeError(
        "Final model registry is missing required columns: "
        + str(missing_registry_columns)
    )


# =============================================================================
# 5. VERIFY ALL 15 FINAL MODEL FILES
# =============================================================================

final_model_records = []

for index, row in model_registry.iterrows():

    dataset = str(row["dataset"]).strip().lower()

    repetition = safe_int(
        row["repetition_id"]
    )

    model_path = resolve_path(
        row["final_model_path"]
    )

    expected_hash = str(
        row["final_model_sha256"]
    ).strip().lower()

    expected_size = safe_int(
        row["final_model_size_bytes"]
    )

    actual_hash = None
    actual_size = None

    exists = model_path is not None

    if exists:

        actual_hash = sha256_file(model_path)
        actual_size = model_path.stat().st_size

    hash_match = (
        actual_hash == expected_hash
        if actual_hash and re.fullmatch(r"[a-f0-9]{64}", expected_hash)
        else None
    )

    size_match = (
        actual_size == expected_size
        if actual_size is not None and expected_size is not None
        else None
    )

    if not exists:

        status = "FAIL"

    elif hash_match is False or size_match is False:

        status = "FAIL"

    elif hash_match is None or size_match is None:

        status = "REVIEW_REQUIRED"

    else:

        status = "PASS"

    result = {
        "dataset": dataset,
        "repetition_id": repetition,
        "registry_status": row["status"],
        "final_model_path": str(model_path) if model_path else str(row["final_model_path"]),
        "exists": exists,
        "expected_sha256": expected_hash,
        "actual_sha256": actual_hash,
        "hash_match": hash_match,
        "expected_size_bytes": expected_size,
        "actual_size_bytes": actual_size,
        "size_match": size_match,
        "status": status
    }

    final_model_records.append(result)

    record_check(
        "FINAL_MODELS",
        f"{dataset}_repetition_{repetition}",
        status,
        result,
        model_path if model_path else row["final_model_path"]
    )


final_model_df = pd.DataFrame(
    final_model_records
)

write_csv(
    final_model_df,
    "section_08_final_model_integrity.csv"
)


# =============================================================================
# 6. VERIFY SOURCE CHECKPOINTS
# =============================================================================

source_checkpoint_records = []

for index, row in model_registry.iterrows():

    dataset = str(row["dataset"]).strip().lower()

    repetition = safe_int(
        row["repetition_id"]
    )

    checkpoint_path = resolve_path(
        row["source_checkpoint"]
    )

    expected_hash = str(
        row["source_checkpoint_sha256"]
    ).strip().lower()

    exists = checkpoint_path is not None

    actual_hash = (
        sha256_file(checkpoint_path)
        if exists else None
    )

    hash_match = (
        actual_hash == expected_hash
        if actual_hash and re.fullmatch(r"[a-f0-9]{64}", expected_hash)
        else None
    )

    if not exists:

        status = "FAIL"

    elif hash_match is False:

        status = "FAIL"

    elif hash_match is None:

        status = "REVIEW_REQUIRED"

    else:

        status = "PASS"

    result = {
        "dataset": dataset,
        "repetition_id": repetition,
        "source_checkpoint": str(checkpoint_path) if checkpoint_path else str(row["source_checkpoint"]),
        "exists": exists,
        "expected_sha256": expected_hash,
        "actual_sha256": actual_hash,
        "hash_match": hash_match,
        "status": status
    }

    source_checkpoint_records.append(result)

    record_check(
        "SOURCE_CHECKPOINTS",
        f"{dataset}_repetition_{repetition}",
        status,
        result,
        checkpoint_path if checkpoint_path else row["source_checkpoint"]
    )


source_checkpoint_df = pd.DataFrame(
    source_checkpoint_records
)

write_csv(
    source_checkpoint_df,
    "section_08_source_checkpoint_integrity.csv"
)


print("\nFINAL MODEL INTEGRITY")
print("-" * 100)

print("Final model records:", len(final_model_df))
print("Existing final models:", int(final_model_df["exists"].sum()))
print("Final model hash matches:", int(final_model_df["hash_match"].eq(True).sum()))
print("Final model size matches:", int(final_model_df["size_match"].eq(True).sum()))

print("\nSOURCE CHECKPOINT INTEGRITY")
print("-" * 100)

print("Source checkpoint records:", len(source_checkpoint_df))
print("Existing source checkpoints:", int(source_checkpoint_df["exists"].sum()))
print("Source checkpoint hash matches:", int(source_checkpoint_df["hash_match"].eq(True).sum()))


# =============================================================================
# 7. TRAINING REPETITION AND SEED COVERAGE
# =============================================================================

required_training_columns = [
    "dataset",
    "repetition_id",
    "experiment_seed",
    "status"
]

missing_training_columns = [
    column
    for column in required_training_columns
    if column not in training_metadata.columns
]


record_check(
    "TRAINING_COVERAGE",
    "actual_training_metadata_schema",
    "PASS" if not missing_training_columns else "FAIL",
    {
        "required_columns": required_training_columns,
        "missing_columns": missing_training_columns
    },
    TRAINING_METADATA_PATH
)


if missing_training_columns:
    raise RuntimeError(
        "Training metadata is missing required fields: "
        + str(missing_training_columns)
    )


training_coverage_records = []

for dataset in DATASETS:

    subset = training_metadata[
        training_metadata["dataset"]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq(dataset)
    ].copy()

    subset["repetition_id"] = pd.to_numeric(
        subset["repetition_id"],
        errors="coerce"
    )

    subset["experiment_seed"] = pd.to_numeric(
        subset["experiment_seed"],
        errors="coerce"
    )

    repetitions = sorted(
        subset["repetition_id"].dropna().astype(int).unique().tolist()
    )

    seeds = sorted(
        subset["experiment_seed"].dropna().astype(int).unique().tolist()
    )

    successful = subset[
        subset["status"].astype(str).str.upper().isin(
            ["SUCCESS", "PASS", "COMPLETED"]
        )
    ]

    repetition_match = (
        len(subset) == EXPECTED_REPETITIONS
        and repetitions == [1, 2, 3, 4, 5]
    )

    seed_match = (
        seeds == EXPECTED_SEEDS
    )

    status_match = (
        len(successful) == EXPECTED_REPETITIONS
    )

    coverage_status = (
        "PASS"
        if repetition_match and seed_match and status_match
        else "FAIL"
    )

    result = {
        "dataset": dataset,
        "metadata_rows": len(subset),
        "observed_repetitions": json.dumps(repetitions),
        "expected_repetitions": EXPECTED_REPETITIONS,
        "observed_seeds": json.dumps(seeds),
        "expected_seeds": json.dumps(EXPECTED_SEEDS),
        "successful_training_runs": len(successful),
        "repetition_match": repetition_match,
        "seed_match": seed_match,
        "successful_status_match": status_match,
        "status": coverage_status
    }

    training_coverage_records.append(result)

    record_check(
        "TRAINING_COVERAGE",
        f"{dataset}_repetition_seed_coverage",
        coverage_status,
        result,
        TRAINING_METADATA_PATH
    )


training_coverage_df = pd.DataFrame(
    training_coverage_records
)

write_csv(
    training_coverage_df,
    "section_08_training_coverage.csv"
)


print("\nTRAINING REPETITION AND SEED COVERAGE")
print("-" * 100)
print(training_coverage_df.to_string(index=False))


# =============================================================================
# 8. DATASET-SPECIFIC ARCHITECTURE VERIFICATION
# =============================================================================

architecture_records = []


for dataset in DATASETS:

    scope = dataset_scope(
        architecture_snapshot or {},
        dataset
    )

    # Include only the matching dataset's own training metadata rows.
    subset = training_metadata[
        training_metadata["dataset"]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq(dataset)
    ]

    evidence = {
        "dataset_architecture_snapshot": scope,
        "dataset_training_records": subset.to_dict("records")
    }

    transformed_values = extract_numeric_values(
        recursive_values(evidence, "transformed_dim")
    )

    latent_values = extract_numeric_values(
        recursive_values(evidence, "latent_dim")
    )

    transformed_match = (
        EXPECTED_TRANSFORMED_DIMS[dataset]
        in transformed_values
    )

    latent_match = (
        EXPECTED_LATENT_DIM
        in latent_values
    )

    metadata_available = (
        bool(scope)
        or not subset.empty
    )

    status = (
        "PASS"
        if metadata_available and transformed_match and latent_match
        else "REVIEW_REQUIRED"
    )

    result = {
        "dataset": dataset,
        "expected_transformed_dim": EXPECTED_TRANSFORMED_DIMS[dataset],
        "observed_transformed_dims": json.dumps(transformed_values),
        "transformed_dim_match": transformed_match,
        "expected_latent_dim": EXPECTED_LATENT_DIM,
        "observed_latent_dims": json.dumps(latent_values),
        "latent_dim_match": latent_match,
        "dataset_specific_metadata_available": metadata_available,
        "status": status
    }

    architecture_records.append(result)

    record_check(
        "ARCHITECTURE",
        f"{dataset}_architecture",
        status,
        result,
        ARCHITECTURE_PATH
    )


architecture_df = pd.DataFrame(
    architecture_records
)

write_csv(
    architecture_df,
    "section_08_architecture_verification.csv"
)


print("\nARCHITECTURE VERIFICATION")
print("-" * 100)
print(architecture_df.to_string(index=False))


# =============================================================================
# 9. NOTEBOOK 13 GENERATION SUMMARY
# =============================================================================

generation_summary = load_json(
    GENERATION_SUMMARY_PATH
)

generation_log = load_csv(
    GENERATION_LOG_PATH
)

generation_seed_plan = load_csv(
    GENERATION_SEED_PLAN_PATH
)

generation_configuration = load_json(
    GENERATION_CONFIG_PATH
)

generation_metadata = load_json(
    GENERATION_METADATA_PATH
)


for name, path, obj in [
    ("generation_summary", GENERATION_SUMMARY_PATH, generation_summary),
    ("generation_log", GENERATION_LOG_PATH, generation_log),
    ("generation_seed_plan", GENERATION_SEED_PLAN_PATH, generation_seed_plan)
]:

    record_check(
        "GENERATION",
        name,
        "PASS" if obj is not None else "REVIEW_REQUIRED",
        {"loaded": obj is not None},
        path
    )


print("\nGENERATION SUMMARY")
print("-" * 100)

if isinstance(generation_summary, dict):

    for key, value in generation_summary.items():
        print(f"{key}: {value}")

else:

    print("Generation summary unavailable.")


# =============================================================================
# 10. GENERATION LOG CLASSIFICATION
# =============================================================================

generation_records = []
generation_failures = []
generation_successes = []
generation_unclassified = []


if generation_log is not None:

    required_generation_columns = [
        "dataset",
        "repetition",
        "seed",
        "model_path",
        "model_sha256",
        "output_path",
        "status",
        "error"
    ]

    missing_generation_columns = [
        col for col in required_generation_columns
        if col not in generation_log.columns
    ]

    record_check(
        "GENERATION",
        "generation_log_schema",
        "PASS" if not missing_generation_columns else "REVIEW_REQUIRED",
        {
            "columns": generation_log.columns.tolist(),
            "missing_columns": missing_generation_columns
        },
        GENERATION_LOG_PATH
    )

    for index, row in generation_log.iterrows():

        explicit_status = status_value(
            row.get("status")
        )

        error = (
            str(row["error"]).strip()
            if "error" in generation_log.columns
            and pd.notna(row["error"])
            else ""
        )

        if explicit_status == "FAILED" or error:

            classification = "FAILED"

        elif explicit_status == "SUCCESS":

            classification = "SUCCESS"

        else:

            classification = "UNCLASSIFIED"

        record = {
            "row_index": int(index),
            "dataset": row.get("dataset"),
            "repetition": safe_int(row.get("repetition")),
            "seed": safe_int(row.get("seed")),
            "model_path": row.get("model_path"),
            "model_sha256": row.get("model_sha256"),
            "output_path": row.get("output_path"),
            "status": explicit_status,
            "error": error,
            "classification": classification
        }

        generation_records.append(record)

        if classification == "FAILED":
            generation_failures.append(record)

        elif classification == "SUCCESS":
            generation_successes.append(record)

        else:
            generation_unclassified.append(record)


generation_records_df = pd.DataFrame(generation_records)
generation_failure_df = pd.DataFrame(generation_failures)
generation_success_df = pd.DataFrame(generation_successes)
generation_unclassified_df = pd.DataFrame(generation_unclassified)


write_csv(
    generation_records_df,
    "section_08_generation_records.csv"
)

write_csv(
    generation_failure_df,
    "section_08_generation_failure_records.csv"
)

write_csv(
    generation_success_df,
    "section_08_generation_success_records.csv"
)

write_csv(
    generation_unclassified_df,
    "section_08_generation_unclassified_records.csv"
)


print("\nGENERATION EXECUTION AUDIT")
print("-" * 100)

print(
    "Generation records:",
    len(generation_records_df)
)

print(
    "Successful generation records:",
    len(generation_success_df)
)

print(
    "Failed generation records:",
    len(generation_failure_df)
)

print(
    "Unclassified generation records:",
    len(generation_unclassified_df)
)


# =============================================================================
# 11. RECONCILE GENERATION SUMMARY AND EXECUTION LOG
# =============================================================================

summary_expected = (
    safe_int(generation_summary.get("expected_generations"))
    if isinstance(generation_summary, dict)
    else None
)

summary_successful = (
    safe_int(generation_summary.get("successful_generations"))
    if isinstance(generation_summary, dict)
    else None
)

summary_failed = (
    safe_int(generation_summary.get("failed_generations"))
    if isinstance(generation_summary, dict)
    else None
)


generation_summary_match = (
    summary_expected == len(generation_records_df)
    and summary_successful == len(generation_success_df)
    and summary_failed == len(generation_failure_df)
) if summary_expected is not None else False


generation_log_datasets = (
    sorted(
        generation_log["dataset"]
        .dropna()
        .astype(str)
        .str.lower()
        .unique()
        .tolist()
    )
    if generation_log is not None
    and "dataset" in generation_log.columns
    else []
)


generation_log_seeds = (
    sorted(
        pd.to_numeric(
            generation_log["seed"],
            errors="coerce"
        ).dropna().astype(int).unique().tolist()
    )
    if generation_log is not None
    and "seed" in generation_log.columns
    else []
)


generation_provenance = {
    "summary_expected_generations": summary_expected,
    "summary_successful_generations": summary_successful,
    "summary_failed_generations": summary_failed,
    "log_rows": len(generation_records_df),
    "log_successful": len(generation_success_df),
    "log_failed": len(generation_failure_df),
    "log_unclassified": len(generation_unclassified_df),
    "summary_and_log_match": generation_summary_match,
    "datasets_in_log": generation_log_datasets,
    "seeds_in_log": generation_log_seeds,
    "expected_training_seeds": EXPECTED_SEEDS
}


if generation_summary_match:

    generation_reconciliation_status = "PASS"

else:

    generation_reconciliation_status = "REVIEW_REQUIRED"


record_check(
    "GENERATION",
    "generation_summary_log_reconciliation",
    generation_reconciliation_status,
    generation_provenance,
    GENERATION_SUMMARY_PATH
)


save_json(
    AUDIT_DIR / "section_08_generation_reconciliation.json",
    generation_provenance
)


# =============================================================================
# 12. VERIFY CANONICAL SYNTHETIC CSV FILES
# =============================================================================

synthetic_records = []


for dataset in DATASETS:

    csv_path = SYNTHETIC_PATHS[dataset]

    result = {
        "dataset": dataset,
        "path": str(csv_path),
        "exists": csv_path.is_file(),
        "rows": None,
        "columns": None,
        "expected_rows": EXPECTED_ROWS[dataset],
        "expected_columns": EXPECTED_COLUMNS[dataset],
        "row_match": False,
        "column_match": False,
        "target_present": False,
        "actual_sha256": None,
        "expected_sha256": EXPECTED_SYNTHETIC_HASHES[dataset],
        "hash_match": None,
        "size_bytes": None,
        "status": "FAIL"
    }

    if csv_path.is_file():

        df = load_csv(csv_path)

        if df is not None:

            actual_hash = sha256_file(csv_path)

            row_match = (
                len(df) == EXPECTED_ROWS[dataset]
            )

            column_match = (
                len(df.columns) == EXPECTED_COLUMNS[dataset]
            )

            target_present = (
                EXPECTED_TARGETS[dataset] in df.columns
            )

            hash_match = (
                actual_hash == EXPECTED_SYNTHETIC_HASHES[dataset]
            )

            result.update({
                "rows": len(df),
                "columns": len(df.columns),
                "row_match": row_match,
                "column_match": column_match,
                "target_present": target_present,
                "actual_sha256": actual_hash,
                "hash_match": hash_match,
                "size_bytes": csv_path.stat().st_size
            })

            if (
                row_match
                and column_match
                and target_present
                and hash_match
            ):

                result["status"] = "PASS"

            else:

                result["status"] = "FAIL"

    synthetic_records.append(result)

    record_check(
        "SYNTHETIC_DATA",
        f"{dataset}_synthetic_csv",
        result["status"],
        result,
        csv_path
    )


synthetic_df = pd.DataFrame(
    synthetic_records
)

write_csv(
    synthetic_df,
    "section_08_synthetic_data_integrity.csv"
)


print("\nSYNTHETIC DATA INTEGRITY")
print("-" * 100)
print(synthetic_df.to_string(index=False))


# =============================================================================
# 13. GENERATION OUTPUT PATH RECONCILIATION
# =============================================================================

output_path_records = []

if generation_log is not None and "output_path" in generation_log.columns:

    for _, row in generation_log.iterrows():

        raw_path = row.get("output_path")

        resolved = resolve_path(raw_path)

        output_path_records.append({
            "dataset": row.get("dataset"),
            "repetition": row.get("repetition"),
            "seed": row.get("seed"),
            "logged_output_path": raw_path,
            "resolved_output_path": str(resolved) if resolved else None,
            "output_exists": resolved is not None,
            "generation_status": row.get("status"),
            "error": row.get("error")
        })


generation_output_df = pd.DataFrame(
    output_path_records
)

write_csv(
    generation_output_df,
    "section_08_generation_output_path_audit.csv"
)


# =============================================================================
# 14. RESTORATION EVIDENCE AUDIT
# =============================================================================

RESTORATION_ROOT = NOTEBOOK_13 / "logs"

restoration_names = [
    "feature_type_restoration_audit.csv",
    "target_restoration_errors.csv",
    "target_restoration_metadata.csv",
    "target_restoration_summary.json",
    "target_restoration_metadata.json"
]

restoration_records = []


for filename in restoration_names:

    path = RESTORATION_ROOT / filename

    if not path.is_file():

        status = "MISSING"
        rows = None
        columns = None

    elif path.stat().st_size == 0:

        status = "EMPTY_REVIEW_REQUIRED"
        rows = None
        columns = None

    elif path.suffix.lower() == ".csv":

        df = load_csv(path)

        if df is None:

            status = "UNREADABLE_REVIEW_REQUIRED"
            rows = None
            columns = None

        else:

            status = "READABLE"
            rows = len(df)
            columns = len(df.columns)

    else:

        obj = load_json(path)

        status = (
            "READABLE"
            if obj is not None
            else "UNREADABLE_REVIEW_REQUIRED"
        )

        rows = None
        columns = None

    restoration_records.append({
        "file_name": filename,
        "path": str(path),
        "status": status,
        "rows": rows,
        "columns": columns,
        "size_bytes": path.stat().st_size if path.is_file() else None
    })


restoration_df = pd.DataFrame(
    restoration_records
)

write_csv(
    restoration_df,
    "section_08_restoration_evidence_audit.csv"
)


print("\nRESTORATION EVIDENCE")
print("-" * 100)
print(restoration_df.to_string(index=False))


# =============================================================================
# 15. DIFFERENTIAL PRIVACY AUDIT USING ACTUAL REGISTRY FIELDS
# =============================================================================

privacy_columns = [
    "achieved_epsilon",
    "target_epsilon",
    "dp_noise_multiplier",
    "dp_delta",
    "privacy_scope",
    "generator_private",
    "statistical_guidance_private",
    "preprocessing_private",
    "end_to_end_privacy_claim"
]


missing_privacy_columns = [
    column
    for column in privacy_columns
    if column not in model_registry.columns
]


privacy_records = []

for _, row in model_registry.iterrows():

    privacy_records.append({
        "dataset": row["dataset"],
        "repetition_id": row["repetition_id"],
        "achieved_epsilon": row.get("achieved_epsilon"),
        "target_epsilon": row.get("target_epsilon"),
        "dp_noise_multiplier": row.get("dp_noise_multiplier"),
        "dp_delta": row.get("dp_delta"),
        "privacy_scope": row.get("privacy_scope"),
        "generator_private": row.get("generator_private"),
        "statistical_guidance_private": row.get("statistical_guidance_private"),
        "preprocessing_private": row.get("preprocessing_private"),
        "end_to_end_privacy_claim": row.get("end_to_end_privacy_claim")
    })


privacy_df = pd.DataFrame(
    privacy_records
)

write_csv(
    privacy_df,
    "section_08_privacy_evidence.csv"
)


if not missing_privacy_columns:

    epsilon_values = pd.to_numeric(
        privacy_df["achieved_epsilon"],
        errors="coerce"
    )

    delta_values = pd.to_numeric(
        privacy_df["dp_delta"],
        errors="coerce"
    )

    noise_values = pd.to_numeric(
        privacy_df["dp_noise_multiplier"],
        errors="coerce"
    )

    epsilon_complete = (
        epsilon_values.notna().all()
    )

    delta_complete = (
        delta_values.notna().all()
    )

    noise_complete = (
        noise_values.notna().all()
    )

    privacy_scope_values = (
        privacy_df["privacy_scope"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    generator_private_values = (
        privacy_df["generator_private"]
        .dropna()
        .astype(str)
        .str.lower()
        .unique()
        .tolist()
    )

    end_to_end_values = (
        privacy_df["end_to_end_privacy_claim"]
        .dropna()
        .astype(str)
        .str.lower()
        .unique()
        .tolist()
    )

else:

    epsilon_complete = False
    delta_complete = False
    noise_complete = False
    privacy_scope_values = []
    generator_private_values = []
    end_to_end_values = []


privacy_audit = {
    "registry_records": len(privacy_df),
    "epsilon_complete": bool(epsilon_complete),
    "delta_complete": bool(delta_complete),
    "noise_multiplier_complete": bool(noise_complete),
    "privacy_scope_values": privacy_scope_values,
    "generator_private_values": generator_private_values,
    "end_to_end_privacy_claim_values": end_to_end_values,
    "missing_privacy_columns": missing_privacy_columns,
    "formal_parameter_records_complete": (
        bool(epsilon_complete)
        and bool(delta_complete)
        and bool(noise_complete)
    ),
    "end_to_end_dp_supported": False,
    "interpretation": (
        "The recorded privacy scope is critic/discriminator-only. "
        "The generator, statistical guidance and preprocessing are "
        "not represented as private components. Do not claim end-to-end "
        "differential privacy."
    )
}


privacy_status = (
    "PASS"
    if (
        privacy_audit["formal_parameter_records_complete"]
        and len(privacy_df) == EXPECTED_MODEL_COUNT
    )
    else "REVIEW_REQUIRED"
)


save_json(
    AUDIT_DIR / "section_08_privacy_scope_audit.json",
    privacy_audit
)


record_check(
    "PRIVACY",
    "registry_privacy_evidence",
    privacy_status,
    privacy_audit,
    MODEL_REGISTRY_PATH
)


print("\nPRIVACY AUDIT")
print("-" * 100)
print(json.dumps(privacy_audit, indent=2, default=str))


# =============================================================================
# 16. GENERATOR IMPLEMENTATION SOURCE AUDIT
# =============================================================================

source_roots = [
    NOTEBOOK_09,
    NOTEBOOK_10,
    NOTEBOOK_12,
    NOTEBOOK_13
]

source_files = []

for root in source_roots:

    if root.is_dir():

        source_files.extend(
            list(root.rglob("*.py"))
        )

        source_files.extend(
            list(root.rglob("*.ipynb"))
        )


source_files = sorted(set(source_files))

generator_matches = []

for path in source_files:

    try:

        content = path.read_text(
            encoding="utf-8",
            errors="ignore"
        ).lower()

        if (
            "class sppgangenerator" in content
            or "sppgangenerator" in content
        ):

            generator_matches.append(path)

    except Exception:
        continue


source_status = (
    "PASS"
    if generator_matches
    else "REVIEW_REQUIRED"
)


source_inventory = pd.DataFrame([
    {
        "path": str(path),
        "file_name": path.name,
        "extension": path.suffix,
        "size_bytes": path.stat().st_size
    }
    for path in source_files
])


write_csv(
    source_inventory,
    "section_08_source_inventory.csv"
)


record_check(
    "IMPLEMENTATION",
    "generator_source_availability",
    source_status,
    {
        "source_files_discovered": len(source_files),
        "generator_source_matches": [
            str(path) for path in generator_matches
        ]
    }
)


print("\nGENERATOR SOURCE AUDIT")
print("-" * 100)
print("Source files discovered:", len(source_files))

for path in generator_matches:
    print("Generator source:", path)

if not generator_matches:
    print(
        "Original generator class was not located. "
        "Checkpoint presence does not establish source-code availability."
    )


# =============================================================================
# 17. PROVENANCE RECONCILIATION
# =============================================================================

final_models_verified = (
    len(final_model_df) == EXPECTED_MODEL_COUNT
    and bool(final_model_df["hash_match"].eq(True).all())
    and bool(final_model_df["size_match"].eq(True).all())
)

source_checkpoints_verified = (
    len(source_checkpoint_df) == EXPECTED_MODEL_COUNT
    and bool(source_checkpoint_df["hash_match"].eq(True).all())
)

training_coverage_pass = (
    len(training_coverage_df) == len(DATASETS)
    and bool(training_coverage_df["status"].eq("PASS").all())
)

synthetic_outputs_verified = (
    len(synthetic_df) == len(DATASETS)
    and bool(synthetic_df["status"].eq("PASS").all())
)

generation_runs_all_successful = (
    len(generation_success_df) == EXPECTED_MODEL_COUNT
    and len(generation_failure_df) == 0
    and len(generation_unclassified_df) == 0
)


provenance_review_items = []

if not final_models_verified:
    provenance_review_items.append(
        "Final model hash or size verification incomplete."
    )

if not source_checkpoints_verified:
    provenance_review_items.append(
        "Source checkpoint hash verification incomplete."
    )

if not training_coverage_pass:
    provenance_review_items.append(
        "Training repetitions or experiment seeds do not fully match."
    )

if not synthetic_outputs_verified:
    provenance_review_items.append(
        "One or more canonical synthetic CSV integrity checks failed."
    )

if not generation_runs_all_successful:
    provenance_review_items.append(
        "Not all 15 generation runs are explicitly successful."
    )

if not generation_summary_match:
    provenance_review_items.append(
        "Generation summary and execution log do not reconcile."
    )

if not generator_matches:
    provenance_review_items.append(
        "Original generator implementation source is unavailable."
    )


provenance_status = (
    "PASS"
    if not provenance_review_items
    else "REVIEW_REQUIRED"
)


provenance_report = {
    "expected_models": EXPECTED_MODEL_COUNT,
    "final_models_verified": final_models_verified,
    "source_checkpoints_verified": source_checkpoints_verified,
    "training_coverage_pass": training_coverage_pass,
    "synthetic_outputs_verified": synthetic_outputs_verified,
    "generation_runs_all_successful": generation_runs_all_successful,
    "generation_summary_matches_log": generation_summary_match,
    "generator_source_available": bool(generator_matches),
    "generation_summary_expected": summary_expected,
    "generation_summary_successful": summary_successful,
    "generation_summary_failed": summary_failed,
    "generation_log_successful": len(generation_success_df),
    "generation_log_failed": len(generation_failure_df),
    "generation_log_unclassified": len(generation_unclassified_df),
    "review_items": provenance_review_items,
    "status": provenance_status
}


save_json(
    AUDIT_DIR / "section_08_provenance_reconciliation.json",
    provenance_report
)


record_check(
    "PROVENANCE",
    "complete_provenance_reconciliation",
    provenance_status,
    provenance_report
)


print("\nPROVENANCE RECONCILIATION")
print("-" * 100)
print(json.dumps(provenance_report, indent=2, default=str))


# =============================================================================
# 18. CONSOLIDATED AUDIT RESULTS
# =============================================================================

audit_df = pd.DataFrame(
    AUDIT_RESULTS
)

write_csv(
    audit_df,
    "section_08_complete_audit_results.csv"
)


status_counts = (
    audit_df["status"]
    .value_counts()
    .to_dict()
)

pass_count = int(
    status_counts.get("PASS", 0)
)

fail_count = int(
    status_counts.get("FAIL", 0)
)

review_count = int(
    status_counts.get("REVIEW_REQUIRED", 0)
)


print("\n" + "=" * 100)
print("SECTION 8 CONSOLIDATED AUDIT")
print("=" * 100)

print("Total checks:", len(audit_df))
print("PASS:", pass_count)
print("FAIL:", fail_count)
print("REVIEW_REQUIRED:", review_count)

print("\nCATEGORY SUMMARY")

print(
    audit_df.groupby(
        ["category", "status"]
    ).size().unstack(fill_value=0).to_string()
)


# =============================================================================
# 19. STRICT FINAL VALIDATION GATE
# =============================================================================

critical_conditions = {
    "all_15_final_models_registered":
        len(model_registry) == EXPECTED_MODEL_COUNT,

    "all_15_final_model_files_exist":
        len(final_model_df) == EXPECTED_MODEL_COUNT
        and bool(final_model_df["exists"].all()),

    "all_15_final_model_hashes_match":
        final_models_verified,

    "all_15_source_checkpoints_exist":
        len(source_checkpoint_df) == EXPECTED_MODEL_COUNT
        and bool(source_checkpoint_df["exists"].all()),

    "all_15_source_checkpoint_hashes_match":
        source_checkpoints_verified,

    "all_three_datasets_have_five_correct_seeds":
        training_coverage_pass,

    "all_three_architecture_checks_pass":
        bool(architecture_df["status"].eq("PASS").all()),

    "all_three_synthetic_csvs_pass_integrity":
        synthetic_outputs_verified,

    "generation_summary_matches_execution_log":
        generation_summary_match,

    "all_15_generation_runs_successful":
        generation_runs_all_successful,

    "privacy_evidence_is_recorded":
        privacy_status == "PASS",

    "generator_source_available":
        bool(generator_matches)
}


print("\nCRITICAL VALIDATION CONDITIONS")
print("-" * 100)

for name, passed in critical_conditions.items():

    print(
        f"{name:<60}",
        "PASS" if passed else "NOT VERIFIED"
    )


all_critical_pass = all(
    critical_conditions.values()
)


if fail_count > 0:

    FINAL_SECTION_8_STATUS = "FAIL"

elif all_critical_pass and review_count == 0:

    FINAL_SECTION_8_STATUS = "PASS"

else:

    FINAL_SECTION_8_STATUS = "REVIEW_REQUIRED"


# =============================================================================
# 20. FINAL SECTION 8 REPORT
# =============================================================================

section_8_report = {
    "notebook": "Notebook 22",
    "section": "Section 8 — Verify SPP-GAN Artifacts",
    "audit_started_utc": SECTION_8_STARTED,
    "audit_completed_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "audit_directory": str(AUDIT_DIR),
    "datasets": DATASETS,
    "expected_repetitions": EXPECTED_REPETITIONS,
    "expected_seeds": EXPECTED_SEEDS,
    "expected_final_models": EXPECTED_MODEL_COUNT,
    "final_model_records": len(final_model_df),
    "final_model_files_existing": int(final_model_df["exists"].sum()),
    "final_model_hash_matches": int(final_model_df["hash_match"].eq(True).sum()),
    "final_model_size_matches": int(final_model_df["size_match"].eq(True).sum()),
    "source_checkpoint_records": len(source_checkpoint_df),
    "source_checkpoint_hash_matches": int(source_checkpoint_df["hash_match"].eq(True).sum()),
    "training_coverage_pass": training_coverage_pass,
    "architecture_checks_pass": int(architecture_df["status"].eq("PASS").sum()),
    "synthetic_outputs_found": int(synthetic_df["exists"].sum()),
    "synthetic_integrity_pass": int(synthetic_df["status"].eq("PASS").sum()),
    "generation_summary_expected": summary_expected,
    "generation_summary_successful": summary_successful,
    "generation_summary_failed": summary_failed,
    "generation_log_successful": len(generation_success_df),
    "generation_log_failed": len(generation_failure_df),
    "generation_log_unclassified": len(generation_unclassified_df),
    "generation_summary_matches_log": generation_summary_match,
    "privacy_status": privacy_status,
    "generator_source_available": bool(generator_matches),
    "provenance_status": provenance_status,
    "total_checks": len(audit_df),
    "passed_checks": pass_count,
    "failed_checks": fail_count,
    "review_required_checks": review_count,
    "critical_conditions": critical_conditions,
    "review_items": provenance_review_items,
    "final_status": FINAL_SECTION_8_STATUS
}


REPORT_PATH = (
    AUDIT_DIR / "section_08_final_audit_report.json"
)

save_json(
    REPORT_PATH,
    section_8_report
)


# =============================================================================
# 21. COMPLETION SUMMARY
# =============================================================================

print("\n" + "=" * 100)
print("NOTEBOOK 22 — SECTION 8 COMPLETION SUMMARY")
print("=" * 100)

print("Expected datasets:", len(DATASETS))
print("Expected final models:", EXPECTED_MODEL_COUNT)

print(
    "Existing final models:",
    int(final_model_df["exists"].sum())
)

print(
    "Verified final model hashes:",
    int(final_model_df["hash_match"].eq(True).sum())
)

print(
    "Verified final model sizes:",
    int(final_model_df["size_match"].eq(True).sum())
)

print(
    "Existing source checkpoints:",
    int(source_checkpoint_df["exists"].sum())
)

print(
    "Verified source checkpoint hashes:",
    int(source_checkpoint_df["hash_match"].eq(True).sum())
)

print(
    "Training coverage passed:",
    int(training_coverage_df["status"].eq("PASS").sum()),
    "/",
    len(DATASETS)
)

print(
    "Architecture checks passed:",
    int(architecture_df["status"].eq("PASS").sum()),
    "/",
    len(DATASETS)
)

print(
    "Synthetic CSV integrity passed:",
    int(synthetic_df["status"].eq("PASS").sum()),
    "/",
    len(DATASETS)
)

print(
    "Generation successful:",
    len(generation_success_df)
)

print(
    "Generation failed:",
    len(generation_failure_df)
)

print(
    "Generation unclassified:",
    len(generation_unclassified_df)
)

print("\nPASS checks:", pass_count)
print("FAIL checks:", fail_count)
print("REVIEW_REQUIRED checks:", review_count)

print("\nFINAL SECTION 8 STATUS:", FINAL_SECTION_8_STATUS)

print("\nAudit directory:")
print(AUDIT_DIR)

print("\nFinal report:")
print(REPORT_PATH)


if FINAL_SECTION_8_STATUS == "PASS":

    print(
        "\nSECTION 8 PASSED. All critical validation conditions "
        "have been verified."
    )

elif FINAL_SECTION_8_STATUS == "REVIEW_REQUIRED":

    print(
        "\nSECTION 8 REQUIRES REVIEW. Existing artifacts have been "
        "audited, but one or more provenance or implementation "
        "conditions remain unresolved."
    )

else:

    print(
        "\nSECTION 8 FAILED. One or more required artifact "
        "integrity checks failed."
    )


print("\n" + "=" * 100)
print("SECTION 8 EXECUTION COMPLETED")
print("=" * 100)


NOTEBOOK 22 — SECTION 8: VERIFY SPP-GAN ARTIFACTS
Project root: /content/drive/MyDrive/SPP_GAN_Research
Datasets: ['adult_income', 'bank_marketing', 'diabetes_130us']
Expected repetitions: 5
Expected seeds: [3026, 3027, 3028, 3029, 3030]
Expected final models: 15
Audit directory: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_22/section_08_sppgan_audit_final

NOTEBOOK 12 SUMMARY
----------------------------------------------------------------------------------------------------
Final model registry rows: 15
Training metadata rows: 15
Training manifest status: TRAINING_COMPLETE

FINAL MODEL REGISTRY COLUMNS
['dataset', 'repetition_id', 'status', 'final_epoch', 'source_checkpoint', 'source_checkpoint_sha256', 'final_model_path', 'final_model_sha256', 'final_model_size_bytes', 'achieved_epsilon', 'target_epsilon', 'dp_noise_multiplier', 'dp_delta', 'privacy_scope', 'generator_private', 'statistical_guidance_private', 'preprocessing_private', 'end_to_end_privacy_claim',

In [9]:
# ============================================================
# SECTION 9 — VERIFY SYNTHETIC DATA ARTIFACTS
# ============================================================

import hashlib
import pandas as pd

SYNTHETIC_DIR = (
    PROJECT_ROOT
    / "results"
    / "notebooks"
    / "notebook_13"
    / "synthetic_datasets"
)

EXPECTED_SYNTHETIC = {
    "adult_income": {
        "file": "adult_income_sppgan_synthetic.csv",
        "rows": 34189,
        "columns": 15,
        "sha256": "8db2fff4a874767e2620a39e2081b71edd06226d6f206536de7d5e257c00c9cf"
    },
    "bank_marketing": {
        "file": "bank_marketing_sppgan_synthetic.csv",
        "rows": 31647,
        "columns": 17,
        "sha256": "ba2825bf7e372d64c5c5b14d70700305e4d88617532b6fcc419a0dc2e9e70936"
    },
    "diabetes_130us": {
        "file": "diabetes_130us_sppgan_synthetic.csv",
        "rows": 71236,
        "columns": 48,
        "sha256": "d8ccfde666814d01e740e8cef0338639b73ef21b2d2a38b086103d99bcfd0f26"
    }
}

def calculate_sha256(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


synthetic_records = []

for dataset, expected in EXPECTED_SYNTHETIC.items():

    path = SYNTHETIC_DIR / expected["file"]

    record = {
        "dataset": dataset,
        "file_name": expected["file"],
        "path": str(path),
        "exists": path.is_file(),
        "size_bytes": None,
        "rows": None,
        "columns": None,
        "expected_rows": expected["rows"],
        "expected_columns": expected["columns"],
        "actual_sha256": None,
        "expected_sha256": expected["sha256"],
        "row_match": False,
        "column_match": False,
        "hash_match": False,
        "readable": False,
        "status": "MISSING"
    }

    if path.is_file():

        record["size_bytes"] = path.stat().st_size

        if record["size_bytes"] > 0:

            try:
                df = pd.read_csv(path)

                record["rows"] = len(df)
                record["columns"] = len(df.columns)
                record["readable"] = True

                record["row_match"] = (
                    record["rows"] == expected["rows"]
                )

                record["column_match"] = (
                    record["columns"] == expected["columns"]
                )

                record["actual_sha256"] = calculate_sha256(path)

                record["hash_match"] = (
                    record["actual_sha256"] == expected["sha256"]
                )

                if all([
                    record["readable"],
                    record["row_match"],
                    record["column_match"],
                    record["hash_match"]
                ]):
                    record["status"] = "PASS"
                else:
                    record["status"] = "REVIEW_REQUIRED"

            except Exception as exc:
                record["status"] = "READ_ERROR"
                record["error"] = str(exc)

        else:
            record["status"] = "EMPTY_FILE"

    synthetic_records.append(record)


synthetic_audit = pd.DataFrame(synthetic_records)

synthetic_audit.to_csv(
    TABLES_DIR / "synthetic_data_artifact_audit.csv",
    index=False
)

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

total_expected = len(EXPECTED_SYNTHETIC)

passed = int(
    (synthetic_audit["status"] == "PASS").sum()
)

missing = int(
    (synthetic_audit["status"] == "MISSING").sum()
)

review_required = int(
    (synthetic_audit["status"] == "REVIEW_REQUIRED").sum()
)

read_errors = int(
    (synthetic_audit["status"] == "READ_ERROR").sum()
)

empty_files = int(
    (synthetic_audit["status"] == "EMPTY_FILE").sum()
)

print("=" * 80)
print("SECTION 9 — SYNTHETIC DATA ARTIFACT AUDIT")
print("=" * 80)

print("Synthetic directory:", SYNTHETIC_DIR)
print("Expected datasets:", total_expected)
print("Verified datasets:", passed)
print("Missing datasets:", missing)
print("Review required:", review_required)
print("Read errors:", read_errors)
print("Empty files:", empty_files)

print("\nDATASET AUDIT")
print(
    synthetic_audit[
        [
            "dataset",
            "exists",
            "rows",
            "columns",
            "row_match",
            "column_match",
            "hash_match",
            "status"
        ]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# FINAL STATUS
# ------------------------------------------------------------

if passed == total_expected:
    SECTION_09_STATUS = "PASS"

elif missing or review_required or read_errors or empty_files:
    SECTION_09_STATUS = "REVIEW_REQUIRED"

else:
    SECTION_09_STATUS = "FAIL"

print("\nSECTION 9 STATUS:", SECTION_09_STATUS)

print("=" * 80)
print("SECTION 9 EXECUTION COMPLETED")
print("=" * 80)


SECTION 9 — SYNTHETIC DATA ARTIFACT AUDIT
Synthetic directory: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/synthetic_datasets
Expected datasets: 3
Verified datasets: 3
Missing datasets: 0
Review required: 0
Read errors: 0
Empty files: 0

DATASET AUDIT
       dataset  exists  rows  columns  row_match  column_match  hash_match status
  adult_income    True 34189       15       True          True        True   PASS
bank_marketing    True 31647       17       True          True        True   PASS
diabetes_130us    True 71236       48       True          True        True   PASS

SECTION 9 STATUS: PASS
SECTION 9 EXECUTION COMPLETED


In [10]:
# ============================================================
# SECTION 10 — VERIFY EVALUATION ARTIFACTS
# ============================================================

import hashlib
import json
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

EVALUATION_NOTEBOOKS = {
    "notebook_14": PROJECT_ROOT / "results/notebooks/notebook_14",
    "notebook_15": PROJECT_ROOT / "results/notebooks/notebook_15",
    "notebook_16": PROJECT_ROOT / "results/notebooks/notebook_16"
}

SUPPORTED_EXTENSIONS = {
    ".csv",
    ".json",
    ".parquet"
}

AUDIT_PATH = TABLES_DIR / "evaluation_artifact_audit.csv"
SUMMARY_PATH = TABLES_DIR / "evaluation_artifact_audit_summary.json"

# ------------------------------------------------------------
# 2. FILE HASH FUNCTION
# ------------------------------------------------------------

def evaluation_sha256(path):

    digest = hashlib.sha256()

    with open(path, "rb") as file:

        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


# ------------------------------------------------------------
# 3. FILE READABILITY VALIDATION
# ------------------------------------------------------------

def validate_evaluation_file(path):

    extension = path.suffix.lower()

    try:

        if extension == ".csv":
            pd.read_csv(path, nrows=5)

        elif extension == ".json":

            with open(path, "r", encoding="utf-8") as file:
                json.load(file)

        elif extension == ".parquet":
            pd.read_parquet(path)

        else:
            return "UNSUPPORTED"

        return "READABLE"

    except Exception as exc:

        return f"READ_ERROR: {str(exc)[:250]}"


# ------------------------------------------------------------
# 4. RECURSIVE ARTIFACT DISCOVERY
# ------------------------------------------------------------

evaluation_records = []
directory_records = []

for notebook_name, root in EVALUATION_NOTEBOOKS.items():

    directory_exists = root.is_dir()

    directory_records.append({
        "notebook": notebook_name,
        "path": str(root),
        "exists": directory_exists
    })

    if not directory_exists:
        continue

    for path in sorted(root.rglob("*")):

        if not path.is_file():
            continue

        if path.suffix.lower() not in SUPPORTED_EXTENSIONS:
            continue

        size_bytes = path.stat().st_size

        record = {
            "notebook": notebook_name,
            "file_name": path.name,
            "relative_path": str(path.relative_to(root)),
            "path": str(path),
            "extension": path.suffix.lower(),
            "size_bytes": size_bytes,
            "sha256": None,
            "readability": "NOT_CHECKED",
            "status": "PENDING"
        }

        if size_bytes == 0:

            record["status"] = "EMPTY_FILE"
            record["readability"] = "NOT_READABLE"

        else:

            try:

                record["sha256"] = evaluation_sha256(path)

                record["readability"] = (
                    validate_evaluation_file(path)
                )

                if record["readability"] == "READABLE":
                    record["status"] = "PASS"
                else:
                    record["status"] = "REVIEW_REQUIRED"

            except Exception as exc:

                record["status"] = "READ_ERROR"
                record["readability"] = str(exc)[:250]

        evaluation_records.append(record)


# ------------------------------------------------------------
# 5. CREATE AUDIT DATAFRAMES
# ------------------------------------------------------------

evaluation_audit = pd.DataFrame(evaluation_records)

evaluation_directory_audit = pd.DataFrame(
    directory_records
)

# ------------------------------------------------------------
# 6. SAVE AUDIT TABLES
# ------------------------------------------------------------

evaluation_audit.to_csv(
    AUDIT_PATH,
    index=False
)

evaluation_directory_audit.to_csv(
    TABLES_DIR / "evaluation_directory_audit.csv",
    index=False
)

# ------------------------------------------------------------
# 7. NOTEBOOK-LEVEL COVERAGE
# ------------------------------------------------------------

notebook_coverage = []

for notebook_name in EVALUATION_NOTEBOOKS:

    if evaluation_audit.empty:
        subset = pd.DataFrame()
    else:
        subset = evaluation_audit[
            evaluation_audit["notebook"] == notebook_name
        ]

    notebook_coverage.append({
        "notebook": notebook_name,
        "artifact_count": len(subset),
        "passed": (
            int((subset["status"] == "PASS").sum())
            if not subset.empty else 0
        ),
        "review_required": (
            int(
                (subset["status"] == "REVIEW_REQUIRED").sum()
            )
            if not subset.empty else 0
        ),
        "empty_files": (
            int((subset["status"] == "EMPTY_FILE").sum())
            if not subset.empty else 0
        ),
        "read_errors": (
            int((subset["status"] == "READ_ERROR").sum())
            if not subset.empty else 0
        )
    })

notebook_coverage_df = pd.DataFrame(notebook_coverage)

notebook_coverage_df.to_csv(
    TABLES_DIR / "evaluation_notebook_coverage.csv",
    index=False
)

# ------------------------------------------------------------
# 8. CONSOLIDATED SUMMARY
# ------------------------------------------------------------

total_artifacts = len(evaluation_audit)

passed_artifacts = (
    int((evaluation_audit["status"] == "PASS").sum())
    if total_artifacts else 0
)

review_artifacts = (
    int(
        (evaluation_audit["status"] == "REVIEW_REQUIRED").sum()
    )
    if total_artifacts else 0
)

empty_artifacts = (
    int((evaluation_audit["status"] == "EMPTY_FILE").sum())
    if total_artifacts else 0
)

read_errors = (
    int((evaluation_audit["status"] == "READ_ERROR").sum())
    if total_artifacts else 0
)

existing_directories = int(
    evaluation_directory_audit["exists"].sum()
)

all_directories_exist = (
    existing_directories == len(EVALUATION_NOTEBOOKS)
)

all_artifacts_valid = (
    total_artifacts > 0
    and passed_artifacts == total_artifacts
)

# ------------------------------------------------------------
# 9. FINAL STATUS
# ------------------------------------------------------------

if all_directories_exist and all_artifacts_valid:

    SECTION_10_STATUS = "PASS"

elif total_artifacts == 0:

    SECTION_10_STATUS = "REVIEW_REQUIRED"

else:

    SECTION_10_STATUS = "REVIEW_REQUIRED"

# ------------------------------------------------------------
# 10. SAVE SUMMARY JSON
# ------------------------------------------------------------

summary = {
    "section": 10,
    "section_name": "Verify Evaluation Artifacts",
    "expected_notebooks": list(EVALUATION_NOTEBOOKS.keys()),
    "existing_notebook_directories": existing_directories,
    "total_artifacts": total_artifacts,
    "passed_artifacts": passed_artifacts,
    "review_required_artifacts": review_artifacts,
    "empty_artifacts": empty_artifacts,
    "read_errors": read_errors,
    "all_artifacts_readable": all_artifacts_valid,
    "status": SECTION_10_STATUS
}

with open(SUMMARY_PATH, "w", encoding="utf-8") as file:

    json.dump(
        summary,
        file,
        indent=4
    )

# ------------------------------------------------------------
# 11. COMPLETION SUMMARY
# ------------------------------------------------------------

print("=" * 80)
print("SECTION 10 — EVALUATION ARTIFACT AUDIT")
print("=" * 80)

print("Expected notebook directories:", len(EVALUATION_NOTEBOOKS))
print("Existing notebook directories:", existing_directories)
print("Total evaluation artifacts:", total_artifacts)
print("Verified readable artifacts:", passed_artifacts)
print("Review required:", review_artifacts)
print("Empty files:", empty_artifacts)
print("Read errors:", read_errors)

print("\nNOTEBOOK COVERAGE")

print(
    notebook_coverage_df.to_string(index=False)
)

print("\nARTIFACT EXTENSION COVERAGE")

if not evaluation_audit.empty:

    print(
        evaluation_audit.groupby(
            ["notebook", "extension"]
        ).size().to_string()
    )

print("\nSECTION 10 STATUS:", SECTION_10_STATUS)

print("\nSaved audit:", AUDIT_PATH)
print("Saved summary:", SUMMARY_PATH)

print("=" * 80)
print("SECTION 10 EXECUTION COMPLETED")
print("=" * 80)


SECTION 10 — EVALUATION ARTIFACT AUDIT
Expected notebook directories: 3
Existing notebook directories: 3
Total evaluation artifacts: 127
Verified readable artifacts: 127
Review required: 0
Empty files: 0
Read errors: 0

NOTEBOOK COVERAGE
   notebook  artifact_count  passed  review_required  empty_files  read_errors
notebook_14              22      22                0            0            0
notebook_15              69      69                0            0            0
notebook_16              36      36                0            0            0

ARTIFACT EXTENSION COVERAGE
notebook     extension
notebook_14  .csv         19
             .json         3
notebook_15  .csv         46
             .json        23
notebook_16  .csv         33
             .json         3

SECTION 10 STATUS: PASS

Saved audit: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_22/tables/evaluation_artifact_audit.csv
Saved summary: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/n

In [11]:
# ============================================================
# SECTION 11 — VERIFY ABLATION ARTIFACTS
# ============================================================

import hashlib
import json
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

ABLATION_ROOT = (
    PROJECT_ROOT
    / "results"
    / "notebooks"
    / "notebook_17"
)

ABLATION_PATTERNS = [
    "*ablation*.csv",
    "*ablation*.json",
    "*variant*.csv",
    "*variant*.json"
]

ABLATION_AUDIT_PATH = (
    TABLES_DIR / "ablation_artifact_audit.csv"
)

ABLATION_SUMMARY_PATH = (
    TABLES_DIR / "ablation_artifact_audit_summary.json"
)

# ------------------------------------------------------------
# 2. HASH FUNCTION
# ------------------------------------------------------------

def ablation_sha256(path):

    digest = hashlib.sha256()

    with open(path, "rb") as file:

        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


# ------------------------------------------------------------
# 3. READABILITY VALIDATION
# ------------------------------------------------------------

def validate_ablation_file(path):

    extension = path.suffix.lower()

    try:

        if extension == ".csv":
            pd.read_csv(path, nrows=5)

        elif extension == ".json":

            with open(path, "r", encoding="utf-8") as file:
                json.load(file)

        else:
            return "UNSUPPORTED"

        return "READABLE"

    except Exception as exc:

        return f"READ_ERROR: {str(exc)[:250]}"


# ------------------------------------------------------------
# 4. DIRECTORY VALIDATION
# ------------------------------------------------------------

print("=" * 80)
print("SECTION 11 — ABLATION ARTIFACT AUDIT")
print("=" * 80)

print("Ablation directory:", ABLATION_ROOT)
print("Directory exists:", ABLATION_ROOT.is_dir())

# ------------------------------------------------------------
# 5. RECURSIVE ARTIFACT DISCOVERY
# ------------------------------------------------------------

ablation_files = []

if ABLATION_ROOT.is_dir():

    discovered = set()

    for pattern in ABLATION_PATTERNS:

        discovered.update(
            path
            for path in ABLATION_ROOT.rglob(pattern)
            if path.is_file()
        )

    ablation_files = sorted(discovered)


# ------------------------------------------------------------
# 6. ARTIFACT AUDIT
# ------------------------------------------------------------

ablation_records = []

for path in ablation_files:

    size_bytes = path.stat().st_size

    record = {
        "notebook": "notebook_17",
        "file_name": path.name,
        "relative_path": str(
            path.relative_to(ABLATION_ROOT)
        ),
        "path": str(path),
        "extension": path.suffix.lower(),
        "size_bytes": size_bytes,
        "sha256": None,
        "readability": "NOT_CHECKED",
        "status": "PENDING"
    }

    if size_bytes == 0:

        record["status"] = "EMPTY_FILE"
        record["readability"] = "NOT_READABLE"

    else:

        try:

            record["sha256"] = ablation_sha256(path)

            record["readability"] = (
                validate_ablation_file(path)
            )

            if record["readability"] == "READABLE":
                record["status"] = "PASS"
            else:
                record["status"] = "REVIEW_REQUIRED"

        except Exception as exc:

            record["status"] = "READ_ERROR"
            record["readability"] = str(exc)[:250]

    ablation_records.append(record)


# ------------------------------------------------------------
# 7. SAVE AUDIT TABLE
# ------------------------------------------------------------

ablation_audit = pd.DataFrame(ablation_records)

ablation_audit.to_csv(
    ABLATION_AUDIT_PATH,
    index=False
)


# ------------------------------------------------------------
# 8. SUMMARY
# ------------------------------------------------------------

total_artifacts = len(ablation_audit)

passed = (
    int((ablation_audit["status"] == "PASS").sum())
    if total_artifacts else 0
)

review_required = (
    int(
        (ablation_audit["status"] == "REVIEW_REQUIRED").sum()
    )
    if total_artifacts else 0
)

empty_files = (
    int((ablation_audit["status"] == "EMPTY_FILE").sum())
    if total_artifacts else 0
)

read_errors = (
    int((ablation_audit["status"] == "READ_ERROR").sum())
    if total_artifacts else 0
)


# ------------------------------------------------------------
# 9. FINAL STATUS
# ------------------------------------------------------------

if not ABLATION_ROOT.is_dir():

    SECTION_11_STATUS = "REVIEW_REQUIRED"

elif total_artifacts == 0:

    SECTION_11_STATUS = "REVIEW_REQUIRED"

elif passed == total_artifacts:

    SECTION_11_STATUS = "PASS"

else:

    SECTION_11_STATUS = "REVIEW_REQUIRED"


# ------------------------------------------------------------
# 10. SAVE SUMMARY JSON
# ------------------------------------------------------------

summary = {
    "section": 11,
    "section_name": "Verify Ablation Artifacts",
    "directory": str(ABLATION_ROOT),
    "directory_exists": ABLATION_ROOT.is_dir(),
    "total_artifacts": total_artifacts,
    "passed_artifacts": passed,
    "review_required_artifacts": review_required,
    "empty_files": empty_files,
    "read_errors": read_errors,
    "status": SECTION_11_STATUS
}

with open(
    ABLATION_SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(summary, file, indent=4)


# ------------------------------------------------------------
# 11. COMPLETION SUMMARY
# ------------------------------------------------------------

print("\nTOTAL ABLATION ARTIFACTS:", total_artifacts)
print("Verified readable artifacts:", passed)
print("Review required:", review_required)
print("Empty files:", empty_files)
print("Read errors:", read_errors)

if not ablation_audit.empty:

    print("\nABLATION ARTIFACT DETAILS")

    print(
        ablation_audit[
            [
                "file_name",
                "relative_path",
                "size_bytes",
                "readability",
                "status"
            ]
        ].to_string(index=False)
    )

print("\nSECTION 11 STATUS:", SECTION_11_STATUS)

print("\nSaved audit:", ABLATION_AUDIT_PATH)
print("Saved summary:", ABLATION_SUMMARY_PATH)

print("=" * 80)
print("SECTION 11 EXECUTION COMPLETED")
print("=" * 80)


SECTION 11 — ABLATION ARTIFACT AUDIT
Ablation directory: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_17
Directory exists: True

TOTAL ABLATION ARTIFACTS: 17
Verified readable artifacts: 17
Review required: 0
Empty files: 0
Read errors: 0

ABLATION ARTIFACT DETAILS
                              file_name                                   relative_path  size_bytes readability status
        synthetic_ablation_manifest.csv       manifests/synthetic_ablation_manifest.csv          81    READABLE   PASS
 ablation_experiment_configuration.json metadata/ablation_experiment_configuration.json        3861    READABLE   PASS
       ablation_variant_definitions.csv       metadata/ablation_variant_definitions.csv         832    READABLE   PASS
       ablation_experiment_registry.csv     registries/ablation_experiment_registry.csv        1955    READABLE   PASS
          ablation_variant_registry.csv        registries/ablation_variant_registry.csv         832    READABLE   PAS

In [12]:
# ============================================================
# SECTION 12 — VERIFY TRADE-OFF ARTIFACTS
# ============================================================

import hashlib
import json
import pandas as pd

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

TRADEOFF_ROOT = (
    PROJECT_ROOT
    / "results"
    / "notebooks"
    / "notebook_18"
)

TRADEOFF_PATTERNS = [
    "*privacy_vs_fidelity*",
    "*privacy_vs_utility*",
    "*fidelity_vs_utility*",
    "*pareto*"
]

EXPECTED_CORE_FILES = [
    "results/privacy_vs_fidelity_results.csv",
    "results/privacy_vs_utility_results.csv",
    "results/fidelity_vs_utility_results.csv",
    "results/pareto_evaluation.csv",
    "tables/privacy_vs_fidelity_summary.csv",
    "tables/privacy_vs_utility_summary.csv",
    "tables/fidelity_vs_utility_summary.csv"
]

TRADEOFF_AUDIT_PATH = (
    TABLES_DIR / "tradeoff_artifact_audit.csv"
)

TRADEOFF_SUMMARY_PATH = (
    TABLES_DIR / "tradeoff_artifact_audit_summary.json"
)

# ------------------------------------------------------------
# 2. SHA-256 FUNCTION
# ------------------------------------------------------------

def tradeoff_sha256(path):

    digest = hashlib.sha256()

    with open(path, "rb") as file:

        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


# ------------------------------------------------------------
# 3. READABILITY VALIDATION
# ------------------------------------------------------------

def validate_tradeoff_file(path):

    extension = path.suffix.lower()

    try:

        if extension == ".csv":
            pd.read_csv(path, nrows=5)

        elif extension == ".json":

            with open(path, "r", encoding="utf-8") as file:
                json.load(file)

        else:
            return "UNSUPPORTED"

        return "READABLE"

    except Exception as exc:

        return f"READ_ERROR: {str(exc)[:250]}"


# ------------------------------------------------------------
# 4. DIRECTORY VALIDATION
# ------------------------------------------------------------

print("=" * 80)
print("SECTION 12 — TRADE-OFF ARTIFACT AUDIT")
print("=" * 80)

print("Trade-off directory:", TRADEOFF_ROOT)
print("Directory exists:", TRADEOFF_ROOT.is_dir())


# ------------------------------------------------------------
# 5. RECURSIVE ARTIFACT DISCOVERY
# ------------------------------------------------------------

tradeoff_files = []

if TRADEOFF_ROOT.is_dir():

    discovered = set()

    for pattern in TRADEOFF_PATTERNS:

        discovered.update(
            path
            for path in TRADEOFF_ROOT.rglob(pattern)
            if path.is_file()
            and path.suffix.lower() in {".csv", ".json"}
        )

    tradeoff_files = sorted(discovered)


# ------------------------------------------------------------
# 6. ARTIFACT INTEGRITY AUDIT
# ------------------------------------------------------------

tradeoff_records = []

for path in tradeoff_files:

    size_bytes = path.stat().st_size

    record = {
        "notebook": "notebook_18",
        "file_name": path.name,
        "relative_path": str(
            path.relative_to(TRADEOFF_ROOT)
        ),
        "path": str(path),
        "extension": path.suffix.lower(),
        "size_bytes": size_bytes,
        "sha256": None,
        "readability": "NOT_CHECKED",
        "status": "PENDING"
    }

    if size_bytes == 0:

        record["status"] = "EMPTY_FILE"
        record["readability"] = "NOT_READABLE"

    else:

        try:

            record["sha256"] = tradeoff_sha256(path)

            record["readability"] = (
                validate_tradeoff_file(path)
            )

            if record["readability"] == "READABLE":
                record["status"] = "PASS"
            else:
                record["status"] = "REVIEW_REQUIRED"

        except Exception as exc:

            record["status"] = "READ_ERROR"
            record["readability"] = str(exc)[:250]

    tradeoff_records.append(record)


# ------------------------------------------------------------
# 7. CORE ARTIFACT COVERAGE
# ------------------------------------------------------------

discovered_relative_paths = {
    str(path.relative_to(TRADEOFF_ROOT))
    for path in tradeoff_files
}

core_coverage_records = []

for relative_path in EXPECTED_CORE_FILES:

    path = TRADEOFF_ROOT / relative_path

    core_coverage_records.append({
        "expected_file": relative_path,
        "exists": path.is_file(),
        "size_bytes": (
            path.stat().st_size if path.is_file() else None
        )
    })

core_coverage_df = pd.DataFrame(
    core_coverage_records
)

core_coverage_df.to_csv(
    TABLES_DIR / "tradeoff_core_coverage.csv",
    index=False
)


# ------------------------------------------------------------
# 8. CREATE AND SAVE AUDIT
# ------------------------------------------------------------

tradeoff_audit = pd.DataFrame(tradeoff_records)

tradeoff_audit.to_csv(
    TRADEOFF_AUDIT_PATH,
    index=False
)


# ------------------------------------------------------------
# 9. SUMMARY
# ------------------------------------------------------------

total_artifacts = len(tradeoff_audit)

passed = (
    int((tradeoff_audit["status"] == "PASS").sum())
    if total_artifacts else 0
)

review_required = (
    int(
        (tradeoff_audit["status"] == "REVIEW_REQUIRED").sum()
    )
    if total_artifacts else 0
)

empty_files = (
    int((tradeoff_audit["status"] == "EMPTY_FILE").sum())
    if total_artifacts else 0
)

read_errors = (
    int((tradeoff_audit["status"] == "READ_ERROR").sum())
    if total_artifacts else 0
)

expected_core = len(EXPECTED_CORE_FILES)

verified_core = int(
    core_coverage_df["exists"].sum()
)

all_core_present = (
    verified_core == expected_core
)

all_artifacts_readable = (
    total_artifacts > 0
    and passed == total_artifacts
)


# ------------------------------------------------------------
# 10. FINAL STATUS
# ------------------------------------------------------------

if not TRADEOFF_ROOT.is_dir():

    SECTION_12_STATUS = "REVIEW_REQUIRED"

elif total_artifacts == 0:

    SECTION_12_STATUS = "REVIEW_REQUIRED"

elif all_artifacts_readable and all_core_present:

    SECTION_12_STATUS = "PASS"

else:

    SECTION_12_STATUS = "REVIEW_REQUIRED"


# ------------------------------------------------------------
# 11. SAVE SUMMARY JSON
# ------------------------------------------------------------

summary = {
    "section": 12,
    "section_name": "Verify Trade-off Artifacts",
    "directory": str(TRADEOFF_ROOT),
    "directory_exists": TRADEOFF_ROOT.is_dir(),
    "total_artifacts": total_artifacts,
    "passed_artifacts": passed,
    "review_required_artifacts": review_required,
    "empty_files": empty_files,
    "read_errors": read_errors,
    "expected_core_artifacts": expected_core,
    "verified_core_artifacts": verified_core,
    "all_core_artifacts_present": all_core_present,
    "all_artifacts_readable": all_artifacts_readable,
    "status": SECTION_12_STATUS
}

with open(
    TRADEOFF_SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(summary, file, indent=4)


# ------------------------------------------------------------
# 12. COMPLETION SUMMARY
# ------------------------------------------------------------

print("\nEXPECTED CORE ARTIFACTS:", expected_core)
print("VERIFIED CORE ARTIFACTS:", verified_core)

print("\nTOTAL TRADE-OFF ARTIFACTS:", total_artifacts)
print("Verified readable artifacts:", passed)
print("Review required:", review_required)
print("Empty files:", empty_files)
print("Read errors:", read_errors)

print("\nCORE ARTIFACT COVERAGE")

print(
    core_coverage_df.to_string(index=False)
)

if not tradeoff_audit.empty:

    print("\nTRADE-OFF ARTIFACT DETAILS")

    print(
        tradeoff_audit[
            [
                "file_name",
                "relative_path",
                "size_bytes",
                "readability",
                "status"
            ]
        ].to_string(index=False)
    )

print("\nSECTION 12 STATUS:", SECTION_12_STATUS)

print("\nSaved audit:", TRADEOFF_AUDIT_PATH)
print("Saved summary:", TRADEOFF_SUMMARY_PATH)

print("=" * 80)
print("SECTION 12 EXECUTION COMPLETED")
print("=" * 80)


SECTION 12 — TRADE-OFF ARTIFACT AUDIT
Trade-off directory: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_18
Directory exists: True

EXPECTED CORE ARTIFACTS: 7
VERIFIED CORE ARTIFACTS: 7

TOTAL TRADE-OFF ARTIFACTS: 9
Verified readable artifacts: 9
Review required: 0
Empty files: 0
Read errors: 0

CORE ARTIFACT COVERAGE
                          expected_file  exists  size_bytes
results/privacy_vs_fidelity_results.csv    True          55
 results/privacy_vs_utility_results.csv    True          55
results/fidelity_vs_utility_results.csv    True          55
          results/pareto_evaluation.csv    True         301
 tables/privacy_vs_fidelity_summary.csv    True         163
  tables/privacy_vs_utility_summary.csv    True         160
 tables/fidelity_vs_utility_summary.csv    True         163

TRADE-OFF ARTIFACT DETAILS
                      file_name                           relative_path  size_bytes readability status
fidelity_vs_utility_results.csv results/fidelity

In [13]:
# ============================================================
# SECTION 13 — VERIFY SIGNIFICANCE ARTIFACTS
# ============================================================

import hashlib
import json
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

NOTEBOOK_19_DIR = (
    PROJECT_ROOT / "results" / "notebooks" / "notebook_19"
)

audit_dir = TABLES_DIR
audit_dir.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("SECTION 13 — SIGNIFICANCE ARTIFACT AUDIT")
print("=" * 80)

print("Significance directory:", NOTEBOOK_19_DIR)
print("Directory exists:", NOTEBOOK_19_DIR.exists())

# ------------------------------------------------------------
# 1. DISCOVER SIGNIFICANCE ARTIFACTS
# ------------------------------------------------------------

patterns = [
    "*significance*.csv",
    "*significance*.json",
    "*wilcoxon*.csv",
    "*wilcoxon*.json",
    "*friedman*.csv",
    "*friedman*.json",
    "*posthoc*.csv",
    "*posthoc*.json",
    "*statistical_test*.csv",
    "*statistical_test*.json",
    "*pvalue*.csv",
    "*p_value*.csv",
    "*multiple_comparison*.csv",
    "*multiple_comparison*.json"
]

significance_files = []

if NOTEBOOK_19_DIR.exists():
    for pattern in patterns:
        significance_files.extend(
            NOTEBOOK_19_DIR.rglob(pattern)
        )

significance_files = sorted(
    set(p.resolve() for p in significance_files)
)

# ------------------------------------------------------------
# 2. AUDIT ARTIFACTS
# ------------------------------------------------------------

audit_records = []

for path in significance_files:

    record = {
        "file_name": path.name,
        "relative_path": str(
            path.relative_to(NOTEBOOK_19_DIR)
        ),
        "size_bytes": path.stat().st_size,
        "sha256": None,
        "readability": "REVIEW_REQUIRED",
        "row_count": None,
        "column_count": None,
        "status": "REVIEW_REQUIRED"
    }

    try:
        if path.stat().st_size == 0:
            record["readability"] = "EMPTY"
            record["status"] = "FAIL"

        else:
            with open(path, "rb") as f:
                record["sha256"] = hashlib.sha256(
                    f.read()
                ).hexdigest()

            if path.suffix.lower() == ".csv":
                df = pd.read_csv(path)

                record["row_count"] = len(df)
                record["column_count"] = len(df.columns)

                if len(df) == 0:
                    record["readability"] = "READABLE_HEADER_ONLY"
                    record["status"] = "REVIEW_REQUIRED"
                else:
                    record["readability"] = "READABLE"
                    record["status"] = "PASS"

            elif path.suffix.lower() == ".json":
                with open(path, "r", encoding="utf-8") as f:
                    json.load(f)

                record["readability"] = "READABLE"
                record["status"] = "PASS"

    except Exception as e:
        record["readability"] = "READ_ERROR"
        record["status"] = "FAIL"
        record["error"] = str(e)

    audit_records.append(record)

significance_audit = pd.DataFrame(audit_records)

# ------------------------------------------------------------
# 3. SAVE AUDIT
# ------------------------------------------------------------

audit_path = audit_dir / "significance_artifact_audit.csv"

significance_audit.to_csv(
    audit_path,
    index=False
)

# ------------------------------------------------------------
# 4. SUMMARIZE RESULTS
# ------------------------------------------------------------

total_files = len(significance_audit)

readable_files = (
    int(significance_audit["readability"].isin(
        ["READABLE", "READABLE_HEADER_ONLY"]
    ).sum())
    if total_files else 0
)

valid_data_files = (
    int((significance_audit["status"] == "PASS").sum())
    if total_files else 0
)

review_files = (
    int((significance_audit["status"] == "REVIEW_REQUIRED").sum())
    if total_files else 0
)

failed_files = (
    int((significance_audit["status"] == "FAIL").sum())
    if total_files else 0
)

header_only_files = (
    int((significance_audit["readability"] ==
         "READABLE_HEADER_ONLY").sum())
    if total_files else 0
)

print("\nSIGNIFICANCE ARTIFACT SUMMARY")
print("-" * 80)
print("Total artifacts:", total_files)
print("Readable artifacts:", readable_files)
print("Artifacts with data rows:", valid_data_files)
print("Header-only artifacts:", header_only_files)
print("Review required:", review_files)
print("Failed artifacts:", failed_files)

if total_files:
    print("\nSIGNIFICANCE ARTIFACT DETAILS")
    print(significance_audit.to_string(index=False))

# ------------------------------------------------------------
# 5. FINAL STATUS
# ------------------------------------------------------------

if (
    NOTEBOOK_19_DIR.exists()
    and total_files > 0
    and valid_data_files > 0
    and review_files == 0
    and failed_files == 0
):
    section_13_status = "PASS"

elif total_files == 0 or review_files > 0:
    section_13_status = "REVIEW_REQUIRED"

else:
    section_13_status = "FAIL"

summary = {
    "section": 13,
    "section_name": "Verify Significance Artifacts",
    "audit_timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "directory": str(NOTEBOOK_19_DIR),
    "total_artifacts": total_files,
    "readable_artifacts": readable_files,
    "valid_data_artifacts": valid_data_files,
    "header_only_artifacts": header_only_files,
    "review_required": review_files,
    "failed_artifacts": failed_files,
    "status": section_13_status
}

summary_path = (
    audit_dir / "significance_artifact_audit_summary.json"
)

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4)

print("\nSECTION 13 STATUS:", section_13_status)
print("Saved audit:", audit_path)
print("Saved summary:", summary_path)

print("=" * 80)
print("SECTION 13 EXECUTION COMPLETED")
print("=" * 80)


SECTION 13 — SIGNIFICANCE ARTIFACT AUDIT
Significance directory: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_19
Directory exists: True

SIGNIFICANCE ARTIFACT SUMMARY
--------------------------------------------------------------------------------
Total artifacts: 9
Readable artifacts: 9
Artifacts with data rows: 9
Header-only artifacts: 0
Review required: 0
Failed artifacts: 0

SIGNIFICANCE ARTIFACT DETAILS
                        file_name                              relative_path  size_bytes                                                           sha256 readability  row_count  column_count status
statistical_testing_protocol.json metadata/statistical_testing_protocol.json        3412 5bed3bdc513aa807370c9eeceadbafe50e959144885a516e72afe232c6a5db2c    READABLE        NaN           NaN   PASS
   dataset_level_significance.csv      tables/dataset_level_significance.csv        1019 ccc81979d13d8ff909e6acfa92cd7c761c0ead34484af1c11e9e43ac44192565    READABLE     

In [14]:
# ============================================================
# SECTION 14 — VERIFY PUBLICATION ARTIFACTS
# ============================================================

import hashlib
import json
import pandas as pd
import xml.etree.ElementTree as ET

from pathlib import Path
from datetime import datetime, timezone

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

NOTEBOOK_21 = (
    PROJECT_ROOT / "results" / "notebooks" / "notebook_21"
)

TABLES_DIR.mkdir(parents=True, exist_ok=True)

SUPPORTED_EXTENSIONS = {
    ".csv", ".json", ".png",
    ".pdf", ".svg", ".tex"
}

print("=" * 80)
print("SECTION 14 — PUBLICATION ARTIFACT AUDIT")
print("=" * 80)

print("Publication directory:", NOTEBOOK_21)
print("Directory exists:", NOTEBOOK_21.exists())

# ------------------------------------------------------------
# 2. RECURSIVE DISCOVERY
# ------------------------------------------------------------

publication_files = []

if NOTEBOOK_21.exists():
    publication_files = sorted([
        p for p in NOTEBOOK_21.rglob("*")
        if p.is_file()
        and p.suffix.lower() in SUPPORTED_EXTENSIONS
    ])

# ------------------------------------------------------------
# 3. FILE VALIDATION
# ------------------------------------------------------------

publication_records = []

for path in publication_files:

    record = {
        "file_name": path.name,
        "relative_path": str(
            path.relative_to(NOTEBOOK_21)
        ),
        "extension": path.suffix.lower(),
        "size_bytes": path.stat().st_size,
        "sha256": None,
        "readability": "REVIEW_REQUIRED",
        "row_count": None,
        "column_count": None,
        "status": "REVIEW_REQUIRED"
    }

    try:
        if path.stat().st_size == 0:
            record["readability"] = "EMPTY"
            record["status"] = "FAIL"

        else:
            with open(path, "rb") as f:
                file_bytes = f.read()

            record["sha256"] = hashlib.sha256(
                file_bytes
            ).hexdigest()

            ext = path.suffix.lower()

            if ext == ".csv":
                df = pd.read_csv(path)

                record["row_count"] = len(df)
                record["column_count"] = len(df.columns)

                if len(df) == 0:
                    record["readability"] = "HEADER_ONLY"
                    record["status"] = "REVIEW_REQUIRED"
                else:
                    record["readability"] = "READABLE"
                    record["status"] = "PASS"

            elif ext == ".json":
                with open(path, "r", encoding="utf-8") as f:
                    json.load(f)

                record["readability"] = "READABLE"
                record["status"] = "PASS"

            elif ext == ".png":
                from PIL import Image

                with Image.open(path) as img:
                    img.verify()

                record["readability"] = "READABLE"
                record["status"] = "PASS"

            elif ext == ".pdf":
                if not file_bytes.startswith(b"%PDF-"):
                    raise ValueError("Invalid PDF signature")

                record["readability"] = "SIGNATURE_VALID"
                record["status"] = "PASS"

            elif ext == ".svg":
                ET.parse(path)

                record["readability"] = "VALID_XML"
                record["status"] = "PASS"

            elif ext == ".tex":
                content = path.read_text(
                    encoding="utf-8",
                    errors="strict"
                )

                if not content.strip():
                    record["readability"] = "EMPTY_CONTENT"
                    record["status"] = "FAIL"
                else:
                    record["readability"] = "NONEMPTY_SOURCE"
                    record["status"] = "PASS"

    except Exception as e:
        record["readability"] = "READ_ERROR"
        record["status"] = "FAIL"
        record["error"] = str(e)

    publication_records.append(record)

publication_audit = pd.DataFrame(publication_records)

# ------------------------------------------------------------
# 4. SAVE ARTIFACT AUDIT
# ------------------------------------------------------------

audit_path = (
    TABLES_DIR / "publication_artifact_audit.csv"
)

publication_audit.to_csv(
    audit_path,
    index=False
)

# ------------------------------------------------------------
# 5. SUMMARY
# ------------------------------------------------------------

total_files = len(publication_audit)

passed_files = (
    int((publication_audit["status"] == "PASS").sum())
    if total_files else 0
)

review_files = (
    int((publication_audit["status"] == "REVIEW_REQUIRED").sum())
    if total_files else 0
)

failed_files = (
    int((publication_audit["status"] == "FAIL").sum())
    if total_files else 0
)

print("\nPUBLICATION ARTIFACT SUMMARY")
print("-" * 80)

print("Total artifacts:", total_files)
print("Validated artifacts:", passed_files)
print("Review required:", review_files)
print("Failed artifacts:", failed_files)

if total_files > 0:
    print("\nARTIFACT DETAILS")
    print(publication_audit.to_string(index=False))

# ------------------------------------------------------------
# 6. FINAL STATUS
# ------------------------------------------------------------

if (
    NOTEBOOK_21.exists()
    and total_files > 0
    and review_files == 0
    and failed_files == 0
):
    section_14_status = "PASS"

elif total_files == 0 or review_files > 0:
    section_14_status = "REVIEW_REQUIRED"

else:
    section_14_status = "FAIL"

summary = {
    "section": 14,
    "section_name": "Verify Publication Artifacts",
    "audit_timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "publication_directory": str(NOTEBOOK_21),
    "total_artifacts": total_files,
    "validated_artifacts": passed_files,
    "review_required": review_files,
    "failed_artifacts": failed_files,
    "status": section_14_status
}

summary_path = (
    TABLES_DIR / "publication_artifact_audit_summary.json"
)

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4)

print("\nSECTION 14 STATUS:", section_14_status)
print("Saved audit:", audit_path)
print("Saved summary:", summary_path)

print("=" * 80)
print("SECTION 14 EXECUTION COMPLETED")
print("=" * 80)


SECTION 14 — PUBLICATION ARTIFACT AUDIT
Publication directory: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_21
Directory exists: True

PUBLICATION ARTIFACT SUMMARY
--------------------------------------------------------------------------------
Total artifacts: 103
Validated artifacts: 86
Review required: 17
Failed artifacts: 0

ARTIFACT DETAILS
                                file_name                                       relative_path extension  size_bytes                                                           sha256     readability  row_count  column_count          status
                             ablation.pdf                                figures/ablation.pdf      .pdf       21915 aad8868a2c04984d668704ba9ac155c46b689358fe55ad7d8c7edaf42baa2fe4 SIGNATURE_VALID        NaN           NaN            PASS
                             ablation.png                                figures/ablation.png      .png      221892 5ec827f812956b13bed35022bda1e875b115ff

In [15]:
# ============================================================
# SECTION 15 — DATASET COVERAGE AUDIT
# ============================================================

import pandas as pd
import json
from pathlib import Path
from datetime import datetime, timezone

# ------------------------------------------------------------
# 1. DATASET REGISTRY
# ------------------------------------------------------------

DATASET_METADATA = {
    "adult_income": {
        "rows": 48842,
        "columns": 15,
        "target": "income",
        "generative_columns": 15,
        "aliases": [
            "adult_income",
            "adult-income"
        ]
    },

    "bank_marketing": {
        "rows": 45211,
        "columns": 17,
        "target": "y",
        "generative_columns": 17,
        "aliases": [
            "bank_marketing",
            "bank-marketing",
            "bank_marketing_original"
        ]
    },

    "diabetes_130us": {
        "rows": 101766,
        "columns": 50,
        "target": "readmitted",
        "generative_columns": 48,
        "aliases": [
            "diabetes_130us",
            "diabetes-130us",
            "diabetic_data"
        ]
    }
}

DATASET_ROOT = PROJECT_ROOT

TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("SECTION 15 — DATASET COVERAGE AUDIT")
print("=" * 80)

print("Project root:", DATASET_ROOT)
print("Project root exists:", DATASET_ROOT.exists())

# ------------------------------------------------------------
# 2. DATA FILE DISCOVERY
# ------------------------------------------------------------

SUPPORTED_EXTENSIONS = {
    ".csv",
    ".parquet"
}

all_data_files = []

if DATASET_ROOT.exists():

    for path in DATASET_ROOT.rglob("*"):

        if not path.is_file():
            continue

        if path.suffix.lower() not in SUPPORTED_EXTENSIONS:
            continue

        # Exclude current audit outputs.
        if "notebook_22" in path.parts:
            continue

        all_data_files.append(path)

all_data_files = sorted(
    set(all_data_files)
)

print(
    "\nTotal CSV/Parquet files discovered:",
    len(all_data_files)
)

# ------------------------------------------------------------
# 3. IDENTIFY DATASET FROM PATH AND FILE NAME
# ------------------------------------------------------------

def identify_dataset(path):

    file_text = (
        path.name.lower()
        + " "
        + str(path.parent).lower()
    )

    matches = []

    for dataset, metadata in DATASET_METADATA.items():

        for alias in metadata["aliases"]:

            if alias in file_text:
                matches.append(dataset)
                break

    return list(dict.fromkeys(matches))


# ------------------------------------------------------------
# 4. DETERMINE CSV DELIMITER
# ------------------------------------------------------------

BANK_SEMICOLON_FILES = {
    "bank_marketing.csv",
    "bank-full.csv",
    "bank.csv",
    "bank-additional-full.csv",
    "bank-additional.csv"
}


def get_file_separator(path, matched_datasets):

    if (
        "bank_marketing" in matched_datasets
        and path.name.lower() in BANK_SEMICOLON_FILES
    ):
        return ";"

    return ","


# ------------------------------------------------------------
# 5. READ DATASET DIMENSIONS WITH LOW RAM USAGE
# ------------------------------------------------------------

def inspect_data_file(path, matched_datasets):

    result = {
        "actual_rows": None,
        "actual_columns": None,
        "column_names": None,
        "readability": "REVIEW_REQUIRED",
        "error": None
    }

    try:

        if path.stat().st_size == 0:
            result["readability"] = "EMPTY"
            return result

        if path.suffix.lower() == ".csv":

            separator = get_file_separator(
                path,
                matched_datasets
            )

            header = pd.read_csv(
                path,
                sep=separator,
                nrows=0
            )

            columns = [
                str(column).strip()
                for column in header.columns
            ]

            row_count = 0

            for chunk in pd.read_csv(
                path,
                sep=separator,
                chunksize=20000
            ):
                row_count += len(chunk)

        elif path.suffix.lower() == ".parquet":

            import pyarrow.parquet as pq

            parquet_file = pq.ParquetFile(path)

            columns = [
                str(column).strip()
                for column in parquet_file.schema.names
            ]

            row_count = parquet_file.metadata.num_rows

        else:
            return result

        result["actual_rows"] = row_count
        result["actual_columns"] = len(columns)
        result["column_names"] = columns

        if row_count > 0 and len(columns) > 0:
            result["readability"] = "READABLE"
        else:
            result["readability"] = "EMPTY_DATA"

    except Exception as e:

        result["readability"] = "READ_ERROR"
        result["error"] = str(e)

    return result


# ------------------------------------------------------------
# 6. BUILD DATA ARTIFACT INVENTORY
# ------------------------------------------------------------

artifact_records = []

for path in all_data_files:

    matched_datasets = identify_dataset(path)

    if not matched_datasets:
        continue

    inspection = inspect_data_file(
        path,
        matched_datasets
    )

    record = {
        "dataset": ";".join(matched_datasets),
        "file_name": path.name,
        "relative_path": str(
            path.relative_to(DATASET_ROOT)
        ),
        "extension": path.suffix.lower(),
        "size_bytes": path.stat().st_size,
        "actual_rows": inspection["actual_rows"],
        "actual_columns": inspection["actual_columns"],
        "target_present": None,
        "dimension_match": False,
        "readability": inspection["readability"],
        "status": "REVIEW_REQUIRED",
        "error": inspection["error"]
    }

    # --------------------------------------------------------
    # Validate target and dimensions for one identified dataset
    # --------------------------------------------------------

    if len(matched_datasets) == 1:

        dataset = matched_datasets[0]
        metadata = DATASET_METADATA[dataset]

        columns = inspection["column_names"] or []

        record["target_present"] = (
            metadata["target"] in columns
        )

        record["dimension_match"] = (
            inspection["actual_rows"] == metadata["rows"]
            and inspection["actual_columns"] == metadata["columns"]
        )

        if inspection["readability"] == "READABLE":
            record["status"] = "PASS"

        elif inspection["readability"] == "READ_ERROR":
            record["status"] = "FAIL"

    elif len(matched_datasets) > 1:

        record["status"] = "REVIEW_REQUIRED"

    else:

        record["status"] = "REVIEW_REQUIRED"

    artifact_records.append(record)


artifact_audit = pd.DataFrame(
    artifact_records
)

# ------------------------------------------------------------
# 7. IDENTIFY ORIGINAL RAW DATASET FILES
# ------------------------------------------------------------

def is_original_raw_file(path, dataset):

    relative_path = str(
        path.relative_to(DATASET_ROOT)
    ).lower()

    if not relative_path.startswith("data/raw/"):
        return False

    filename = path.name.lower()

    original_files = {
        "adult_income": {
            "adult_income.csv"
        },

        "bank_marketing": {
            "bank_marketing.csv",
            "bank-full.csv"
        },

        "diabetes_130us": {
            "diabetes_130us.csv",
            "diabetic_data.csv"
        }
    }

    return filename in original_files[dataset]



# ------------------------------------------------------------
# 8. DATASET-LEVEL COVERAGE AUDIT — CORRECTED
# ------------------------------------------------------------

coverage_records = []

for dataset, metadata in DATASET_METADATA.items():

    if artifact_audit.empty:
        matching = pd.DataFrame()
    else:
        matching = artifact_audit[
            artifact_audit["dataset"].str.split(";").apply(
                lambda values: dataset in values
            )
        ].copy()

    original_records = []

    for _, row in matching.iterrows():

        path = DATASET_ROOT / row["relative_path"]

        if not is_original_raw_file(path, dataset):
            continue

        # Explicit Boolean conversion avoids NumPy/Python
        # Boolean identity comparison errors.
        target_is_present = (
            False
            if pd.isna(row["target_present"])
            else bool(row["target_present"])
        )

        dimensions_match = (
            int(row["actual_rows"]) == metadata["rows"]
            and int(row["actual_columns"]) == metadata["columns"]
        )

        if (
            row["readability"] == "READABLE"
            and dimensions_match
            and target_is_present
        ):
            original_records.append(row["relative_path"])

    readable_count = (
        int((matching["readability"] == "READABLE").sum())
        if not matching.empty else 0
    )

    exact_match_count = len(original_records)

    if exact_match_count > 0:
        status = "PASS"
    elif readable_count > 0:
        status = "REVIEW_REQUIRED"
    else:
        status = "FAIL"

    coverage_records.append({
        "dataset": dataset,
        "expected_rows": metadata["rows"],
        "expected_columns": metadata["columns"],
        "target": metadata["target"],
        "generative_columns": metadata["generative_columns"],
        "matching_data_artifacts": len(matching),
        "readable_artifacts": readable_count,
        "original_raw_exact_matches": exact_match_count,
        "original_raw_files_verified": "; ".join(original_records),
        "artifact_found": len(matching) > 0,
        "original_dataset_verified": exact_match_count > 0,
        "status": status
    })

dataset_coverage_audit = pd.DataFrame(coverage_records)

print("\nDATASET COVERAGE SUMMARY")
print("-" * 80)
print(dataset_coverage_audit.to_string(index=False))

# ------------------------------------------------------------
# 9. DATASET-SPECIFIC ARTIFACT SUMMARY
# ------------------------------------------------------------

print("\nDATASET COVERAGE SUMMARY")
print("-" * 80)

print(
    dataset_coverage_audit.to_string(index=False)
)

print("\nDATA ARTIFACT INVENTORY")
print("-" * 80)

if not artifact_audit.empty:

    display_columns = [
        "dataset",
        "file_name",
        "relative_path",
        "actual_rows",
        "actual_columns",
        "target_present",
        "dimension_match",
        "readability",
        "status"
    ]

    print(
        artifact_audit[display_columns].to_string(
            index=False
        )
    )

else:

    print("No dataset-specific artifacts discovered.")


# ------------------------------------------------------------
# 10. SAVE AUDIT OUTPUTS
# ------------------------------------------------------------

coverage_path = (
    TABLES_DIR / "dataset_coverage_audit.csv"
)

artifact_path = (
    TABLES_DIR / "dataset_artifact_inventory.csv"
)

dataset_coverage_audit.to_csv(
    coverage_path,
    index=False
)

artifact_audit.to_csv(
    artifact_path,
    index=False
)


# ------------------------------------------------------------
# 11. FINAL STATUS
# ------------------------------------------------------------

total_datasets = len(dataset_coverage_audit)

passed_datasets = int(
    (dataset_coverage_audit["status"] == "PASS").sum()
)

review_datasets = int(
    (dataset_coverage_audit["status"] == "REVIEW_REQUIRED").sum()
)

failed_datasets = int(
    (dataset_coverage_audit["status"] == "FAIL").sum()
)

if (
    total_datasets == 3
    and passed_datasets == 3
    and review_datasets == 0
    and failed_datasets == 0
):

    section_15_status = "PASS"

elif failed_datasets > 0:

    section_15_status = "FAIL"

else:

    section_15_status = "REVIEW_REQUIRED"


# ------------------------------------------------------------
# 12. SAVE COMPLETION SUMMARY
# ------------------------------------------------------------

summary = {

    "section": 15,

    "section_name": "Dataset Coverage Audit",

    "audit_timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "project_root": str(DATASET_ROOT),

    "total_datasets": total_datasets,

    "passed_datasets": passed_datasets,

    "review_datasets": review_datasets,

    "failed_datasets": failed_datasets,

    "discovered_data_files": len(all_data_files),

    "matched_dataset_artifacts": len(artifact_audit),

    "status": section_15_status
}

summary_path = (
    TABLES_DIR / "dataset_coverage_audit_summary.json"
)

with open(
    summary_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=4
    )


# ------------------------------------------------------------
# 13. COMPLETION SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("SECTION 15 — COMPLETION SUMMARY")
print("=" * 80)

print("Total datasets:", total_datasets)
print("Passed datasets:", passed_datasets)
print("Review required:", review_datasets)
print("Failed datasets:", failed_datasets)

print("\nSECTION 15 STATUS:", section_15_status)

print("\nSaved coverage audit:", coverage_path)
print("Saved artifact inventory:", artifact_path)
print("Saved summary:", summary_path)

print("=" * 80)
print("SECTION 15 EXECUTION COMPLETED")
print("=" * 80)


SECTION 15 — DATASET COVERAGE AUDIT
Project root: /content/drive/MyDrive/SPP_GAN_Research
Project root exists: True

Total CSV/Parquet files discovered: 689

DATASET COVERAGE SUMMARY
--------------------------------------------------------------------------------
       dataset  expected_rows  expected_columns     target  generative_columns  matching_data_artifacts  readable_artifacts  original_raw_exact_matches                                                     original_raw_files_verified  artifact_found  original_dataset_verified status
  adult_income          48842                15     income                  15                       50                  50                           1                                                       data/raw/adult_income.csv            True                       True   PASS
bank_marketing          45211                17          y                  17                       54                  54                           2     data/raw/bank_ma

In [16]:
# ============================================================
# SECTION 16 — MODEL COVERAGE AUDIT
# ============================================================

import pandas as pd
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

MODEL_REGISTRY = {
    "statistical_baseline": {
        "aliases": [
            "gaussian_copula",
            "gaussian copula",
            "independent_marginal",
            "independent marginal",
            "bayesian_network",
            "bayesian network"
        ],
        "expected_notebooks": ["notebook_04"]
    },

    "tvae": {
        "aliases": ["tvae"],
        "expected_notebooks": ["notebook_05"]
    },

    "ctgan": {
        "aliases": ["ctgan"],
        "expected_notebooks": ["notebook_06"]
    },

    "dp_ctgan": {
        "aliases": ["dp_ctgan", "dp-ctgan"],
        "expected_notebooks": ["notebook_07"]
    },

    "sppgan": {
        "aliases": ["sppgan", "spp_gan", "spp-gan"],
        "expected_notebooks": [
            "notebook_12",
            "notebook_13"
        ]
    }
}

MODEL_EXTENSIONS = {
    ".csv", ".json", ".parquet",
    ".pkl", ".pickle", ".joblib",
    ".pt", ".pth", ".ckpt",
    ".h5", ".keras", ".onnx",
    ".yaml", ".yml", ".txt",
    ".log", ".npz", ".npy"
}

# ------------------------------------------------------------
# 2. PROJECT FILE DISCOVERY
# ------------------------------------------------------------

print("=" * 80)
print("SECTION 16 — MODEL COVERAGE AUDIT")
print("=" * 80)

print("Project root:", PROJECT_ROOT)
print("Project root exists:", PROJECT_ROOT.exists())

all_model_files = []

for path in PROJECT_ROOT.rglob("*"):

    if not path.is_file():
        continue

    # Exclude Notebook 22's own audit outputs.
    if "notebook_22" in path.parts:
        continue

    if path.suffix.lower() not in MODEL_EXTENSIONS:
        continue

    all_model_files.append(path)

all_model_files = sorted(set(all_model_files))

print(
    "\nTotal candidate project artifacts:",
    len(all_model_files)
)

# ------------------------------------------------------------
# 3. MODEL IDENTIFICATION
# ------------------------------------------------------------

def normalize_text(value):

    return str(value).lower().replace("-", "_").replace(" ", "_")


def identify_model(path):

    text_value = normalize_text(
        str(path.relative_to(PROJECT_ROOT))
    )

    matches = []

    for model, config in MODEL_REGISTRY.items():

        for alias in config["aliases"]:

            normalized_alias = normalize_text(alias)

            if normalized_alias in text_value:
                matches.append(model)
                break

    # DP-CTGAN must not also be counted as ordinary CTGAN.
    if "dp_ctgan" in matches:
        matches = [
            model for model in matches
            if model != "ctgan"
        ]

    return matches


# ------------------------------------------------------------
# 4. BUILD MODEL ARTIFACT INVENTORY
# ------------------------------------------------------------

model_artifact_records = []

for path in all_model_files:

    matched_models = identify_model(path)

    if not matched_models:
        continue

    for model in matched_models:

        model_artifact_records.append({

            "model": model,

            "file_name": path.name,

            "relative_path": str(
                path.relative_to(PROJECT_ROOT)
            ),

            "extension": path.suffix.lower(),

            "size_bytes": path.stat().st_size,

            "notebook_source": next(
                (
                    notebook
                    for notebook in MODEL_REGISTRY[model][
                        "expected_notebooks"
                    ]
                    if notebook in path.parts
                ),
                "OTHER_LOCATION"
            ),

            "readability": (
                "NONEMPTY"
                if path.stat().st_size > 0
                else "EMPTY"
            )
        })


model_artifact_inventory = pd.DataFrame(
    model_artifact_records
)

# ------------------------------------------------------------
# 5. MODEL COVERAGE SUMMARY
# ------------------------------------------------------------

model_coverage_records = []

for model, config in MODEL_REGISTRY.items():

    if model_artifact_inventory.empty:

        matching = pd.DataFrame()

    else:

        matching = model_artifact_inventory[
            model_artifact_inventory["model"] == model
        ]

    artifact_count = len(matching)

    nonempty_count = (
        int((matching["readability"] == "NONEMPTY").sum())
        if not matching.empty else 0
    )

    notebook_coverage = sorted(
        matching["notebook_source"].unique().tolist()
    ) if not matching.empty else []

    expected_notebooks = config["expected_notebooks"]

    found_expected_notebooks = [
        notebook
        for notebook in expected_notebooks
        if notebook in notebook_coverage
    ]

    missing_notebooks = [
        notebook
        for notebook in expected_notebooks
        if notebook not in found_expected_notebooks
    ]

    # At least one nonempty artifact must exist.
    if nonempty_count > 0 and not missing_notebooks:
        status = "PASS"

    elif artifact_count > 0:
        status = "REVIEW_REQUIRED"

    else:
        status = "FAIL"

    model_coverage_records.append({

        "model": model,

        "artifact_count": artifact_count,

        "nonempty_artifacts": nonempty_count,

        "expected_notebooks": "; ".join(
            expected_notebooks
        ),

        "verified_notebooks": "; ".join(
            found_expected_notebooks
        ),

        "missing_notebooks": "; ".join(
            missing_notebooks
        ),

        "artifact_found": artifact_count > 0,

        "status": status
    })


model_coverage_audit = pd.DataFrame(
    model_coverage_records
)

# ------------------------------------------------------------
# 6. DISPLAY MODEL COVERAGE
# ------------------------------------------------------------

print("\nMODEL COVERAGE SUMMARY")
print("-" * 80)

print(
    model_coverage_audit.to_string(index=False)
)

print("\nMODEL ARTIFACT INVENTORY")
print("-" * 80)

if not model_artifact_inventory.empty:

    print(
        model_artifact_inventory.to_string(
            index=False
        )
    )

else:

    print("No model-specific artifacts discovered.")

# ------------------------------------------------------------
# 7. SAVE AUDIT TABLES
# ------------------------------------------------------------

model_coverage_path = (
    TABLES_DIR / "model_coverage_audit.csv"
)

model_inventory_path = (
    TABLES_DIR / "model_artifact_inventory.csv"
)

model_coverage_audit.to_csv(
    model_coverage_path,
    index=False
)

model_artifact_inventory.to_csv(
    model_inventory_path,
    index=False
)

# ------------------------------------------------------------
# 8. FINAL STATUS
# ------------------------------------------------------------

total_models = len(model_coverage_audit)

passed_models = int(
    (model_coverage_audit["status"] == "PASS").sum()
)

review_models = int(
    (
        model_coverage_audit["status"]
        == "REVIEW_REQUIRED"
    ).sum()
)

failed_models = int(
    (model_coverage_audit["status"] == "FAIL").sum()
)

if (
    total_models == 5
    and passed_models == 5
    and review_models == 0
    and failed_models == 0
):

    section_16_status = "PASS"

elif failed_models > 0:

    section_16_status = "FAIL"

else:

    section_16_status = "REVIEW_REQUIRED"

# ------------------------------------------------------------
# 9. SAVE COMPLETION SUMMARY
# ------------------------------------------------------------

summary = {

    "section": 16,

    "section_name": "Model Coverage Audit",

    "audit_timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "project_root": str(PROJECT_ROOT),

    "candidate_artifacts_discovered": len(
        all_model_files
    ),

    "total_models": total_models,

    "passed_models": passed_models,

    "review_models": review_models,

    "failed_models": failed_models,

    "total_model_artifacts": len(
        model_artifact_inventory
    ),

    "status": section_16_status
}

summary_path = (
    TABLES_DIR / "model_coverage_audit_summary.json"
)

with open(
    summary_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 10. COMPLETION SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("SECTION 16 — COMPLETION SUMMARY")
print("=" * 80)

print("Total models:", total_models)
print("Passed models:", passed_models)
print("Review required:", review_models)
print("Failed models:", failed_models)

print(
    "Total model artifacts:",
    len(model_artifact_inventory)
)

print("\nSECTION 16 STATUS:", section_16_status)

print("\nSaved coverage audit:", model_coverage_path)
print("Saved artifact inventory:", model_inventory_path)
print("Saved summary:", summary_path)

print("=" * 80)
print("SECTION 16 EXECUTION COMPLETED")
print("=" * 80)


SECTION 16 — MODEL COVERAGE AUDIT
Project root: /content/drive/MyDrive/SPP_GAN_Research
Project root exists: True

Total candidate project artifacts: 1636

MODEL COVERAGE SUMMARY
--------------------------------------------------------------------------------
               model  artifact_count  nonempty_artifacts       expected_notebooks       verified_notebooks missing_notebooks  artifact_found status
statistical_baseline              88                  88              notebook_04              notebook_04                              True   PASS
                tvae              24                  24              notebook_05              notebook_05                              True   PASS
               ctgan              26                  26              notebook_06              notebook_06                              True   PASS
            dp_ctgan              51                  51              notebook_07              notebook_07                              True   PASS


In [17]:
# ============================================================
# SECTION 17 — REPETITION COVERAGE AUDIT
# ============================================================

import pandas as pd
import json
import re
from pathlib import Path
from datetime import datetime, timezone

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

EXPECTED_SEEDS = [
    int(seed) for seed in REPETITION_SEEDS
]

EXPECTED_REPETITIONS = {
    repetition: seed
    for repetition, seed in enumerate(
        EXPECTED_SEEDS,
        start=1
    )
}

EXPECTED_DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us"
]

# Search only formats that can contain readable metadata.
TEXT_EXTENSIONS = {
    ".csv",
    ".json",
    ".jsonl",
    ".txt",
    ".log",
    ".yaml",
    ".yml"
}

# Exclude Notebook 22 audit outputs.
EXCLUDED_DIRECTORY = "notebook_22"

print("=" * 80)
print("SECTION 17 — REPETITION COVERAGE AUDIT")
print("=" * 80)

print("Project root:", PROJECT_ROOT)
print("Expected seeds:", EXPECTED_SEEDS)

# ------------------------------------------------------------
# 2. DISCOVER PROJECT METADATA FILES
# ------------------------------------------------------------

metadata_files = []

for path in PROJECT_ROOT.rglob("*"):

    if not path.is_file():
        continue

    if EXCLUDED_DIRECTORY in path.parts:
        continue

    if path.suffix.lower() not in TEXT_EXTENSIONS:
        continue

    metadata_files.append(path)

metadata_files = sorted(set(metadata_files))

print(
    "\nMetadata files discovered:",
    len(metadata_files)
)

# ------------------------------------------------------------
# 3. NORMALIZE SEED FIELD NAMES
# ------------------------------------------------------------

SEED_KEYS = {
    "seed",
    "experiment_seed",
    "random_seed",
    "training_seed",
    "repetition_seed",
    "repeat_seed",
    "run_seed",
    "model_seed"
}


def normalize_key(key):

    return re.sub(
        r"[^a-z0-9]",
        "",
        str(key).lower()
    )


NORMALIZED_SEED_KEYS = {
    normalize_key(key)
    for key in SEED_KEYS
}


def extract_seed_values(obj, source_path, evidence):

    """
    Extract only values explicitly associated with
    seed-related metadata keys.
    """

    if isinstance(obj, dict):

        for key, value in obj.items():

            normalized_key = normalize_key(key)

            if normalized_key in NORMALIZED_SEED_KEYS:

                try:
                    seed_value = int(value)

                    if seed_value in EXPECTED_SEEDS:

                        evidence.append({
                            "seed": seed_value,
                            "source_path": str(
                                source_path.relative_to(
                                    PROJECT_ROOT
                                )
                            ),
                            "evidence_type": "JSON_SEED_FIELD",
                            "field": str(key)
                        })

                except (TypeError, ValueError):
                    pass

            extract_seed_values(
                value,
                source_path,
                evidence
            )

    elif isinstance(obj, list):

        for item in obj:

            extract_seed_values(
                item,
                source_path,
                evidence
            )


# ------------------------------------------------------------
# 4. SCAN JSON AND CSV METADATA
# ------------------------------------------------------------

seed_evidence = []

for path in metadata_files:

    suffix = path.suffix.lower()

    try:

        if suffix == ".json":

            with open(
                path,
                "r",
                encoding="utf-8"
            ) as f:

                content = json.load(f)

            extract_seed_values(
                content,
                path,
                seed_evidence
            )

        elif suffix == ".csv":

            # Inspect headers first to locate explicit seed fields.
            header = pd.read_csv(
                path,
                nrows=0,
                encoding="utf-8"
            )

            seed_columns = [
                column
                for column in header.columns
                if normalize_key(column)
                in NORMALIZED_SEED_KEYS
            ]

            if not seed_columns:
                continue

            # Read only seed columns to keep memory usage low.
            for chunk in pd.read_csv(
                path,
                usecols=seed_columns,
                chunksize=20000,
                encoding="utf-8"
            ):

                for column in seed_columns:

                    values = pd.to_numeric(
                        chunk[column],
                        errors="coerce"
                    )

                    for seed in EXPECTED_SEEDS:

                        if (values == seed).any():

                            seed_evidence.append({

                                "seed": seed,

                                "source_path": str(
                                    path.relative_to(
                                        PROJECT_ROOT
                                    )
                                ),

                                "evidence_type": "CSV_SEED_COLUMN",

                                "field": str(column)
                            })

        elif suffix in {
            ".txt",
            ".log",
            ".yaml",
            ".yml",
            ".jsonl"
        }:

            # Scan line by line without loading large files.
            with open(
                path,
                "r",
                encoding="utf-8",
                errors="ignore"
            ) as f:

                for line_number, line in enumerate(
                    f,
                    start=1
                ):

                    # Match explicit key-value seed declarations.
                    pattern = (
                        r"(?i)\b"
                        r"(experiment_seed|random_seed|"
                        r"training_seed|repetition_seed|"
                        r"repeat_seed|run_seed|model_seed|seed)"
                        r"\b\s*[:=]\s*(3026|3027|3028|3029|3030)\b"
                    )

                    for match in re.finditer(
                        pattern,
                        line
                    ):

                        seed = int(match.group(2))

                        seed_evidence.append({

                            "seed": seed,

                            "source_path": str(
                                path.relative_to(
                                    PROJECT_ROOT
                                )
                            ),

                            "evidence_type": "TEXT_SEED_DECLARATION",

                            "field": match.group(1),

                            "line_number": line_number
                        })

    except Exception as e:

        print(
            "Metadata inspection warning:",
            path.name,
            str(e)[:200]
        )


# ------------------------------------------------------------
# 5. REMOVE DUPLICATE EVIDENCE RECORDS
# ------------------------------------------------------------

if seed_evidence:

    seed_evidence_df = pd.DataFrame(
        seed_evidence
    )

    seed_evidence_df = seed_evidence_df.drop_duplicates(
        subset=[
            "seed",
            "source_path",
            "evidence_type",
            "field"
        ]
    ).reset_index(drop=True)

else:

    seed_evidence_df = pd.DataFrame(
        columns=[
            "seed",
            "source_path",
            "evidence_type",
            "field"
        ]
    )


print(
    "\nExplicit seed evidence records:",
    len(seed_evidence_df)
)

# ------------------------------------------------------------
# 6. IDENTIFY REPETITION-SPECIFIC MODEL ARTIFACTS
# ------------------------------------------------------------

repetition_artifacts = []

MODEL_ARTIFACT_EXTENSIONS = {
    ".pt",
    ".pth",
    ".ckpt",
    ".pkl",
    ".pickle",
    ".joblib",
    ".h5",
    ".keras",
    ".onnx"
}

for path in PROJECT_ROOT.rglob("*"):

    if not path.is_file():
        continue

    if EXCLUDED_DIRECTORY in path.parts:
        continue

    if path.suffix.lower() not in MODEL_ARTIFACT_EXTENSIONS:
        continue

    relative_path = str(
        path.relative_to(PROJECT_ROOT)
    ).lower()

    repetition_match = re.search(
        r"repetition[_-]?0?([1-5])",
        relative_path
    )

    if not repetition_match:
        continue

    repetition_number = int(
        repetition_match.group(1)
    )

    repetition_artifacts.append({

        "repetition": repetition_number,

        "artifact_path": str(
            path.relative_to(PROJECT_ROOT)
        ),

        "size_bytes": path.stat().st_size,

        "nonempty": path.stat().st_size > 0
    })


repetition_artifacts_df = pd.DataFrame(
    repetition_artifacts
)

# ------------------------------------------------------------
# 7. BUILD SEED-TO-REPETITION COVERAGE
# ------------------------------------------------------------

repetition_records = []

for repetition, seed in EXPECTED_REPETITIONS.items():

    if seed_evidence_df.empty:

        matching_evidence = pd.DataFrame()

    else:

        matching_evidence = seed_evidence_df[
            seed_evidence_df["seed"] == seed
        ]

    if repetition_artifacts_df.empty:

        matching_artifacts = pd.DataFrame()

    else:

        matching_artifacts = repetition_artifacts_df[
            repetition_artifacts_df["repetition"]
            == repetition
        ]

    evidence_count = len(matching_evidence)

    artifact_count = len(matching_artifacts)

    evidence_paths = (
        matching_evidence["source_path"]
        .drop_duplicates()
        .tolist()
        if not matching_evidence.empty
        else []
    )

    artifact_paths = (
        matching_artifacts["artifact_path"]
        .drop_duplicates()
        .tolist()
        if not matching_artifacts.empty
        else []
    )

    # Seed is verified only when an explicit metadata field
    # contains that exact seed.
    seed_verified = evidence_count > 0

    # Repetition artifacts are reported separately because
    # their filenames alone do not prove the seed mapping.
    repetition_artifacts_found = artifact_count > 0

    if seed_verified and repetition_artifacts_found:
        status = "PASS"

    elif seed_verified or repetition_artifacts_found:
        status = "REVIEW_REQUIRED"

    else:
        status = "FAIL"

    repetition_records.append({

        "repetition": repetition,

        "expected_seed": seed,

        "seed_evidence_count": evidence_count,

        "seed_verified": seed_verified,

        "repetition_artifact_count": artifact_count,

        "repetition_artifacts_found": (
            repetition_artifacts_found
        ),

        "evidence_paths": "; ".join(
            evidence_paths[:20]
        ),

        "artifact_paths": "; ".join(
            artifact_paths[:20]
        ),

        "status": status
    })


repetition_coverage_audit = pd.DataFrame(
    repetition_records
)

# ------------------------------------------------------------
# 8. DATASET-LEVEL SEED COVERAGE
# ------------------------------------------------------------

dataset_seed_records = []

for dataset in EXPECTED_DATASETS:

    for repetition, seed in EXPECTED_REPETITIONS.items():

        if seed_evidence_df.empty:

            matching = pd.DataFrame()

        else:

            matching = seed_evidence_df[
                (
                    seed_evidence_df["seed"] == seed
                )
                &
                (
                    seed_evidence_df["source_path"].str.contains(
                        dataset,
                        case=False,
                        regex=False
                    )
                )
            ]

        evidence_count = len(matching)

        dataset_seed_records.append({

            "dataset": dataset,

            "repetition": repetition,

            "expected_seed": seed,

            "seed_evidence_count": evidence_count,

            "seed_verified": evidence_count > 0,

            "status": (
                "PASS"
                if evidence_count > 0
                else "REVIEW_REQUIRED"
            )
        })


dataset_repetition_coverage = pd.DataFrame(
    dataset_seed_records
)

# ------------------------------------------------------------
# 9. DISPLAY RESULTS
# ------------------------------------------------------------

print("\nREPETITION COVERAGE SUMMARY")
print("-" * 80)

print(
    repetition_coverage_audit.to_string(
        index=False
    )
)

print("\nDATASET-LEVEL REPETITION COVERAGE")
print("-" * 80)

print(
    dataset_repetition_coverage.to_string(
        index=False
    )
)

# ------------------------------------------------------------
# 10. SAVE AUDIT OUTPUTS
# ------------------------------------------------------------

repetition_coverage_path = (
    TABLES_DIR / "repetition_coverage_audit.csv"
)

seed_evidence_path = (
    TABLES_DIR / "repetition_seed_evidence.csv"
)

repetition_artifact_path = (
    TABLES_DIR / "repetition_model_artifact_inventory.csv"
)

dataset_repetition_path = (
    TABLES_DIR / "dataset_repetition_coverage.csv"
)

repetition_coverage_audit.to_csv(
    repetition_coverage_path,
    index=False
)

seed_evidence_df.to_csv(
    seed_evidence_path,
    index=False
)

repetition_artifacts_df.to_csv(
    repetition_artifact_path,
    index=False
)

dataset_repetition_coverage.to_csv(
    dataset_repetition_path,
    index=False
)

# ------------------------------------------------------------
# 11. FINAL STATUS
# ------------------------------------------------------------

total_repetitions = len(
    repetition_coverage_audit
)

passed_repetitions = int(
    (
        repetition_coverage_audit["status"]
        == "PASS"
    ).sum()
)

review_repetitions = int(
    (
        repetition_coverage_audit["status"]
        == "REVIEW_REQUIRED"
    ).sum()
)

failed_repetitions = int(
    (
        repetition_coverage_audit["status"]
        == "FAIL"
    ).sum()
)

expected_dataset_repetitions = (
    len(EXPECTED_DATASETS)
    * len(EXPECTED_SEEDS)
)

verified_dataset_repetitions = int(
    dataset_repetition_coverage[
        "seed_verified"
    ].sum()
)

if (
    total_repetitions == 5
    and passed_repetitions == 5
    and verified_dataset_repetitions
    == expected_dataset_repetitions
):

    section_17_status = "PASS"

elif failed_repetitions > 0:

    section_17_status = "FAIL"

else:

    section_17_status = "REVIEW_REQUIRED"


# ------------------------------------------------------------
# 12. SAVE COMPLETION SUMMARY
# ------------------------------------------------------------

summary = {

    "section": 17,

    "section_name": "Repetition Coverage Audit",

    "audit_timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "expected_seeds": EXPECTED_SEEDS,

    "expected_repetitions": total_repetitions,

    "passed_repetitions": passed_repetitions,

    "review_repetitions": review_repetitions,

    "failed_repetitions": failed_repetitions,

    "expected_dataset_repetitions": (
        expected_dataset_repetitions
    ),

    "verified_dataset_repetitions": (
        verified_dataset_repetitions
    ),

    "explicit_seed_evidence_records": len(
        seed_evidence_df
    ),

    "repetition_model_artifacts": len(
        repetition_artifacts_df
    ),

    "status": section_17_status
}

summary_path = (
    TABLES_DIR / "repetition_coverage_audit_summary.json"
)

with open(
    summary_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=4
    )


# ------------------------------------------------------------
# 13. COMPLETION SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("SECTION 17 — COMPLETION SUMMARY")
print("=" * 80)

print("Expected repetitions:", total_repetitions)
print("Passed repetitions:", passed_repetitions)
print("Review required:", review_repetitions)
print("Failed repetitions:", failed_repetitions)

print(
    "Expected dataset-repetition combinations:",
    expected_dataset_repetitions
)

print(
    "Verified dataset-repetition combinations:",
    verified_dataset_repetitions
)

print(
    "Explicit seed evidence records:",
    len(seed_evidence_df)
)

print(
    "Repetition model artifacts:",
    len(repetition_artifacts_df)
)

print("\nSECTION 17 STATUS:", section_17_status)

print("\nSaved repetition audit:", repetition_coverage_path)
print("Saved seed evidence:", seed_evidence_path)
print("Saved repetition artifact inventory:", repetition_artifact_path)
print("Saved dataset coverage:", dataset_repetition_path)
print("Saved summary:", summary_path)

print("=" * 80)
print("SECTION 17 EXECUTION COMPLETED")
print("=" * 80)


SECTION 17 — REPETITION COVERAGE AUDIT
Project root: /content/drive/MyDrive/SPP_GAN_Research
Expected seeds: [3026, 3027, 3028, 3029, 3030]

Metadata files discovered: 987
Metadata inspection warning: notebook_12B_training_registry.csv No columns to parse from file
Metadata inspection warning: feature_type_restoration_audit.csv No columns to parse from file
Metadata inspection warning: target_restoration_errors.csv No columns to parse from file
Metadata inspection warning: dataset_level_comparison.csv No columns to parse from file
Metadata inspection warning: metric_level_comparison.csv No columns to parse from file
Metadata inspection warning: privacy_utility_ranking.csv No columns to parse from file
Metadata inspection warning: robustness_invalid_input_records.csv No columns to parse from file
Metadata inspection warning: robustness_valid_input_records.csv No columns to parse from file
Metadata inspection warning: section_04_fidelity_exceptions.csv No columns to parse from file
Metad

In [19]:

# ============================================================
# SECTION 18 — METRIC COVERAGE AUDIT
# ============================================================

import json
import re
import numpy as np
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 1. VALIDATE PROJECT PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path(PROJECT_ROOT)
TABLES_DIR = Path(TABLES_DIR)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"Project root not found: {PROJECT_ROOT}")

TABLES_DIR.mkdir(parents=True, exist_ok=True)

AUDIT_DIR = PROJECT_ROOT / "results" / "notebooks" / "notebook_22"

# ------------------------------------------------------------
# 2. DEFINE EXPECTED METRICS AND ALIASES
# ------------------------------------------------------------

METRIC_REGISTRY = {
    "fidelity": {
        "wasserstein": ["wasserstein", "wasserstein_distance"],
        "js_divergence": ["js_divergence", "jensen_shannon"],
        "mmd": ["mmd", "maximum_mean_discrepancy"],
        "correlation_error": ["correlation_error", "corr_error"]
    },
    "utility": {
        "accuracy": ["accuracy", "acc"],
        "macro_f1": ["macro_f1", "f1_macro"],
        "weighted_f1": ["weighted_f1", "f1_weighted"],
        "precision": ["precision"],
        "recall": ["recall"],
        "tstr": ["tstr", "train_synthetic_test_real"]
    },
    "privacy": {
        "epsilon": ["epsilon", "achieved_epsilon"],
        "delta": ["delta", "dp_delta"],
        "reidentification_risk": [
            "reidentification_risk",
            "re_identification_risk",
            "reid_risk"
        ],
        "membership_inference": [
            "membership_inference",
            "membership_inference_auc",
            "mia_auc"
        ],
        "nearest_neighbor": [
            "nearest_neighbor",
            "nearest_neighbour",
            "nn_distance",
            "nearest_neighbor_distance"
        ]
    },
    "computational": {
        "runtime_seconds": [
            "runtime_seconds",
            "runtime_sec",
            "runtime"
        ],
        "training_time": [
            "training_time",
            "training_time_seconds",
            "training_duration"
        ],
        "memory_mb": [
            "memory_mb",
            "memory_usage_mb",
            "peak_memory_mb"
        ]
    }
}

# ------------------------------------------------------------
# 3. NORMALIZE METRIC NAMES
# ------------------------------------------------------------

def normalize_metric_name(value):
    return re.sub(r"[^a-z0-9]", "", str(value).lower())


NORMALIZED_REGISTRY = {
    category: {
        metric: [
            normalize_metric_name(alias)
            for alias in aliases
        ]
        for metric, aliases in metrics.items()
    }
    for category, metrics in METRIC_REGISTRY.items()
}

# ------------------------------------------------------------
# 4. DISCOVER SOURCE ARTIFACTS
# ------------------------------------------------------------

EXCLUDED_DIRS = {
    "notebook_22",
    ".ipynb_checkpoints",
    "__pycache__"
}

source_files = []

for path in PROJECT_ROOT.rglob("*"):

    if not path.is_file():
        continue

    if any(part in EXCLUDED_DIRS for part in path.parts):
        continue

    if path.suffix.lower() in {".csv", ".json"}:
        source_files.append(path)

source_files = sorted(set(source_files))

print("Metric source artifacts discovered:", len(source_files))

# ------------------------------------------------------------
# 5. INITIALIZE METRIC-LEVEL RECORDS
# ------------------------------------------------------------

metric_records = []

for category, metrics in METRIC_REGISTRY.items():

    for metric, aliases in metrics.items():

        metric_records.append({
            "category": category,
            "metric": metric,
            "aliases": "; ".join(aliases),
            "source_artifact_count": 0,
            "numeric_measurements_found": 0,
            "source_artifacts": [],
            "coverage_status": "NOT_FOUND"
        })

metric_lookup = {
    (record["category"], record["metric"]): record
    for record in metric_records
}

# ------------------------------------------------------------
# 6. MATCH METRIC COLUMNS
# ------------------------------------------------------------

def find_metric_matches(column_names):

    matches = {}

    normalized_columns = {
        column: normalize_metric_name(column)
        for column in column_names
    }

    for category, metrics in NORMALIZED_REGISTRY.items():

        for metric, aliases in metrics.items():

            matched_columns = []

            for original_column, normalized_column in normalized_columns.items():

                if any(
                    alias in normalized_column
                    for alias in aliases
                ):
                    matched_columns.append(original_column)

            if matched_columns:
                matches[(category, metric)] = matched_columns

    return matches

# ------------------------------------------------------------
# 7. INSPECT CSV FILES IN BOUNDED CHUNKS
# ------------------------------------------------------------

csv_read_errors = []
json_read_errors = []

for path in source_files:

    if path.suffix.lower() != ".csv":
        continue

    try:
        header = pd.read_csv(
            path,
            nrows=0,
            low_memory=True
        ).columns.astype(str).tolist()

        matches = find_metric_matches(header)

        if not matches:
            continue

        required_columns = sorted({
            column
            for columns in matches.values()
            for column in columns
        })

        artifact_counts = {
            key: 0
            for key in matches
        }

        for chunk in pd.read_csv(
            path,
            usecols=required_columns,
            chunksize=5000,
            low_memory=True
        ):

            for key, columns in matches.items():

                for column in columns:

                    numeric_values = pd.to_numeric(
                        chunk[column],
                        errors="coerce"
                    )

                    numeric_values = numeric_values.replace(
                        [np.inf, -np.inf],
                        np.nan
                    )

                    artifact_counts[key] += int(
                        numeric_values.notna().sum()
                    )

        for key, count in artifact_counts.items():

            if count > 0:

                record = metric_lookup[key]

                record["source_artifact_count"] += 1
                record["numeric_measurements_found"] += count
                record["source_artifacts"].append(
                    str(path.relative_to(PROJECT_ROOT))
                )

    except pd.errors.EmptyDataError:
        continue

    except Exception as exc:
        csv_read_errors.append({
            "artifact": str(path.relative_to(PROJECT_ROOT)),
            "error": str(exc)[:250]
        })

# ------------------------------------------------------------
# 8. INSPECT JSON METADATA
# ------------------------------------------------------------

def extract_json_numeric_metrics(obj, matches=None):

    if matches is None:
        matches = {}

    if isinstance(obj, dict):

        for key, value in obj.items():

            normalized_key = normalize_metric_name(key)

            if isinstance(value, (int, float)) and not isinstance(value, bool):

                if np.isfinite(value):

                    for category, metrics in NORMALIZED_REGISTRY.items():

                        for metric, aliases in metrics.items():

                            if any(
                                alias in normalized_key
                                for alias in aliases
                            ):

                                matches.setdefault(
                                    (category, metric), 0
                                )

                                matches[(category, metric)] += 1

            extract_json_numeric_metrics(value, matches)

    elif isinstance(obj, list):

        for item in obj:
            extract_json_numeric_metrics(item, matches)

    return matches


for path in source_files:

    if path.suffix.lower() != ".json":
        continue

    try:

        with open(path, "r", encoding="utf-8") as file:
            data = json.load(file)

        json_matches = extract_json_numeric_metrics(data)

        for key, count in json_matches.items():

            if count > 0:

                record = metric_lookup[key]

                record["source_artifact_count"] += 1
                record["numeric_measurements_found"] += count
                record["source_artifacts"].append(
                    str(path.relative_to(PROJECT_ROOT))
                )

    except Exception as exc:

        json_read_errors.append({
            "artifact": str(path.relative_to(PROJECT_ROOT)),
            "error": str(exc)[:250]
        })

# ------------------------------------------------------------
# 9. FINALIZE METRIC-LEVEL COVERAGE
# ------------------------------------------------------------

for record in metric_records:

    record["source_artifacts"] = "; ".join(
        sorted(set(record["source_artifacts"]))
    )

    if record["numeric_measurements_found"] > 0:
        record["coverage_status"] = "NUMERIC_EVIDENCE_FOUND"
    else:
        record["coverage_status"] = "NO_NUMERIC_EVIDENCE"

metric_coverage_audit = pd.DataFrame(metric_records)

# ------------------------------------------------------------
# 10. CATEGORY-LEVEL SUMMARY
# ------------------------------------------------------------

category_records = []

for category, metrics in METRIC_REGISTRY.items():

    subset = metric_coverage_audit[
        metric_coverage_audit["category"] == category
    ]

    expected_count = len(metrics)

    found_count = int(
        (subset["numeric_measurements_found"] > 0).sum()
    )

    category_records.append({
        "category": category,
        "metrics_expected": expected_count,
        "metrics_with_numeric_evidence": found_count,
        "metrics_without_numeric_evidence": (
            expected_count - found_count
        ),
        "coverage_percentage": round(
            100 * found_count / expected_count, 2
        ) if expected_count else 0,
        "coverage_status": (
            "COMPLETE"
            if found_count == expected_count
            else "PARTIAL"
            if found_count > 0
            else "NO_EVIDENCE"
        )
    })

category_coverage_audit = pd.DataFrame(category_records)

# ------------------------------------------------------------
# 11. SAVE AUDIT OUTPUTS
# ------------------------------------------------------------

metric_coverage_audit.to_csv(
    TABLES_DIR / "metric_coverage_audit.csv",
    index=False
)

category_coverage_audit.to_csv(
    TABLES_DIR / "metric_category_coverage_audit.csv",
    index=False
)

pd.DataFrame(csv_read_errors).to_csv(
    TABLES_DIR / "metric_csv_read_errors.csv",
    index=False
)

pd.DataFrame(json_read_errors).to_csv(
    TABLES_DIR / "metric_json_read_errors.csv",
    index=False
)

# ------------------------------------------------------------
# 12. GENERATE SUMMARY AND STATUS
# ------------------------------------------------------------

total_metrics = len(metric_coverage_audit)

metrics_found = int(
    (metric_coverage_audit["numeric_measurements_found"] > 0).sum()
)

metrics_missing = total_metrics - metrics_found

total_measurements = int(
    metric_coverage_audit["numeric_measurements_found"].sum()
)

summary = {
    "expected_metrics": total_metrics,
    "metrics_with_numeric_evidence": metrics_found,
    "metrics_without_numeric_evidence": metrics_missing,
    "numeric_measurements_found": total_measurements,
    "source_artifacts_discovered": len(source_files),
    "csv_read_errors": len(csv_read_errors),
    "json_read_errors": len(json_read_errors),
    "coverage_percentage": round(
        100 * metrics_found / total_metrics, 2
    ) if total_metrics else 0
}

# A missing metric is not silently treated as PASS.
# Some metrics may be model-specific or not applicable.
if metrics_missing == 0 and not csv_read_errors and not json_read_errors:
    section_status = "PASS"
elif metrics_found > 0:
    section_status = "REVIEW_REQUIRED"
else:
    section_status = "FAIL"

summary["section_status"] = section_status

with open(
    TABLES_DIR / "metric_coverage_audit_summary.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(summary, file, indent=4)

# ------------------------------------------------------------
# 13. DISPLAY RESULTS
# ------------------------------------------------------------

print("\nMETRIC-LEVEL COVERAGE")
print("=" * 90)
print(
    metric_coverage_audit[
        [
            "category",
            "metric",
            "source_artifact_count",
            "numeric_measurements_found",
            "coverage_status"
        ]
    ].to_string(index=False)
)

print("\nCATEGORY-LEVEL COVERAGE")
print("=" * 90)
print(category_coverage_audit.to_string(index=False))

print("\nSECTION 18 COMPLETION SUMMARY")
print("=" * 90)

for key, value in summary.items():
    print(f"{key}: {value}")

print("\nSECTION 18 STATUS:", section_status)

print("\nSaved metric audit files:")
print(TABLES_DIR / "metric_coverage_audit.csv")
print(TABLES_DIR / "metric_category_coverage_audit.csv")
print(TABLES_DIR / "metric_coverage_audit_summary.json")
print(TABLES_DIR / "metric_csv_read_errors.csv")
print(TABLES_DIR / "metric_json_read_errors.csv")

print("=" * 90)
print("SECTION 18 EXECUTION COMPLETED")
print("=" * 90)


Metric source artifacts discovered: 984

METRIC-LEVEL COVERAGE
     category                metric  source_artifact_count  numeric_measurements_found        coverage_status
     fidelity           wasserstein                      9                         166 NUMERIC_EVIDENCE_FOUND
     fidelity         js_divergence                      6                         224 NUMERIC_EVIDENCE_FOUND
     fidelity                   mmd                      1                           1 NUMERIC_EVIDENCE_FOUND
     fidelity     correlation_error                      2                          16 NUMERIC_EVIDENCE_FOUND
      utility              accuracy                     37                         802 NUMERIC_EVIDENCE_FOUND
      utility              macro_f1                     21                         517 NUMERIC_EVIDENCE_FOUND
      utility           weighted_f1                     21                         511 NUMERIC_EVIDENCE_FOUND
      utility             precision                     1

In [22]:

# ============================================================
# SECTION 19 — PRIVACY COVERAGE AUDIT
# Complete Updated Version
# ============================================================

import json
import re
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# 1. PROJECT PATH VALIDATION
# ============================================================

PROJECT_ROOT = Path(PROJECT_ROOT)
TABLES_DIR = Path(TABLES_DIR)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project root not found: {PROJECT_ROOT}"
    )

TABLES_DIR.mkdir(parents=True, exist_ok=True)

NOTEBOOK_12_DIR = (
    PROJECT_ROOT / "results" / "notebooks" / "notebook_12"
)

EXCLUDED_DIRS = {
    "notebook_22",
    ".ipynb_checkpoints",
    "__pycache__"
}

# ============================================================
# 2. PRIVACY FIELD REGISTRY
# ============================================================

PRIVACY_FIELDS = {
    "formal_dp": {
        "achieved_epsilon": [
            "achieved_epsilon",
            "epsilon"
        ],
        "achieved_delta": [
            "achieved_delta",
            "dp_delta",
            "delta"
        ],
        "noise_multiplier": [
            "noise_multiplier",
            "dp_noise_multiplier"
        ],
        "accountant": [
            "accountant",
            "privacy_accountant",
            "accountant_type"
        ],
        "privacy_boundary": [
            "privacy_boundary",
            "privacy_scope",
            "protected_component",
            "protection_scope"
        ],
        "rdp": [
            "rdp",
            "renyi_differential_privacy",
            "renyi_accountant"
        ],
        "generator_private": [
            "generator_private",
            "generator_dp_enabled",
            "is_generator_private"
        ],
        "discriminator_private": [
            "discriminator_private",
            "critic_private",
            "discriminator_dp_enabled",
            "critic_dp_enabled",
            "is_discriminator_private",
            "is_critic_private"
        ],
        "end_to_end_claim": [
            "end_to_end_privacy_claim",
            "end_to_end_dp",
            "end_to_end_private"
        ]
    },

    "empirical_privacy": {
        "reidentification": [
            "reidentification",
            "re_identification",
            "reidentification_risk",
            "reid_risk"
        ],
        "membership_inference": [
            "membership_inference",
            "membership_inference_auc",
            "mia_auc",
            "mia_accuracy"
        ],
        "nearest_neighbor": [
            "nearest_neighbor",
            "nearest_neighbour",
            "nearest_neighbor_distance",
            "nn_distance"
        ],
        "disclosure": [
            "disclosure",
            "disclosure_risk",
            "information_disclosure"
        ],
        "attack": [
            "attack",
            "attack_success_rate",
            "attack_accuracy",
            "attack_auc"
        ]
    }
}

# ============================================================
# 3. NORMALIZATION AND EVIDENCE HELPERS
# ============================================================

def normalize_name(value):
    return re.sub(
        r"[^a-z0-9]",
        "",
        str(value).lower()
    )


NORMALIZED_FIELDS = {
    category: {
        field: [
            normalize_name(alias)
            for alias in aliases
        ]
        for field, aliases in fields.items()
    }
    for category, fields in PRIVACY_FIELDS.items()
}


def match_privacy_fields(names):

    matches = {}

    for category, fields in NORMALIZED_FIELDS.items():

        for field, aliases in fields.items():

            matched_columns = []

            for name in names:

                normalized = normalize_name(name)

                if any(
                    alias == normalized
                    or alias in normalized
                    for alias in aliases
                ):
                    matched_columns.append(str(name))

            if matched_columns:

                matches[(category, field)] = (
                    matched_columns
                )

    return matches


def is_numeric_value(value):

    if isinstance(value, (bool, np.bool_)):
        return False

    try:

        number = float(value)

        return bool(np.isfinite(number))

    except (TypeError, ValueError):
        return False


def is_nonempty(value):

    if value is None:
        return False

    if isinstance(value, str):
        return bool(value.strip())

    if isinstance(value, (dict, list)):
        return len(value) > 0

    return True


def classify_boolean(value):

    if isinstance(value, (bool, np.bool_)):
        return bool(value)

    if isinstance(value, (int, float)):

        if value == 1:
            return True

        if value == 0:
            return False

    if isinstance(value, str):

        normalized = value.strip().lower()

        if normalized in {"true", "yes", "1", "enabled"}:
            return True

        if normalized in {"false", "no", "0", "disabled"}:
            return False

    return None


# ============================================================
# 4. DISCOVER PROJECT ARTIFACTS
# ============================================================

source_files = []

for path in PROJECT_ROOT.rglob("*"):

    if not path.is_file():
        continue

    relative_parts = path.relative_to(
        PROJECT_ROOT
    ).parts

    if any(
        part in EXCLUDED_DIRS
        for part in relative_parts
    ):
        continue

    if path.suffix.lower() in {".csv", ".json"}:
        source_files.append(path)

source_files = sorted(set(source_files))

print(
    "Privacy source artifacts discovered:",
    len(source_files)
)

# ============================================================
# 5. INITIALIZE FIELD-LEVEL RECORDS
# ============================================================

privacy_records = []

for category, fields in PRIVACY_FIELDS.items():

    for field, aliases in fields.items():

        privacy_records.append({
            "privacy_category": category,
            "privacy_field": field,
            "aliases": "; ".join(aliases),
            "source_artifact_count": 0,
            "numeric_measurements_found": 0,
            "nonempty_values_found": 0,
            "true_values_found": 0,
            "false_values_found": 0,
            "source_artifacts": [],
            "matched_source_fields": [],
            "evidence_status": "NOT_FOUND"
        })

privacy_lookup = {
    (
        record["privacy_category"],
        record["privacy_field"]
    ): record
    for record in privacy_records
}

# ============================================================
# 6. INSPECT CSV ARTIFACTS
# ============================================================

csv_read_errors = []

for path in source_files:

    if path.suffix.lower() != ".csv":
        continue

    try:

        header = pd.read_csv(
            path,
            nrows=0,
            low_memory=True
        ).columns.astype(str).tolist()

        matches = match_privacy_fields(header)

        if not matches:
            continue

        selected_columns = sorted({
            column
            for columns in matches.values()
            for column in columns
        })

        file_counts = {
            key: {
                "numeric": 0,
                "nonempty": 0,
                "true": 0,
                "false": 0
            }
            for key in matches
        }

        for chunk in pd.read_csv(
            path,
            usecols=selected_columns,
            chunksize=5000,
            low_memory=True
        ):

            for key, columns in matches.items():

                for column in columns:

                    values = chunk[column]

                    file_counts[key]["nonempty"] += int(
                        values.notna().sum()
                    )

                    numeric = pd.to_numeric(
                        values,
                        errors="coerce"
                    )

                    numeric = numeric.replace(
                        [np.inf, -np.inf],
                        np.nan
                    )

                    file_counts[key]["numeric"] += int(
                        numeric.notna().sum()
                    )

                    boolean_values = values.map(
                        classify_boolean
                    )

                    file_counts[key]["true"] += int(
                        (boolean_values == True).sum()
                    )

                    file_counts[key]["false"] += int(
                        (boolean_values == False).sum()
                    )

        relative_path = str(
            path.relative_to(PROJECT_ROOT)
        )

        for key, counts in file_counts.items():

            record = privacy_lookup[key]

            if (
                counts["numeric"] > 0
                or counts["nonempty"] > 0
            ):

                record["source_artifact_count"] += 1

                record["numeric_measurements_found"] += (
                    counts["numeric"]
                )

                record["nonempty_values_found"] += (
                    counts["nonempty"]
                )

                record["true_values_found"] += (
                    counts["true"]
                )

                record["false_values_found"] += (
                    counts["false"]
                )

                record["source_artifacts"].append(
                    relative_path
                )

                record["matched_source_fields"].extend(
                    matches[key]
                )

    except pd.errors.EmptyDataError:
        continue

    except Exception as exc:

        csv_read_errors.append({
            "artifact": str(
                path.relative_to(PROJECT_ROOT)
            ),
            "error": str(exc)[:250]
        })

# ============================================================
# 7. INSPECT JSON ARTIFACTS
# ============================================================

json_read_errors = []


def collect_json_evidence(obj, matches=None):

    if matches is None:
        matches = {}

    if isinstance(obj, dict):

        for key, value in obj.items():

            normalized_key = normalize_name(key)

            for category, fields in NORMALIZED_FIELDS.items():

                for field, aliases in fields.items():

                    if any(
                        alias == normalized_key
                        or alias in normalized_key
                        for alias in aliases
                    ):

                        evidence_key = (
                            category,
                            field
                        )

                        if is_nonempty(value):

                            matches.setdefault(
                                evidence_key,
                                {
                                    "numeric": 0,
                                    "nonempty": 0,
                                    "true": 0,
                                    "false": 0,
                                    "source_fields": []
                                }
                            )

                            matches[evidence_key][
                                "nonempty"
                            ] += 1

                            matches[evidence_key][
                                "source_fields"
                            ].append(str(key))

                            if is_numeric_value(value):

                                matches[evidence_key][
                                    "numeric"
                                ] += 1

                            boolean_value = classify_boolean(
                                value
                            )

                            if boolean_value is True:

                                matches[evidence_key][
                                    "true"
                                ] += 1

                            elif boolean_value is False:

                                matches[evidence_key][
                                    "false"
                                ] += 1

            collect_json_evidence(value, matches)

    elif isinstance(obj, list):

        for item in obj:
            collect_json_evidence(item, matches)

    return matches


for path in source_files:

    if path.suffix.lower() != ".json":
        continue

    try:

        with open(
            path,
            "r",
            encoding="utf-8"
        ) as file:

            data = json.load(file)

        matches = collect_json_evidence(data)

        relative_path = str(
            path.relative_to(PROJECT_ROOT)
        )

        for key, counts in matches.items():

            record = privacy_lookup[key]

            if counts["nonempty"] > 0:

                record["source_artifact_count"] += 1

                record["numeric_measurements_found"] += (
                    counts["numeric"]
                )

                record["nonempty_values_found"] += (
                    counts["nonempty"]
                )

                record["true_values_found"] += (
                    counts["true"]
                )

                record["false_values_found"] += (
                    counts["false"]
                )

                record["source_artifacts"].append(
                    relative_path
                )

                record["matched_source_fields"].extend(
                    counts["source_fields"]
                )

    except Exception as exc:

        json_read_errors.append({
            "artifact": str(
                path.relative_to(PROJECT_ROOT)
            ),
            "error": str(exc)[:250]
        })

# ============================================================
# 8. FINALIZE FIELD-LEVEL EVIDENCE
# ============================================================

for record in privacy_records:

    record["source_artifacts"] = "; ".join(
        sorted(set(record["source_artifacts"]))
    )

    record["matched_source_fields"] = "; ".join(
        sorted(set(record["matched_source_fields"]))
    )

    if record["numeric_measurements_found"] > 0:

        record["evidence_status"] = (
            "NUMERIC_EVIDENCE_FOUND"
        )

    elif record["nonempty_values_found"] > 0:

        record["evidence_status"] = (
            "NONEMPTY_METADATA_FOUND"
        )

    else:

        record["evidence_status"] = "NOT_FOUND"


privacy_field_audit = pd.DataFrame(
    privacy_records
)

# ============================================================
# 9. TARGETED NOTEBOOK 12 PRIVACY REGISTRY AUDIT
# ============================================================

registry_candidates = []

if NOTEBOOK_12_DIR.exists():

    for path in NOTEBOOK_12_DIR.rglob("*"):

        if not path.is_file():
            continue

        if path.suffix.lower() not in {
            ".csv",
            ".json"
        }:
            continue

        name = path.name.lower()

        if any(
            term in name
            for term in [
                "registry",
                "manifest",
                "training_metadata",
                "model_metadata",
                "privacy"
            ]
        ):
            registry_candidates.append(path)

registry_records = []
registry_errors = []

for path in sorted(set(registry_candidates)):

    try:

        if path.suffix.lower() == ".csv":

            df = pd.read_csv(
                path,
                low_memory=True
            )

            columns = df.columns.astype(str).tolist()

            matches = match_privacy_fields(columns)

            if not matches:
                continue

            for (category, field), matched_columns in matches.items():

                for column in matched_columns:

                    nonempty = int(
                        df[column].notna().sum()
                    )

                    true_count = int(
                        df[column].map(
                            classify_boolean
                        ).eq(True).sum()
                    )

                    false_count = int(
                        df[column].map(
                            classify_boolean
                        ).eq(False).sum()
                    )

                    registry_records.append({
                        "artifact": str(
                            path.relative_to(PROJECT_ROOT)
                        ),
                        "privacy_category": category,
                        "privacy_field": field,
                        "source_column": column,
                        "record_count": len(df),
                        "nonempty_values": nonempty,
                        "true_values": true_count,
                        "false_values": false_count
                    })

        elif path.suffix.lower() == ".json":

            with open(
                path,
                "r",
                encoding="utf-8"
            ) as file:

                data = json.load(file)

            matches = collect_json_evidence(data)

            for (category, field), counts in matches.items():

                registry_records.append({
                    "artifact": str(
                        path.relative_to(PROJECT_ROOT)
                    ),
                    "privacy_category": category,
                    "privacy_field": field,
                    "source_column": "; ".join(
                        counts["source_fields"]
                    ),
                    "record_count": np.nan,
                    "nonempty_values": counts["nonempty"],
                    "true_values": counts["true"],
                    "false_values": counts["false"]
                })

    except Exception as exc:

        registry_errors.append({
            "artifact": str(
                path.relative_to(PROJECT_ROOT)
            ),
            "error": str(exc)[:250]
        })

registry_audit = pd.DataFrame(
    registry_records
)

# ============================================================
# 10. EXPLICIT PRIVACY BOUNDARY INTERPRETATION
# ============================================================

def field_has_values(field):

    subset = privacy_field_audit[
        privacy_field_audit["privacy_field"] == field
    ]

    if subset.empty:
        return False

    return bool(
        subset["source_artifact_count"].sum() > 0
    )


def boolean_evidence(field):

    subset = privacy_field_audit[
        privacy_field_audit["privacy_field"] == field
    ]

    if subset.empty:
        return {
            "true": 0,
            "false": 0
        }

    return {
        "true": int(
            subset["true_values_found"].sum()
        ),
        "false": int(
            subset["false_values_found"].sum()
        )
    }


generator_status = boolean_evidence(
    "generator_private"
)

discriminator_status = boolean_evidence(
    "discriminator_private"
)

end_to_end_status = boolean_evidence(
    "end_to_end_claim"
)

privacy_scope_found = field_has_values(
    "privacy_boundary"
)

boundary_interpretation = []

boundary_interpretation.append({
    "component": "Generator",
    "metadata_found": field_has_values(
        "generator_private"
    ),
    "true_records": generator_status["true"],
    "false_records": generator_status["false"],
    "interpretation": (
        "Private configuration recorded"
        if generator_status["true"] > 0
        else "No affirmative private configuration evidence"
    )
})

boundary_interpretation.append({
    "component": "Discriminator/Critic",
    "metadata_found": field_has_values(
        "discriminator_private"
    ),
    "true_records": discriminator_status["true"],
    "false_records": discriminator_status["false"],
    "interpretation": (
        "Private configuration recorded"
        if discriminator_status["true"] > 0
        else "No affirmative private configuration evidence"
    )
})

boundary_interpretation.append({
    "component": "End-to-end",
    "metadata_found": field_has_values(
        "end_to_end_claim"
    ),
    "true_records": end_to_end_status["true"],
    "false_records": end_to_end_status["false"],
    "interpretation": (
        "Affirmative end-to-end claim recorded"
        if end_to_end_status["true"] > 0
        else "No affirmative end-to-end claim evidence"
    )
})

privacy_boundary_audit = pd.DataFrame(
    boundary_interpretation
)

# ============================================================
# 11. CATEGORY-LEVEL COVERAGE
# ============================================================

category_records = []

for category, fields in PRIVACY_FIELDS.items():

    subset = privacy_field_audit[
        privacy_field_audit["privacy_category"] == category
    ]

    expected = len(fields)

    found = int(
        (
            subset["source_artifact_count"] > 0
        ).sum()
    )

    numeric_found = int(
        (
            subset["numeric_measurements_found"] > 0
        ).sum()
    )

    category_records.append({
        "privacy_category": category,
        "fields_expected": expected,
        "fields_found": found,
        "fields_with_numeric_evidence": numeric_found,
        "fields_missing": expected - found,
        "coverage_percentage": round(
            100 * found / expected,
            2
        ) if expected else 0,
        "coverage_status": (
            "COMPLETE"
            if found == expected
            else "PARTIAL"
            if found > 0
            else "NO_EVIDENCE"
        )
    })

privacy_category_audit = pd.DataFrame(
    category_records
)

# ============================================================
# 12. FORMAL DP EVIDENCE CHECK
# ============================================================

def numeric_field_found(field):

    subset = privacy_field_audit[
        privacy_field_audit["privacy_field"] == field
    ]

    return bool(
        not subset.empty
        and subset["numeric_measurements_found"].sum() > 0
    )


epsilon_found = numeric_field_found(
    "achieved_epsilon"
)

delta_found = numeric_field_found(
    "achieved_delta"
)

noise_found = numeric_field_found(
    "noise_multiplier"
)

accountant_found = field_has_values(
    "accountant"
)

rdp_found = field_has_values(
    "rdp"
)

dp_boundary_found = field_has_values(
    "privacy_boundary"
)

if all([
    epsilon_found,
    delta_found,
    noise_found,
    accountant_found,
    dp_boundary_found
]):

    formal_dp_status = (
        "CORE_DP_METADATA_PRESENT_REQUIRES_IMPLEMENTATION_VALIDATION"
    )

elif epsilon_found or delta_found:

    formal_dp_status = (
        "PARTIAL_DP_METADATA_REQUIRES_REVIEW"
    )

else:

    formal_dp_status = (
        "INSUFFICIENT_FORMAL_DP_EVIDENCE"
    )

# ============================================================
# 13. EMPIRICAL PRIVACY EVIDENCE CHECK
# ============================================================

empirical_fields = [
    "reidentification",
    "membership_inference",
    "nearest_neighbor",
    "disclosure",
    "attack"
]

empirical_numeric_count = sum(
    numeric_field_found(field)
    for field in empirical_fields
)

empirical_metadata_count = sum(
    field_has_values(field)
    for field in empirical_fields
)

# ============================================================
# 14. SAVE AUDIT ARTIFACTS
# ============================================================

privacy_field_audit.to_csv(
    TABLES_DIR / "privacy_field_coverage_audit.csv",
    index=False
)

privacy_category_audit.to_csv(
    TABLES_DIR / "privacy_category_coverage_audit.csv",
    index=False
)

privacy_boundary_audit.to_csv(
    TABLES_DIR / "privacy_boundary_audit.csv",
    index=False
)

registry_audit.to_csv(
    TABLES_DIR / "privacy_registry_evidence_audit.csv",
    index=False
)

pd.DataFrame(csv_read_errors).to_csv(
    TABLES_DIR / "privacy_csv_read_errors.csv",
    index=False
)

pd.DataFrame(json_read_errors).to_csv(
    TABLES_DIR / "privacy_json_read_errors.csv",
    index=False
)

pd.DataFrame(registry_errors).to_csv(
    TABLES_DIR / "privacy_registry_read_errors.csv",
    index=False
)

# ============================================================
# 15. FINAL SUMMARY
# ============================================================

total_fields = len(privacy_field_audit)

fields_found = int(
    (
        privacy_field_audit["source_artifact_count"] > 0
    ).sum()
)

numeric_fields_found = int(
    (
        privacy_field_audit["numeric_measurements_found"] > 0
    ).sum()
)

missing_fields = total_fields - fields_found

summary = {
    "source_artifacts_discovered": len(source_files),
    "notebook_12_registry_candidates": len(
        registry_candidates
    ),
    "notebook_12_registry_evidence_records": len(
        registry_audit
    ),
    "expected_privacy_fields": total_fields,
    "privacy_fields_found": fields_found,
    "privacy_fields_with_numeric_evidence": (
        numeric_fields_found
    ),
    "privacy_fields_missing": missing_fields,
    "numeric_measurements_found": int(
        privacy_field_audit[
            "numeric_measurements_found"
        ].sum()
    ),
    "csv_read_errors": len(csv_read_errors),
    "json_read_errors": len(json_read_errors),
    "registry_read_errors": len(registry_errors),
    "formal_dp_evidence_status": formal_dp_status,
    "empirical_privacy_fields_with_numeric_evidence": (
        empirical_numeric_count
    ),
    "empirical_privacy_fields_with_metadata": (
        empirical_metadata_count
    ),
    "privacy_scope_metadata_found": privacy_scope_found,
    "generator_private_true_records": (
        generator_status["true"]
    ),
    "generator_private_false_records": (
        generator_status["false"]
    ),
    "discriminator_private_true_records": (
        discriminator_status["true"]
    ),
    "discriminator_private_false_records": (
        discriminator_status["false"]
    ),
    "end_to_end_true_records": (
        end_to_end_status["true"]
    ),
    "end_to_end_false_records": (
        end_to_end_status["false"]
    )
}

# ------------------------------------------------------------
# STATUS RULES
# ------------------------------------------------------------

if (
    csv_read_errors
    or json_read_errors
    or registry_errors
):

    section_status = "REVIEW_REQUIRED"

elif missing_fields > 0:

    section_status = "REVIEW_REQUIRED"

else:

    section_status = "PASS"

summary["section_status"] = section_status

with open(
    TABLES_DIR / "privacy_coverage_audit_summary.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        summary,
        file,
        indent=4
    )

# ============================================================
# 16. DISPLAY RESULTS
# ============================================================

print("\nPRIVACY FIELD-LEVEL COVERAGE")
print("=" * 110)

print(
    privacy_field_audit[
        [
            "privacy_category",
            "privacy_field",
            "source_artifact_count",
            "numeric_measurements_found",
            "nonempty_values_found",
            "true_values_found",
            "false_values_found",
            "evidence_status"
        ]
    ].to_string(index=False)
)

print("\nPRIVACY CATEGORY SUMMARY")
print("=" * 110)

print(
    privacy_category_audit.to_string(index=False)
)

print("\nNOTEBOOK 12 PRIVACY REGISTRY EVIDENCE")
print("=" * 110)

if not registry_audit.empty:
    print(registry_audit.to_string(index=False))
else:
    print("No matching privacy fields found in registry candidates.")

print("\nPRIVACY IMPLEMENTATION BOUNDARY")
print("=" * 110)

print(
    privacy_boundary_audit.to_string(index=False)
)

print("\nFORMAL DP EVIDENCE STATUS")
print("=" * 110)

print(formal_dp_status)

print("\nPRIVACY AUDIT COMPLETION SUMMARY")
print("=" * 110)

for key, value in summary.items():
    print(f"{key}: {value}")

print("\nSECTION 19 STATUS:", section_status)

print("\nSAVED AUDIT FILES")
print("=" * 110)

for filename in [
    "privacy_field_coverage_audit.csv",
    "privacy_category_coverage_audit.csv",
    "privacy_boundary_audit.csv",
    "privacy_registry_evidence_audit.csv",
    "privacy_coverage_audit_summary.json",
    "privacy_csv_read_errors.csv",
    "privacy_json_read_errors.csv",
    "privacy_registry_read_errors.csv"
]:

    print(TABLES_DIR / filename)

print("=" * 110)
print("SECTION 19 EXECUTION COMPLETED")
print("=" * 110)


Privacy source artifacts discovered: 984

PRIVACY FIELD-LEVEL COVERAGE
 privacy_category         privacy_field  source_artifact_count  numeric_measurements_found  nonempty_values_found  true_values_found  false_values_found         evidence_status
        formal_dp      achieved_epsilon                    129                       11605                  11681                 28                 463  NUMERIC_EVIDENCE_FOUND
        formal_dp        achieved_delta                     92                        6363                   6387                  8                 147  NUMERIC_EVIDENCE_FOUND
        formal_dp      noise_multiplier                     84                        5974                   5980                 54                   0  NUMERIC_EVIDENCE_FOUND
        formal_dp            accountant                     67                          79                    221                 15                  73  NUMERIC_EVIDENCE_FOUND
        formal_dp      privacy_boundary     

In [26]:

# ============================================================
# SECTION 20 — CONFIGURATION CONSISTENCY AUDIT
# ============================================================

import json
import math
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 1. PROJECT PATHS AND EXPECTED CONFIGURATION
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/SPP_GAN_Research"
)

CONFIG_DIR = (
    PROJECT_ROOT
    / "results/notebooks/notebook_00/config"
)

MANIFEST_PATH = (
    PROJECT_ROOT
    / "results/notebooks/notebook_00/manifest/"
    / "notebook_00_manifest.json"
)

TABLES_DIR = (
    PROJECT_ROOT
    / "results/notebooks/notebook_22/tables"
)

TABLES_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_MASTER_SEED = 2025
EXPECTED_REPETITIONS = 5

EXPECTED_DATASETS = sorted([
    "adult_income",
    "bank_marketing",
    "diabetes_130us"
])

EXPECTED_SPLIT = {
    "train": 0.70,
    "validation": 0.15,
    "test": 0.15
}

# ------------------------------------------------------------
# 2. LOAD AUTHORITATIVE EXPERIMENT CONFIGURATION
# ------------------------------------------------------------

EXPERIMENT_CONFIG_PATH = (
    CONFIG_DIR / "experiment_config.json"
)

if not EXPERIMENT_CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Missing experiment configuration: "
        f"{EXPERIMENT_CONFIG_PATH}"
    )

with open(
    EXPERIMENT_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as file:
    experiment_config = json.load(file)

if not isinstance(experiment_config, dict):
    raise ValueError(
        "Experiment configuration must be a JSON object."
    )

# ------------------------------------------------------------
# 3. LOAD MASTER CONFIGURATION EVIDENCE
# ------------------------------------------------------------

# Use the already validated Notebook 00 configuration
# files where available. The experiment configuration
# remains authoritative for split ratios.

def load_json_if_exists(path):

    if not path.exists():
        return None

    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)


config_sources = {}

for filename in [
    "master_config.json",
    "project_config.json",
    "dataset_config.json",
    "experiment_config.json"
]:

    path = CONFIG_DIR / filename
    data = load_json_if_exists(path)

    if isinstance(data, dict):
        config_sources[filename] = data

# Include other previously validated Notebook 00 JSON
# configuration files, without replacing the primary source.
for path in sorted(CONFIG_DIR.glob("*.json")):

    if path.name in config_sources:
        continue

    data = load_json_if_exists(path)

    if isinstance(data, dict):
        config_sources[path.name] = data

# ------------------------------------------------------------
# 4. LOAD NOTEBOOK 00 MANIFEST
# ------------------------------------------------------------

manifest = load_json_if_exists(MANIFEST_PATH)

manifest_experiment = {}

if isinstance(manifest, dict):

    configuration = manifest.get(
        "configuration",
        {}
    )

    if isinstance(configuration, dict):

        manifest_experiment = (
            configuration.get("experiment", {})
        )

# ------------------------------------------------------------
# 5. HELPER FUNCTIONS
# ------------------------------------------------------------

def find_value_recursive(obj, aliases):

    aliases = {
        str(alias).lower().replace("-", "_")
        for alias in aliases
    }

    found = []

    def walk(value, parent=""):

        if isinstance(value, dict):

            for key, child in value.items():

                normalized_key = (
                    str(key).lower().replace("-", "_")
                )

                current = (
                    f"{parent}.{key}"
                    if parent else str(key)
                )

                if normalized_key in aliases:
                    found.append({
                        "path": current,
                        "value": child
                    })

                walk(child, current)

        elif isinstance(value, list):

            for index, child in enumerate(value):
                walk(child, f"{parent}[{index}]")

    walk(obj)

    return found


def extract_split(data):

    if not isinstance(data, dict):
        return None

    # Accept either the complete data_split object
    # or separately recorded fraction fields.
    split_object = data.get("data_split")

    if not isinstance(split_object, dict):
        split_object = data

    aliases = {
        "train": [
            "train_fraction",
            "train_ratio",
            "train_size"
        ],
        "validation": [
            "validation_fraction",
            "validation_ratio",
            "validation_size"
        ],
        "test": [
            "test_fraction",
            "test_ratio",
            "test_size"
        ]
    }

    result = {}

    for split_name, keys in aliases.items():

        value = None

        for key in keys:

            if key in split_object:
                value = split_object[key]
                break

        if value is None:
            return None

        if isinstance(value, bool):
            return None

        try:
            result[split_name] = float(value)
        except (TypeError, ValueError):
            return None

    return result


def split_matches(observed, expected):

    if not isinstance(observed, dict):
        return False

    return all(
        math.isclose(
            observed[key],
            expected[key],
            rel_tol=0,
            abs_tol=1e-8
        )
        for key in expected
    )


# ------------------------------------------------------------
# 6. EXTRACT AUTHORITATIVE SPLIT CONFIGURATION
# ------------------------------------------------------------

observed_split = extract_split(experiment_config)

if observed_split is None:
    split_status = "NOT_FOUND"
else:

    split_sum = sum(observed_split.values())

    if (
        split_matches(observed_split, EXPECTED_SPLIT)
        and math.isclose(
            split_sum,
            1.0,
            rel_tol=0,
            abs_tol=1e-8
        )
    ):
        split_status = "PASS"
    else:
        split_status = "MISMATCH"

# Verify split is enabled and training-only fitting is configured.
data_split_config = experiment_config.get(
    "data_split",
    {}
)

split_enabled = (
    data_split_config.get("enabled") is True
)

fit_split = data_split_config.get("fit_split")

# ------------------------------------------------------------
# 7. VERIFY MASTER SEED, REPETITIONS AND DATASETS
# ------------------------------------------------------------

def find_first_valid_value(data, aliases):

    results = find_value_recursive(data, aliases)

    for item in results:

        value = item["value"]

        if isinstance(value, bool):
            continue

        if value is not None:
            return value

    return None


master_seed_observed = find_first_valid_value(
    experiment_config,
    ["master_seed"]
)

repetition_observed = find_first_valid_value(
    experiment_config,
    [
        "n_repetitions",
        "num_repetitions",
        "repetition_count",
        "repetitions"
    ]
)

dataset_observed = find_first_valid_value(
    experiment_config,
    ["dataset_registry", "datasets"]
)

# Search other Notebook 00 configs if not found in
# experiment_config.json.
for data in config_sources.values():

    if master_seed_observed is None:
        master_seed_observed = find_first_valid_value(
            data,
            ["master_seed"]
        )

    if repetition_observed is None:
        repetition_observed = find_first_valid_value(
            data,
            [
                "n_repetitions",
                "num_repetitions",
                "repetition_count",
                "repetitions"
            ]
        )

    if dataset_observed is None:
        dataset_observed = find_first_valid_value(
            data,
            ["dataset_registry", "datasets"]
        )

def normalize_datasets(value):

    if isinstance(value, dict):
        return sorted(
            str(key).strip().lower()
            for key in value.keys()
        )

    if isinstance(value, list):

        names = []

        for item in value:

            if isinstance(item, str):
                names.append(item.strip().lower())

            elif isinstance(item, dict):

                name = (
                    item.get("dataset_id")
                    or item.get("dataset")
                    or item.get("name")
                )

                if name:
                    names.append(str(name).strip().lower())

        return sorted(set(names))

    return None


try:
    master_seed_observed = int(master_seed_observed)
except (TypeError, ValueError):
    master_seed_observed = None

try:
    repetition_observed = int(repetition_observed)
except (TypeError, ValueError):
    repetition_observed = None

dataset_observed = normalize_datasets(
    dataset_observed
)

# ------------------------------------------------------------
# 8. MANIFEST SPLIT CORROBORATION
# ------------------------------------------------------------

manifest_split = extract_split(
    manifest_experiment
)

if manifest_split is None:
    manifest_split_status = "NOT_AVAILABLE"
else:
    manifest_split_status = (
        "MATCH"
        if split_matches(manifest_split, EXPECTED_SPLIT)
        else "MISMATCH"
    )

# ------------------------------------------------------------
# 9. BUILD CONFIGURATION AUDIT
# ------------------------------------------------------------

checks = [
    {
        "parameter": "master_seed",
        "expected": EXPECTED_MASTER_SEED,
        "observed": master_seed_observed,
        "check": (
            "PASS"
            if master_seed_observed == EXPECTED_MASTER_SEED
            else "REVIEW_REQUIRED"
        ),
        "source": "Notebook 00 configuration"
    },
    {
        "parameter": "repetition_count",
        "expected": EXPECTED_REPETITIONS,
        "observed": repetition_observed,
        "check": (
            "PASS"
            if repetition_observed == EXPECTED_REPETITIONS
            else "REVIEW_REQUIRED"
        ),
        "source": "Notebook 00 configuration"
    },
    {
        "parameter": "datasets",
        "expected": EXPECTED_DATASETS,
        "observed": dataset_observed,
        "check": (
            "PASS"
            if dataset_observed == EXPECTED_DATASETS
            else "REVIEW_REQUIRED"
        ),
        "source": "Notebook 00 configuration"
    },
    {
        "parameter": "train_validation_test",
        "expected": EXPECTED_SPLIT,
        "observed": observed_split,
        "check": split_status,
        "source": "experiment_config.json:data_split"
    },
    {
        "parameter": "split_enabled",
        "expected": True,
        "observed": split_enabled,
        "check": (
            "PASS"
            if split_enabled
            else "REVIEW_REQUIRED"
        ),
        "source": "experiment_config.json:data_split.enabled"
    },
    {
        "parameter": "fit_split",
        "expected": "train",
        "observed": fit_split,
        "check": (
            "PASS"
            if fit_split == "train"
            else "REVIEW_REQUIRED"
        ),
        "source": "experiment_config.json:data_split.fit_split"
    }
]

configuration_consistency_audit = pd.DataFrame(checks)

# ------------------------------------------------------------
# 10. FINAL STATUS
# ------------------------------------------------------------

passed = int(
    (configuration_consistency_audit["check"] == "PASS").sum()
)

review = int(
    (
        configuration_consistency_audit["check"]
        != "PASS"
    ).sum()
)

if review == 0:
    section_status = "PASS"
else:
    section_status = "REVIEW_REQUIRED"

summary = {
    "configuration_files_loaded": len(config_sources),
    "master_seed": master_seed_observed,
    "repetition_count": repetition_observed,
    "datasets": dataset_observed,
    "observed_split": observed_split,
    "split_sum": (
        sum(observed_split.values())
        if observed_split else None
    ),
    "split_enabled": split_enabled,
    "fit_split": fit_split,
    "manifest_split_status": manifest_split_status,
    "checks_expected": len(checks),
    "checks_passed": passed,
    "checks_review_required": review,
    "section_status": section_status
}

# ------------------------------------------------------------
# 11. SAVE AUDIT OUTPUTS
# ------------------------------------------------------------

configuration_consistency_audit.to_csv(
    TABLES_DIR / "configuration_consistency_audit.csv",
    index=False
)

pd.DataFrame([
    {
        "source": "experiment_config.json",
        "train": (
            observed_split.get("train")
            if observed_split else None
        ),
        "validation": (
            observed_split.get("validation")
            if observed_split else None
        ),
        "test": (
            observed_split.get("test")
            if observed_split else None
        ),
        "status": split_status
    },
    {
        "source": "notebook_00_manifest.json",
        "train": (
            manifest_split.get("train")
            if manifest_split else None
        ),
        "validation": (
            manifest_split.get("validation")
            if manifest_split else None
        ),
        "test": (
            manifest_split.get("test")
            if manifest_split else None
        ),
        "status": manifest_split_status
    }
]).to_csv(
    TABLES_DIR / "configuration_split_evidence.csv",
    index=False
)

with open(
    TABLES_DIR / "configuration_consistency_audit_summary.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(summary, file, indent=4)

# ------------------------------------------------------------
# 12. DISPLAY RESULTS
# ------------------------------------------------------------

print("=" * 90)
print("SECTION 20 — CONFIGURATION CONSISTENCY AUDIT")
print("=" * 90)

print(
    configuration_consistency_audit.to_string(
        index=False
    )
)

print("\nSPLIT CONFIGURATION EVIDENCE")
print("-" * 90)
print("Primary source:", EXPERIMENT_CONFIG_PATH)
print("Observed split:", observed_split)
print("Split total:", summary["split_sum"])
print("Manifest corroboration:", manifest_split_status)

print("\nCOMPLETION SUMMARY")
print("-" * 90)

for key, value in summary.items():
    print(f"{key}: {value}")

print("\nSECTION 20 STATUS:", section_status)

print("\nSaved outputs:")
print(TABLES_DIR / "configuration_consistency_audit.csv")
print(TABLES_DIR / "configuration_split_evidence.csv")
print(TABLES_DIR / "configuration_consistency_audit_summary.json")

print("=" * 90)
print("SECTION 20 EXECUTION COMPLETED")
print("=" * 90)


SECTION 20 — CONFIGURATION CONSISTENCY AUDIT
            parameter                                         expected                                         observed check                                      source
          master_seed                                             2025                                             2025  PASS                   Notebook 00 configuration
     repetition_count                                                5                                                5  PASS                   Notebook 00 configuration
             datasets   [adult_income, bank_marketing, diabetes_130us]   [adult_income, bank_marketing, diabetes_130us]  PASS                   Notebook 00 configuration
train_validation_test {'train': 0.7, 'validation': 0.15, 'test': 0.15} {'train': 0.7, 'validation': 0.15, 'test': 0.15}  PASS           experiment_config.json:data_split
        split_enabled                                             True                                   

In [30]:

# ============================================================
# SECTION 21 — SEED / REPRODUCIBILITY AUDIT
# ============================================================

import json
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 1. PROJECT CONFIGURATION
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/SPP_GAN_Research"
)

TABLES_DIR = (
    PROJECT_ROOT
    / "results/notebooks/notebook_22/tables"
)

TABLES_DIR.mkdir(parents=True, exist_ok=True)

MASTER_SEED = 2025
EXPECTED_REPETITIONS = 5

EXPECTED_DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us"
]

EXPECTED_REPETITION_SEEDS = {
    rep: MASTER_SEED + 1000 + rep
    for rep in range(1, EXPECTED_REPETITIONS + 1)
}

# ------------------------------------------------------------
# 2. AUTHORITATIVE SOURCE PATHS
# ------------------------------------------------------------

SOURCE_PATHS = {
    "training_results": (
        PROJECT_ROOT
        / "results/notebooks/notebook_12/metadata/"
        / "notebook_12B_training_results.csv"
    ),
    "training_metadata": (
        PROJECT_ROOT
        / "results/notebooks/notebook_12/registries/"
        / "notebook_12B_training_metadata.csv"
    ),
    "training_metadata_json": (
        PROJECT_ROOT
        / "results/notebooks/notebook_12/registries/"
        / "notebook_12B_training_metadata.json"
    ),
    "final_manifest": (
        PROJECT_ROOT
        / "results/notebooks/notebook_12/metadata/"
        / "notebook_12B_final_manifest.json"
    ),
    "generation_seed_plan": (
        PROJECT_ROOT
        / "results/notebooks/notebook_13/metadata/"
        / "generation_seed_plan_normalized.csv"
    ),
    "model_run_log": (
        PROJECT_ROOT
        / "results/notebooks/notebook_15/logs/"
        / "model_run_log.csv"
    ),
    "generation_metadata": (
        PROJECT_ROOT
        / "results/notebooks/notebook_13/logs/"
        / "generation_metadata.csv"
    ),
    "empty_validation_registry": (
        PROJECT_ROOT
        / "results/notebooks/notebook_12/validation/"
        / "notebook_12B_training_registry.csv"
    )
}

# ------------------------------------------------------------
# 3. SOURCE LOADING
# ------------------------------------------------------------

source_errors = []
source_inventory = []

def load_csv_source(name, path):

    if not path.exists():

        source_inventory.append({
            "source": name,
            "path": str(path),
            "status": "MISSING",
            "rows": 0,
            "columns": 0
        })

        return None

    try:

        if path.stat().st_size == 0:

            source_inventory.append({
                "source": name,
                "path": str(path),
                "status": "EMPTY",
                "rows": 0,
                "columns": 0
            })

            return None

        df = pd.read_csv(path, low_memory=False)

        source_inventory.append({
            "source": name,
            "path": str(path),
            "status": "READABLE",
            "rows": len(df),
            "columns": len(df.columns)
        })

        return df

    except Exception as exc:

        source_errors.append({
            "source": name,
            "path": str(path),
            "error": str(exc)[:300]
        })

        source_inventory.append({
            "source": name,
            "path": str(path),
            "status": "READ_ERROR",
            "rows": 0,
            "columns": 0
        })

        return None


def load_json_source(name, path):

    if not path.exists():

        source_inventory.append({
            "source": name,
            "path": str(path),
            "status": "MISSING",
            "rows": 0,
            "columns": 0
        })

        return None

    try:

        if path.stat().st_size == 0:

            source_inventory.append({
                "source": name,
                "path": str(path),
                "status": "EMPTY",
                "rows": 0,
                "columns": 0
            })

            return None

        with open(path, "r", encoding="utf-8") as file:
            data = json.load(file)

        source_inventory.append({
            "source": name,
            "path": str(path),
            "status": "READABLE",
            "rows": (
                len(data)
                if isinstance(data, list)
                else 1
            ),
            "columns": (
                len(data)
                if isinstance(data, dict)
                else 0
            )
        })

        return data

    except Exception as exc:

        source_errors.append({
            "source": name,
            "path": str(path),
            "error": str(exc)[:300]
        })

        source_inventory.append({
            "source": name,
            "path": str(path),
            "status": "READ_ERROR",
            "rows": 0,
            "columns": 0
        })

        return None


training_results = load_csv_source(
    "training_results",
    SOURCE_PATHS["training_results"]
)

training_metadata = load_csv_source(
    "training_metadata",
    SOURCE_PATHS["training_metadata"]
)

training_metadata_json = load_json_source(
    "training_metadata_json",
    SOURCE_PATHS["training_metadata_json"]
)

final_manifest = load_json_source(
    "final_manifest",
    SOURCE_PATHS["final_manifest"]
)

generation_plan = load_csv_source(
    "generation_seed_plan",
    SOURCE_PATHS["generation_seed_plan"]
)

model_run_log = load_csv_source(
    "model_run_log",
    SOURCE_PATHS["model_run_log"]
)

generation_metadata = load_csv_source(
    "generation_metadata",
    SOURCE_PATHS["generation_metadata"]
)

# ------------------------------------------------------------
# 4. CONFIGURATION SEED VERIFICATION
# ------------------------------------------------------------

environment_path = (
    PROJECT_ROOT
    / "results/notebooks/notebook_00/environment/"
    / "environment.json"
)

experiment_path = (
    PROJECT_ROOT
    / "results/notebooks/notebook_00/config/"
    / "experiment_config.json"
)

environment = load_json_source(
    "notebook_00_environment",
    environment_path
)

experiment = load_json_source(
    "notebook_00_experiment_config",
    experiment_path
)

environment = (
    environment
    if isinstance(environment, dict)
    else {}
)

experiment = (
    experiment
    if isinstance(experiment, dict)
    else {}
)

randomness = environment.get("randomness", {})
seed_registry = randomness.get("seed_registry", {})
reproducibility = randomness.get(
    "reproducibility_config", {}
)

configured_seeds = [
    {
        "seed_type": "master_seed",
        "expected": 2025,
        "observed": (
            reproducibility.get("master_seed")
            or seed_registry.get("master")
            or experiment.get("master_seed")
        )
    },
    {
        "seed_type": "data_split_seed",
        "expected": 2035,
        "observed": (
            seed_registry.get("data_split")
            or reproducibility.get("data_split_seed")
        )
    },
    {
        "seed_type": "generation_seed",
        "expected": 2065,
        "observed": (
            seed_registry.get("generation")
            or reproducibility.get("generation_seed")
        )
    }
]

for item in configured_seeds:

    try:
        item["observed"] = int(item["observed"])
    except (TypeError, ValueError):
        item["observed"] = None

    item["status"] = (
        "PASS"
        if item["observed"] == item["expected"]
        else "REVIEW_REQUIRED"
    )

configuration_seed_audit = pd.DataFrame(
    configured_seeds
)

# ------------------------------------------------------------
# 5. COLUMN NORMALIZATION
# ------------------------------------------------------------

def normalize_columns(df):

    if df is None:
        return None

    result = df.copy()

    result.columns = [
        str(column).strip().lower().replace("-", "_")
        for column in result.columns
    ]

    return result


training_results = normalize_columns(training_results)
training_metadata = normalize_columns(training_metadata)
generation_plan = normalize_columns(generation_plan)
model_run_log = normalize_columns(model_run_log)
generation_metadata = normalize_columns(generation_metadata)


def find_column(df, candidates):

    if df is None:
        return None

    for candidate in candidates:

        if candidate in df.columns:
            return candidate

    return None


def normalize_dataset(value):

    if pd.isna(value):
        return None

    value = str(value).strip().lower()

    aliases = {
        "adult": "adult_income",
        "adult_income": "adult_income",
        "bank": "bank_marketing",
        "bank_marketing": "bank_marketing",
        "diabetes": "diabetes_130us",
        "diabetes_130us": "diabetes_130us"
    }

    return aliases.get(value, value)


def normalize_rep(value):

    if pd.isna(value):
        return None

    try:

        number = int(float(value))

        if 1 <= number <= EXPECTED_REPETITIONS:
            return number

    except (TypeError, ValueError):
        pass

    text = str(value).strip().lower()

    import re

    match = re.search(
        r"(?:repetition|rep)[_-]?0*([1-5])",
        text
    )

    return int(match.group(1)) if match else None


def normalize_seed(value):

    if pd.isna(value):
        return None

    try:

        number = float(value)

        if number.is_integer():
            return int(number)

    except (TypeError, ValueError):
        pass

    return None

# ------------------------------------------------------------
# 6. EXTRACT AUTHORITATIVE TRAINING SEED RECORDS
# ------------------------------------------------------------

def extract_seed_records(df, source_name, seed_candidates):

    records = []

    if df is None:
        return records

    dataset_col = find_column(
        df,
        ["dataset", "dataset_id", "dataset_name"]
    )

    repetition_col = find_column(
        df,
        ["repetition", "repetition_id", "rep"]
    )

    seed_col = find_column(
        df,
        seed_candidates
    )

    if seed_col is None:
        print(
            f"WARNING: No recognized seed column in {source_name}"
        )
        return records

    for _, row in df.iterrows():

        dataset = (
            normalize_dataset(row[dataset_col])
            if dataset_col else None
        )

        repetition = (
            normalize_rep(row[repetition_col])
            if repetition_col else None
        )

        seed = normalize_seed(row[seed_col])

        if (
            dataset not in EXPECTED_DATASETS
            or repetition not in EXPECTED_REPETITION_SEEDS
            or seed is None
        ):
            continue

        records.append({
            "source": source_name,
            "dataset": dataset,
            "repetition": repetition,
            "observed_seed": seed,
            "expected_seed": (
                EXPECTED_REPETITION_SEEDS[repetition]
            ),
            "matches": (
                seed == EXPECTED_REPETITION_SEEDS[repetition]
            )
        })

    return records


training_seed_records = []

# Explicitly use experiment_seed as the preferred training seed.
training_seed_records.extend(
    extract_seed_records(
        training_results,
        "notebook_12B_training_results",
        [
            "experiment_seed",
            "repetition_seed",
            "training_seed"
        ]
    )
)

training_seed_records.extend(
    extract_seed_records(
        training_metadata,
        "notebook_12B_training_metadata",
        [
            "experiment_seed",
            "repetition_seed",
            "training_seed"
        ]
    )
)

training_seed_records.extend(
    extract_seed_records(
        model_run_log,
        "notebook_15_model_run_log",
        [
            "experiment_seed",
            "repetition_seed",
            "training_seed"
        ]
    )
)

training_seed_evidence = pd.DataFrame(
    training_seed_records,
    columns=[
        "source",
        "dataset",
        "repetition",
        "observed_seed",
        "expected_seed",
        "matches"
    ]
)

# ------------------------------------------------------------
# 7. BUILD DATASET × REPETITION TRAINING AUDIT
# ------------------------------------------------------------

training_coverage_records = []

for dataset in EXPECTED_DATASETS:

    for repetition, expected_seed in (
        EXPECTED_REPETITION_SEEDS.items()
    ):

        subset = training_seed_evidence[
            (
                training_seed_evidence["dataset"] == dataset
            )
            &
            (
                training_seed_evidence["repetition"] == repetition
            )
        ]

        observed = sorted(
            subset["observed_seed"].dropna().unique().tolist()
        )

        correct_sources = sorted(
            subset.loc[
                subset["matches"] == True,
                "source"
            ].unique().tolist()
        )

        incorrect_sources = sorted(
            subset.loc[
                subset["matches"] == False,
                "source"
            ].unique().tolist()
        )

        if expected_seed in observed and not incorrect_sources:
            status = "PASS"

        elif incorrect_sources:
            status = "REVIEW_REQUIRED"

        else:
            status = "NOT_VERIFIED"

        training_coverage_records.append({
            "dataset": dataset,
            "repetition": repetition,
            "expected_seed": expected_seed,
            "observed_seeds": json.dumps(observed),
            "evidence_records": len(subset),
            "matching_sources": json.dumps(correct_sources),
            "mismatching_sources": json.dumps(incorrect_sources),
            "status": status
        })

training_coverage = pd.DataFrame(
    training_coverage_records
)

# ------------------------------------------------------------
# 8. GENERATION SEED PLAN AUDIT
# ------------------------------------------------------------

generation_seed_records = extract_seed_records(
    generation_plan,
    "notebook_13_generation_seed_plan",
    [
        "experiment_seed",
        "repetition_seed",
        "generation_seed",
        "seed"
    ]
)

generation_seed_evidence = pd.DataFrame(
    generation_seed_records,
    columns=[
        "source",
        "dataset",
        "repetition",
        "observed_seed",
        "expected_seed",
        "matches"
    ]
)

generation_coverage_records = []

for dataset in EXPECTED_DATASETS:

    for repetition, expected_seed in (
        EXPECTED_REPETITION_SEEDS.items()
    ):

        subset = generation_seed_evidence[
            (
                generation_seed_evidence["dataset"] == dataset
            )
            &
            (
                generation_seed_evidence["repetition"] == repetition
            )
        ]

        observed = sorted(
            subset["observed_seed"].dropna().unique().tolist()
        )

        generation_coverage_records.append({
            "dataset": dataset,
            "repetition": repetition,
            "expected_seed": expected_seed,
            "observed_seeds": json.dumps(observed),
            "evidence_records": len(subset),
            "status": (
                "PASS"
                if expected_seed in observed
                and len(observed) == 1
                else "REVIEW_REQUIRED"
            )
        })

generation_coverage = pd.DataFrame(
    generation_coverage_records
)

# ------------------------------------------------------------
# 9. GENERATION METADATA — SEPARATE SEED FIELDS
# ------------------------------------------------------------

generation_metadata_summary = []

if generation_metadata is not None:

    for column in generation_metadata.columns:

        if "seed" not in column.lower():
            continue

        values = sorted(
            generation_metadata[column]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

        generation_metadata_summary.append({
            "field": column,
            "distinct_values": json.dumps(values),
            "record_count": int(
                generation_metadata[column].notna().sum()
            )
        })

generation_metadata_seed_audit = pd.DataFrame(
    generation_metadata_summary,
    columns=[
        "field",
        "distinct_values",
        "record_count"
    ]
)

# ------------------------------------------------------------
# 10. AUXILIARY EMPTY REGISTRY CHECK
# ------------------------------------------------------------

empty_registry_path = SOURCE_PATHS[
    "empty_validation_registry"
]

if empty_registry_path.exists():

    empty_registry_size = empty_registry_path.stat().st_size

    if empty_registry_size == 0:
        empty_registry_status = "EMPTY_AUXILIARY_ARTIFACT"
    else:
        empty_registry_status = "NONEMPTY"

else:
    empty_registry_status = "NOT_FOUND"

# ------------------------------------------------------------
# 11. FINAL COVERAGE AND STATUS
# ------------------------------------------------------------

training_verified = int(
    (training_coverage["status"] == "PASS").sum()
)

generation_verified = int(
    (generation_coverage["status"] == "PASS").sum()
)

expected_combinations = (
    len(EXPECTED_DATASETS) * EXPECTED_REPETITIONS
)

configuration_verified = bool(
    (configuration_seed_audit["status"] == "PASS").all()
)

training_complete = (
    training_verified == expected_combinations
)

generation_plan_complete = (
    generation_verified == expected_combinations
)

if (
    configuration_verified
    and training_complete
    and generation_plan_complete
):
    section_status = "PASS"
else:
    section_status = "REVIEW_REQUIRED"

summary = {
    "master_seed": MASTER_SEED,
    "configured_seed_checks_passed": int(
        (configuration_seed_audit["status"] == "PASS").sum()
    ),
    "configured_seed_checks_expected": len(configuration_seed_audit),
    "expected_repetition_seeds": list(
        EXPECTED_REPETITION_SEEDS.values()
    ),
    "training_combinations_expected": expected_combinations,
    "training_combinations_verified": training_verified,
    "generation_plan_combinations_expected": expected_combinations,
    "generation_plan_combinations_verified": generation_verified,
    "generation_metadata_seed_fields": len(
        generation_metadata_seed_audit
    ),
    "empty_auxiliary_registry_status": empty_registry_status,
    "source_read_errors": len(source_errors),
    "section_status": section_status
}

# ------------------------------------------------------------
# 12. SAVE AUDIT ARTIFACTS
# ------------------------------------------------------------

configuration_seed_audit.to_csv(
    TABLES_DIR / "configuration_seed_audit.csv",
    index=False
)

training_seed_evidence.to_csv(
    TABLES_DIR / "training_seed_evidence.csv",
    index=False
)

training_coverage.to_csv(
    TABLES_DIR / "training_repetition_seed_coverage.csv",
    index=False
)

generation_seed_evidence.to_csv(
    TABLES_DIR / "generation_seed_evidence.csv",
    index=False
)

generation_coverage.to_csv(
    TABLES_DIR / "generation_repetition_seed_coverage.csv",
    index=False
)

generation_metadata_seed_audit.to_csv(
    TABLES_DIR / "generation_metadata_seed_audit.csv",
    index=False
)

pd.DataFrame(source_inventory).to_csv(
    TABLES_DIR / "seed_source_inventory.csv",
    index=False
)

pd.DataFrame(
    source_errors,
    columns=["source", "path", "error"]
).to_csv(
    TABLES_DIR / "seed_source_read_errors.csv",
    index=False
)

with open(
    TABLES_DIR / "seed_reproducibility_audit_summary.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(summary, file, indent=4)

# ------------------------------------------------------------
# 13. DISPLAY RESULTS
# ------------------------------------------------------------

print("=" * 90)
print("SECTION 21 — SEED / REPRODUCIBILITY AUDIT")
print("=" * 90)

print("\nCONFIGURATION SEED AUDIT")
print(configuration_seed_audit.to_string(index=False))

print("\nTRAINING REPETITION SEED COVERAGE")
print(training_coverage.to_string(index=False))

print("\nGENERATION PLAN SEED COVERAGE")
print(generation_coverage.to_string(index=False))

print("\nGENERATION METADATA SEED FIELDS")
print(generation_metadata_seed_audit.to_string(index=False))

print("\nSOURCE INVENTORY")
print(pd.DataFrame(source_inventory).to_string(index=False))

print("\nANCILLARY REGISTRY STATUS:")
print(empty_registry_status)

print("\nCOMPLETION SUMMARY")
for key, value in summary.items():
    print(f"{key}: {value}")

print("\nSECTION 21 STATUS:", section_status)

print("\nSaved audit files:")
for filename in [
    "configuration_seed_audit.csv",
    "training_seed_evidence.csv",
    "training_repetition_seed_coverage.csv",
    "generation_seed_evidence.csv",
    "generation_repetition_seed_coverage.csv",
    "generation_metadata_seed_audit.csv",
    "seed_source_inventory.csv",
    "seed_source_read_errors.csv",
    "seed_reproducibility_audit_summary.json"
]:
    print(TABLES_DIR / filename)

print("=" * 90)
print("SECTION 21 EXECUTION COMPLETED")
print("=" * 90)


SECTION 21 — SEED / REPRODUCIBILITY AUDIT

CONFIGURATION SEED AUDIT
      seed_type  expected  observed status
    master_seed      2025      2025   PASS
data_split_seed      2035      2035   PASS
generation_seed      2065      2065   PASS

TRAINING REPETITION SEED COVERAGE
       dataset  repetition  expected_seed observed_seeds  evidence_records                                                    matching_sources mismatching_sources status
  adult_income           1           3026         [3026]                 2 ["notebook_12B_training_metadata", "notebook_12B_training_results"]                  []   PASS
  adult_income           2           3027         [3027]                 2 ["notebook_12B_training_metadata", "notebook_12B_training_results"]                  []   PASS
  adult_income           3           3028         [3028]                 2 ["notebook_12B_training_metadata", "notebook_12B_training_results"]                  []   PASS
  adult_income           4           3029    

In [33]:

# ============================================================
# SECTION 22 — FILE INTEGRITY AUDIT
# ============================================================

import hashlib
import json
import pandas as pd

from pathlib import Path
from datetime import datetime, timezone

# ------------------------------------------------------------
# 1. VALIDATE PROJECT DIRECTORIES
# ------------------------------------------------------------

PROJECT_ROOT = Path(PROJECT_ROOT)
TABLES_DIR = Path(TABLES_DIR)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project root not found: {PROJECT_ROOT}"
    )

TABLES_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. SHA-256 HASH FUNCTION
# ------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):

    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for chunk in iter(
            lambda: file.read(chunk_size),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


# ------------------------------------------------------------
# 3. DISCOVER PROJECT FILES
# ------------------------------------------------------------

# Exclude the current audit outputs to avoid self-inclusion.
excluded_paths = {
    (TABLES_DIR / "file_integrity_audit.csv").resolve(),
    (TABLES_DIR / "file_integrity_audit_summary.json").resolve()
}

project_files = [
    path
    for path in PROJECT_ROOT.rglob("*")
    if path.is_file()
    and path.resolve() not in excluded_paths
]

integrity_records = []

# ------------------------------------------------------------
# 4. INVENTORY AND HASH FILES
# ------------------------------------------------------------

for path in project_files:

    try:

        stat = path.stat()

        record = {
            "relative_path": str(
                path.relative_to(PROJECT_ROOT)
            ),
            "size_bytes": stat.st_size,
            "modified_time_utc": datetime.fromtimestamp(
                stat.st_mtime,
                timezone.utc
            ).isoformat(),
            "file_state": (
                "EMPTY" if stat.st_size == 0 else "PRESENT"
            ),
            "sha256": None,
            "hash_status": "PENDING"
        }

        try:
            record["sha256"] = sha256_file(path)
            record["hash_status"] = "PASS"

        except Exception as exc:
            record["hash_status"] = (
                f"HASH_ERROR: {type(exc).__name__}: {exc}"
            )

        integrity_records.append(record)

    except Exception as exc:

        integrity_records.append({
            "relative_path": str(path),
            "size_bytes": None,
            "modified_time_utc": None,
            "file_state": "STAT_ERROR",
            "sha256": None,
            "hash_status": (
                f"STAT_ERROR: {type(exc).__name__}: {exc}"
            )
        })


# ------------------------------------------------------------
# 5. BUILD AUDIT DATAFRAME
# ------------------------------------------------------------

columns = [
    "relative_path",
    "size_bytes",
    "modified_time_utc",
    "file_state",
    "sha256",
    "hash_status"
]

file_integrity_audit = pd.DataFrame(
    integrity_records,
    columns=columns
)

# ------------------------------------------------------------
# 6. CALCULATE AUDIT STATISTICS
# ------------------------------------------------------------

total_files = len(file_integrity_audit)

empty_files = int(
    (file_integrity_audit["file_state"] == "EMPTY").sum()
)

stat_errors = int(
    (file_integrity_audit["file_state"] == "STAT_ERROR").sum()
)

hash_successes = int(
    (file_integrity_audit["hash_status"] == "PASS").sum()
)

hash_errors = int(
    file_integrity_audit["hash_status"]
    .astype(str)
    .str.startswith(("HASH_ERROR", "STAT_ERROR"))
    .sum()
)

missing_hashes = int(
    file_integrity_audit["sha256"].isna().sum()
)

# ------------------------------------------------------------
# 7. DETERMINE SECTION STATUS
# ------------------------------------------------------------

if total_files == 0:
    section_status = "REVIEW_REQUIRED"

elif stat_errors > 0 or hash_errors > 0 or missing_hashes > 0:
    section_status = "REVIEW_REQUIRED"

elif empty_files > 0:
    section_status = "REVIEW_REQUIRED"

else:
    section_status = "PASS"


# ------------------------------------------------------------
# 8. SAVE AUDIT OUTPUTS
# ------------------------------------------------------------

audit_path = TABLES_DIR / "file_integrity_audit.csv"

summary_path = (
    TABLES_DIR / "file_integrity_audit_summary.json"
)

file_integrity_audit.to_csv(
    audit_path,
    index=False
)

summary = {
    "section": "SECTION_22_FILE_INTEGRITY_AUDIT",
    "execution_time_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "total_files_inventoried": total_files,
    "successful_hashes": hash_successes,
    "empty_files": empty_files,
    "hash_errors": hash_errors,
    "stat_errors": stat_errors,
    "missing_hashes": missing_hashes,
    "hash_algorithm": "SHA-256",
    "section_status": section_status
}

with open(summary_path, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)


# ------------------------------------------------------------
# 9. COMPLETION REPORT
# ------------------------------------------------------------

print("=" * 85)
print("SECTION 22 — FILE INTEGRITY AUDIT")
print("=" * 85)

print(f"Files inventoried  : {total_files}")
print(f"Successful hashes  : {hash_successes}")
print(f"Empty files        : {empty_files}")
print(f"Hash errors        : {hash_errors}")
print(f"Stat errors        : {stat_errors}")
print(f"Missing hashes     : {missing_hashes}")

print("-" * 85)
print("SECTION 22 STATUS:", section_status)

print("\nSaved files:")
print(audit_path)
print(summary_path)

print("=" * 85)
print("SECTION 22 EXECUTION COMPLETED")
print("=" * 85)


SECTION 22 — FILE INTEGRITY AUDIT
Files inventoried  : 1843
Successful hashes  : 1843
Empty files        : 6
Hash errors        : 0
Stat errors        : 0
Missing hashes     : 0
-------------------------------------------------------------------------------------
SECTION 22 STATUS: REVIEW_REQUIRED

Saved files:
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_22/tables/file_integrity_audit.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_22/tables/file_integrity_audit_summary.json
SECTION 22 EXECUTION COMPLETED


In [36]:

# ============================================================
# SECTION 23 — SHA-256 ARTIFACT VERIFICATION
# ============================================================

import hashlib
import pandas as pd
import json

from pathlib import Path
from datetime import datetime, timezone

# ------------------------------------------------------------
# 1. VALIDATE PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path(PROJECT_ROOT).resolve()
NOTEBOOK_ROOT = Path(NOTEBOOK_ROOT).resolve()
TABLES_DIR = Path(TABLES_DIR)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project root not found: {PROJECT_ROOT}"
    )

TABLES_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. SHA-256 FUNCTION
# ------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):

    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for chunk in iter(
            lambda: file.read(chunk_size),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


# ------------------------------------------------------------
# 3. SELECT HASH CANDIDATES
# ------------------------------------------------------------

hash_candidates = [
    Path(p)
    for p in project_files
    if Path(p).is_file()
    and Path(p).resolve() != NOTEBOOK_ROOT
    and NOTEBOOK_ROOT not in Path(p).resolve().parents
]

excluded_files = len(project_files) - len(hash_candidates)

sha256_records = []

# ------------------------------------------------------------
# 4. HASH AND VERIFY EACH FILE
# ------------------------------------------------------------

for path in hash_candidates:

    try:

        before = path.stat()

        digest = sha256_file(path)

        after = path.stat()

        unchanged_during_hash = (
            before.st_size == after.st_size
            and before.st_mtime_ns == after.st_mtime_ns
        )

        hash_length_valid = (
            isinstance(digest, str)
            and len(digest) == 64
            and all(
                character in "0123456789abcdef"
                for character in digest.lower()
            )
        )

        if not hash_length_valid:
            verification_status = "INVALID_HASH"

        elif not unchanged_during_hash:
            verification_status = "CHANGED_DURING_HASH"

        else:
            verification_status = "HASHED"

        sha256_records.append({
            "relative_path": str(
                path.relative_to(PROJECT_ROOT)
            ),
            "size_bytes": after.st_size,
            "sha256": digest,
            "hash_length_valid": hash_length_valid,
            "unchanged_during_hash": unchanged_during_hash,
            "verification_status": verification_status
        })

    except Exception as exc:

        sha256_records.append({
            "relative_path": str(path),
            "size_bytes": None,
            "sha256": None,
            "hash_length_valid": False,
            "unchanged_during_hash": False,
            "verification_status": (
                f"HASH_ERROR: {type(exc).__name__}: {exc}"
            )
        })


# ------------------------------------------------------------
# 5. BUILD AUDIT DATAFRAME
# ------------------------------------------------------------

sha256_audit = pd.DataFrame(
    sha256_records,
    columns=[
        "relative_path",
        "size_bytes",
        "sha256",
        "hash_length_valid",
        "unchanged_during_hash",
        "verification_status"
    ]
)

# ------------------------------------------------------------
# 6. VALIDATION SUMMARY
# ------------------------------------------------------------

total_hashed = len(sha256_audit)

valid_hashes = int(
    sha256_audit["hash_length_valid"].sum()
)

invalid_hashes = int(
    (~sha256_audit["hash_length_valid"]).sum()
)

changed_files = int(
    (~sha256_audit["unchanged_during_hash"]).sum()
)

hash_errors = int(
    sha256_audit["verification_status"]
    .astype(str)
    .str.startswith("HASH_ERROR").sum()
)

successful_hashes = int(
    (sha256_audit["verification_status"] == "HASHED").sum()
)

# ------------------------------------------------------------
# 7. DETERMINE SECTION STATUS
# ------------------------------------------------------------

if total_hashed == 0:
    section_status = "REVIEW_REQUIRED"

elif (
    invalid_hashes > 0
    or changed_files > 0
    or hash_errors > 0
    or successful_hashes != total_hashed
):
    section_status = "REVIEW_REQUIRED"

else:
    section_status = "PASS"


# ------------------------------------------------------------
# 8. SAVE AUDIT FILES
# ------------------------------------------------------------

audit_path = (
    TABLES_DIR / "sha256_artifact_verification.csv"
)

summary_path = (
    TABLES_DIR / "sha256_artifact_verification_summary.json"
)

sha256_audit.to_csv(
    audit_path,
    index=False
)

summary = {
    "section": "SECTION_23_SHA256_ARTIFACT_VERIFICATION",
    "execution_time_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "hash_algorithm": "SHA-256",
    "total_project_files_considered": len(project_files),
    "excluded_notebook_files": excluded_files,
    "files_hashed": total_hashed,
    "successful_hashes": successful_hashes,
    "valid_hashes": valid_hashes,
    "invalid_hashes": invalid_hashes,
    "changed_during_hash": changed_files,
    "hash_errors": hash_errors,
    "historical_manifest_comparison": False,
    "section_status": section_status
}

with open(summary_path, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)


# ------------------------------------------------------------
# 9. COMPLETION REPORT
# ------------------------------------------------------------

print("=" * 85)
print("SECTION 23 — SHA-256 ARTIFACT VERIFICATION")
print("=" * 85)

print("Project files considered :", len(project_files))
print("Excluded Notebook files  :", excluded_files)
print("Files hashed             :", total_hashed)
print("Successful hashes        :", successful_hashes)
print("Valid hashes             :", valid_hashes)
print("Invalid hashes           :", invalid_hashes)
print("Changed during hashing   :", changed_files)
print("Hash errors              :", hash_errors)

print("-" * 85)

print("Historical manifest comparison: NOT PERFORMED")
print("SECTION 23 STATUS:", section_status)

print("\nSaved audit files:")
print(audit_path)
print(summary_path)

print("=" * 85)
print("SECTION 23 EXECUTION COMPLETED")
print("=" * 85)


SECTION 23 — SHA-256 ARTIFACT VERIFICATION
Project files considered : 1843
Excluded Notebook files  : 141
Files hashed             : 1702
Successful hashes        : 1702
Valid hashes             : 1702
Invalid hashes           : 0
Changed during hashing   : 0
Hash errors              : 0
-------------------------------------------------------------------------------------
Historical manifest comparison: NOT PERFORMED
SECTION 23 STATUS: PASS

Saved audit files:
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_22/tables/sha256_artifact_verification.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_22/tables/sha256_artifact_verification_summary.json
SECTION 23 EXECUTION COMPLETED


In [38]:

# ============================================================
# SECTION 24 — PROVENANCE AUDIT
# ============================================================

import pandas as pd
import json

from pathlib import Path
from datetime import datetime, timezone

# ------------------------------------------------------------
# 1. VALIDATE DIRECTORIES
# ------------------------------------------------------------

PROJECT_ROOT = Path(PROJECT_ROOT).resolve()
TABLES_DIR = Path(TABLES_DIR)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project root not found: {PROJECT_ROOT}"
    )

TABLES_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. DISCOVER PROJECT FILES
# ------------------------------------------------------------

all_project_files = [
    path
    for path in PROJECT_ROOT.rglob("*")
    if path.is_file()
]

# Exclude Section 24's own outputs.
excluded_paths = {
    (TABLES_DIR / "provenance_audit.csv").resolve(),
    (TABLES_DIR / "provenance_audit_summary.json").resolve()
}

provenance_candidates = [
    path for path in all_project_files
    if path.resolve() not in excluded_paths
]

# ------------------------------------------------------------
# 3. PROVENANCE INVENTORY
# ------------------------------------------------------------

provenance_records = []

for path in provenance_candidates:

    try:

        stat = path.stat()

        relative_path = str(
            path.relative_to(PROJECT_ROOT)
        )

        notebook_parts = [
            part
            for part in path.relative_to(PROJECT_ROOT).parts
            if part.startswith("notebook_")
        ]

        notebook_source = (
            notebook_parts[0]
            if notebook_parts
            else "UNKNOWN"
        )

        source_notebook_known = (
            notebook_source != "UNKNOWN"
        )

        provenance_records.append({
            "artifact_path": relative_path,
            "file_name": path.name,
            "extension": path.suffix.lower(),
            "size_bytes": stat.st_size,
            "modified_time_utc": datetime.fromtimestamp(
                stat.st_mtime,
                timezone.utc
            ).isoformat(),
            "notebook_source_candidate": notebook_source,
            "source_path_known": True,
            "source_notebook_known": source_notebook_known,
            "source_attribution_verified": False,
            "provenance_status": (
                "PATH_IDENTIFIED_SOURCE_UNVERIFIED"
                if source_notebook_known
                else "SOURCE_NOTEBOOK_UNKNOWN"
            )
        })

    except Exception as exc:

        provenance_records.append({
            "artifact_path": str(path),
            "file_name": path.name,
            "extension": path.suffix.lower(),
            "size_bytes": None,
            "modified_time_utc": None,
            "notebook_source_candidate": "UNKNOWN",
            "source_path_known": False,
            "source_notebook_known": False,
            "source_attribution_verified": False,
            "provenance_status": (
                f"METADATA_ERROR: {type(exc).__name__}: {exc}"
            )
        })


# ------------------------------------------------------------
# 4. BUILD PROVENANCE AUDIT
# ------------------------------------------------------------

provenance_audit = pd.DataFrame(
    provenance_records,
    columns=[
        "artifact_path",
        "file_name",
        "extension",
        "size_bytes",
        "modified_time_utc",
        "notebook_source_candidate",
        "source_path_known",
        "source_notebook_known",
        "source_attribution_verified",
        "provenance_status"
    ]
)

# ------------------------------------------------------------
# 5. VALIDATE PROVENANCE COVERAGE
# ------------------------------------------------------------

total_records = len(provenance_audit)

identified_paths = int(
    provenance_audit["source_path_known"].sum()
)

identified_notebooks = int(
    provenance_audit["source_notebook_known"].sum()
)

unknown_notebooks = (
    total_records - identified_notebooks
)

metadata_errors = int(
    provenance_audit["provenance_status"]
    .astype(str)
    .str.startswith("METADATA_ERROR").sum()
)

verified_source_attributions = int(
    provenance_audit["source_attribution_verified"].sum()
)

# This section inventories candidate provenance.
# It does not claim that source attribution is verified.
if total_records == 0 or metadata_errors > 0:
    section_status = "REVIEW_REQUIRED"
else:
    section_status = "PASS_WITH_UNVERIFIED_ATTRIBUTION"


# ------------------------------------------------------------
# 6. SAVE AUDIT
# ------------------------------------------------------------

audit_path = (
    TABLES_DIR / "provenance_audit.csv"
)

summary_path = (
    TABLES_DIR / "provenance_audit_summary.json"
)

provenance_audit.to_csv(
    audit_path,
    index=False
)

summary = {
    "section": "SECTION_24_PROVENANCE_AUDIT",
    "execution_time_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "total_provenance_records": total_records,
    "identified_paths": identified_paths,
    "notebook_source_candidates": identified_notebooks,
    "unknown_notebook_sources": unknown_notebooks,
    "verified_source_attributions": verified_source_attributions,
    "metadata_errors": metadata_errors,
    "source_attribution_verification_performed": False,
    "section_status": section_status
}

with open(summary_path, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)


# ------------------------------------------------------------
# 7. COMPLETION REPORT
# ------------------------------------------------------------

print("=" * 85)
print("SECTION 24 — PROVENANCE AUDIT")
print("=" * 85)

print("Files inventoried             :", total_records)
print("Paths identified              :", identified_paths)
print("Notebook source candidates    :", identified_notebooks)
print("Unknown notebook sources      :", unknown_notebooks)
print("Verified source attributions  :", verified_source_attributions)
print("Metadata errors               :", metadata_errors)

print("-" * 85)
print("SECTION 24 STATUS:", section_status)

print("\nSaved files:")
print(audit_path)
print(summary_path)

print("=" * 85)
print("SECTION 24 EXECUTION COMPLETED")
print("=" * 85)


SECTION 24 — PROVENANCE AUDIT
Files inventoried             : 1848
Paths identified              : 1848
Notebook source candidates    : 1796
Unknown notebook sources      : 52
Verified source attributions  : 0
Metadata errors               : 0
-------------------------------------------------------------------------------------
SECTION 24 STATUS: PASS_WITH_UNVERIFIED_ATTRIBUTION

Saved files:
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_22/tables/provenance_audit.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_22/tables/provenance_audit_summary.json
SECTION 24 EXECUTION COMPLETED


In [1]:

# ============================================================
# SECTION 25 — FAILURE / EXCEPTION AUDIT
# ============================================================

import pandas as pd
import json
import re

from pathlib import Path
from datetime import datetime, timezone

# ------------------------------------------------------------
# 1. VALIDATE DIRECTORIES
# ------------------------------------------------------------

PROJECT_ROOT = Path(PROJECT_ROOT).resolve()
RESULTS_ROOT = PROJECT_ROOT / "results"
TABLES_DIR = Path(TABLES_DIR)

if not RESULTS_ROOT.exists():
    raise FileNotFoundError(
        f"Results directory not found: {RESULTS_ROOT}"
    )

TABLES_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. DISCOVER FAILURE / EXCEPTION FILES
# ------------------------------------------------------------

failure_patterns = [
    "*failure*.csv",
    "*failed*.csv",
    "*exception*.csv",
    "*error*.csv",
    "*failure*.json",
    "*failed*.json",
    "*exception*.json",
    "*error*.json",
    "*failure*.txt",
    "*exception*.txt",
    "*error*.txt"
]

failure_files = set()

for pattern in failure_patterns:
    failure_files.update(
        RESULTS_ROOT.rglob(pattern)
    )

failure_files = sorted(
    path for path in failure_files
    if path.is_file()
)

# Exclude this section's own audit outputs.
excluded_paths = {
    (TABLES_DIR / "failure_exception_audit.csv").resolve(),
    (TABLES_DIR / "failure_exception_summary.json").resolve()
}

failure_files = [
    path for path in failure_files
    if path.resolve() not in excluded_paths
]

# ------------------------------------------------------------
# 3. INITIALIZE AUDIT RECORDS
# ------------------------------------------------------------

failure_records = []

# ------------------------------------------------------------
# 4. INSPECT FAILURE / EXCEPTION FILES
# ------------------------------------------------------------

for path in failure_files:

    record = {
        "artifact_path": str(
            path.relative_to(PROJECT_ROOT)
        ),
        "file_name": path.name,
        "extension": path.suffix.lower(),
        "size_bytes": None,
        "rows_inspected": None,
        "status_columns": None,
        "failure_count": None,
        "inspection_status": "PENDING",
        "inspection_note": None
    }

    try:

        record["size_bytes"] = path.stat().st_size

        if record["size_bytes"] == 0:

            record["inspection_status"] = "EMPTY_FILE"
            record["inspection_note"] = (
                "File exists but contains no data."
            )

        elif path.suffix.lower() == ".csv":

            # Bounded read for RAM compatibility.
            frame = pd.read_csv(
                path,
                nrows=10000,
                low_memory=False
            )

            record["rows_inspected"] = len(frame)

            status_columns = [
                column
                for column in frame.columns
                if any(
                    word in str(column).lower()
                    for word in [
                        "status",
                        "error",
                        "exception",
                        "failure",
                        "failed"
                    ]
                )
            ]

            record["status_columns"] = "; ".join(
                map(str, status_columns)
            )

            if status_columns:

                failed = frame[status_columns].astype(
                    str
                ).apply(
                    lambda column: column.str.contains(
                        r"\bfail(?:ed|ure)?\b|"
                        r"\berror\b|"
                        r"\bexception\b",
                        case=False,
                        regex=True,
                        na=False
                    )
                ).any(axis=1)

                record["failure_count"] = int(
                    failed.sum()
                )

                record["inspection_status"] = (
                    "INSPECTED_FAILURES_FOUND"
                    if failed.any()
                    else "INSPECTED_NO_FAILURE_STATUS"
                )

            else:

                record["inspection_status"] = (
                    "INSPECTED_NO_STATUS_COLUMNS"
                )

                record["inspection_note"] = (
                    "No recognizable status/error columns."
                )

        elif path.suffix.lower() == ".json":

            with open(
                path,
                "r",
                encoding="utf-8"
            ) as file:
                data = json.load(file)

            record["inspection_status"] = "JSON_PARSED"

            if isinstance(data, dict):

                text_content = json.dumps(
                    data,
                    ensure_ascii=False
                )

                matches = re.findall(
                    r"\bfail(?:ed|ure)?\b|"
                    r"\berror\b|"
                    r"\bexception\b",
                    text_content,
                    flags=re.IGNORECASE
                )

                record["failure_count"] = len(matches)

            elif isinstance(data, list):

                record["rows_inspected"] = len(data)

                text_content = json.dumps(
                    data,
                    ensure_ascii=False
                )

                matches = re.findall(
                    r"\bfail(?:ed|ure)?\b|"
                    r"\berror\b|"
                    r"\bexception\b",
                    text_content,
                    flags=re.IGNORECASE
                )

                record["failure_count"] = len(matches)

            else:

                record["inspection_note"] = (
                    "JSON parsed; root value is not a dictionary or list."
                )

        elif path.suffix.lower() == ".txt":

            # Count matching lines without loading the full log.
            matching_lines = 0
            lines_inspected = 0

            with open(
                path,
                "r",
                encoding="utf-8",
                errors="replace"
            ) as file:

                for line in file:
                    lines_inspected += 1

                    if re.search(
                        r"\bfail(?:ed|ure)?\b|"
                        r"\berror\b|"
                        r"\bexception\b",
                        line,
                        flags=re.IGNORECASE
                    ):
                        matching_lines += 1

            record["rows_inspected"] = lines_inspected
            record["failure_count"] = matching_lines

            record["inspection_status"] = (
                "INSPECTED_FAILURES_FOUND"
                if matching_lines > 0
                else "INSPECTED_NO_FAILURES"
            )

        else:

            record["inspection_status"] = (
                "UNSUPPORTED_FORMAT"
            )

    except Exception as exc:

        record["inspection_status"] = (
            f"READ_ERROR: {type(exc).__name__}: {exc}"
        )

    failure_records.append(record)


# ------------------------------------------------------------
# 5. BUILD FAILURE AUDIT
# ------------------------------------------------------------

failure_audit = pd.DataFrame(
    failure_records,
    columns=[
        "artifact_path",
        "file_name",
        "extension",
        "size_bytes",
        "rows_inspected",
        "status_columns",
        "failure_count",
        "inspection_status",
        "inspection_note"
    ]
)

# ------------------------------------------------------------
# 6. CALCULATE AUDIT SUMMARY
# ------------------------------------------------------------

total_files = len(failure_audit)

read_errors = int(
    failure_audit["inspection_status"]
    .astype(str)
    .str.startswith("READ_ERROR").sum()
)

empty_files = int(
    (failure_audit["inspection_status"] == "EMPTY_FILE").sum()
)

unsupported_files = int(
    (failure_audit["inspection_status"] == "UNSUPPORTED_FORMAT").sum()
)

failure_files_found = int(
    (
        failure_audit["inspection_status"]
        == "INSPECTED_FAILURES_FOUND"
    ).sum()
)

inspected_files = int(
    failure_audit["inspection_status"]
    .astype(str)
    .str.startswith(
        ("INSPECTED", "JSON_PARSED")
    ).sum()
)

# ------------------------------------------------------------
# 7. DETERMINE SECTION STATUS
# ------------------------------------------------------------

if total_files == 0:
    section_status = "PASS_NO_MATCHING_FAILURE_FILES"

elif read_errors > 0 or unsupported_files > 0:
    section_status = "REVIEW_REQUIRED"

else:
    section_status = "PASS_WITH_FAILURE_EVIDENCE"

# Finding failure records does not itself mean the audit failed.
# It means those records must be reviewed and interpreted.


# ------------------------------------------------------------
# 8. SAVE AUDIT OUTPUTS
# ------------------------------------------------------------

audit_path = (
    TABLES_DIR / "failure_exception_audit.csv"
)

summary_path = (
    TABLES_DIR / "failure_exception_summary.json"
)

failure_audit.to_csv(
    audit_path,
    index=False
)

summary = {
    "section": "SECTION_25_FAILURE_EXCEPTION_AUDIT",
    "execution_time_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "failure_exception_files_found": total_files,
    "files_inspected": inspected_files,
    "files_with_failure_evidence": failure_files_found,
    "empty_files": empty_files,
    "read_errors": read_errors,
    "unsupported_files": unsupported_files,
    "inspection_limit_csv_rows": 10000,
    "section_status": section_status
}

with open(
    summary_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        summary,
        file,
        indent=4
    )


# ------------------------------------------------------------
# 9. COMPLETION REPORT
# ------------------------------------------------------------

print("=" * 85)
print("SECTION 25 — FAILURE / EXCEPTION AUDIT")
print("=" * 85)

print("Failure/exception files found :", total_files)
print("Files inspected               :", inspected_files)
print("Files with failure evidence   :", failure_files_found)
print("Empty files                   :", empty_files)
print("Read errors                   :", read_errors)
print("Unsupported files             :", unsupported_files)

print("-" * 85)

print("SECTION 25 STATUS:", section_status)

print("\nSaved files:")
print(audit_path)
print(summary_path)

print("=" * 85)
print("SECTION 25 EXECUTION COMPLETED")
print("=" * 85)


NameError: name 'PROJECT_ROOT' is not defined

In [ ]:

# ============================================================
# SECTION 26 — GENERATE FINAL PROJECT MANIFEST
# ============================================================

final_manifest_records = []

manifest_candidates = [
    p for p in project_files
    if p.is_file()
    and NOTEBOOK_ROOT not in p.parents
]

for path in manifest_candidates:

    try:
        stat = path.stat()

        final_manifest_records.append({
            "relative_path": str(
                path.relative_to(PROJECT_ROOT)
            ),
            "file_name": path.name,
            "extension": path.suffix.lower(),
            "size_bytes": stat.st_size,
            "modified_time_utc": datetime.fromtimestamp(
                stat.st_mtime,
                timezone.utc
            ).isoformat(),
            "sha256": sha256_file(path),
            "hash_verified": True
        })

    except Exception as exc:

        final_manifest_records.append({
            "relative_path": str(path),
            "file_name": path.name,
            "extension": path.suffix.lower(),
            "size_bytes": None,
            "modified_time_utc": None,
            "sha256": None,
            "hash_verified": False,
            "error": str(exc)
        })

final_project_manifest = pd.DataFrame(
    final_manifest_records
)

FINAL_MANIFEST_PATH = (
    MANIFESTS_DIR / "sppgan_final_project_manifest.csv"
)

final_project_manifest.to_csv(
    FINAL_MANIFEST_PATH,
    index=False
)

print("Manifest records:", len(final_project_manifest))
print("Manifest path:", FINAL_MANIFEST_PATH)
print(
    "Hash failures:",
    int((~final_project_manifest["hash_verified"]).sum())
)
print("SECTION 26 STATUS: PASS")


Manifest records: 0
Manifest path: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_22/manifests/sppgan_final_project_manifest.csv


KeyError: 'hash_verified'

In [ ]:

# ============================================================
# SECTION 27 — GENERATE FINAL AUDIT REPORT
# ============================================================

audit_summary_records = [
    {
        "audit": "Notebook 00 artifacts",
        "records": len(notebook_00_audit)
    },
    {
        "audit": "Notebook 01 artifacts",
        "records": len(notebook_01_audit)
    },
    {
        "audit": "Notebook 02 artifacts",
        "records": len(notebook_02_audit)
    },
    {
        "audit": "Notebook 03 artifacts",
        "records": len(notebook_03_audit)
    },
    {
        "audit": "Baseline artifacts",
        "records": len(baseline_audit)
    },
    {
        "audit": "SPP-GAN artifacts",
        "records": len(sppgan_audit)
    },
    {
        "audit": "Synthetic data artifacts",
        "records": len(synthetic_audit)
    },
    {
        "audit": "Evaluation artifacts",
        "records": len(evaluation_audit)
    },
    {
        "audit": "Ablation artifacts",
        "records": len(ablation_audit)
    },
    {
        "audit": "Trade-off artifacts",
        "records": len(tradeoff_audit)
    },
    {
        "audit": "Significance artifacts",
        "records": len(significance_audit)
    },
    {
        "audit": "Publication artifacts",
        "records": len(publication_audit)
    },
    {
        "audit": "Final project manifest",
        "records": len(final_project_manifest)
    }
]

final_audit_report = pd.DataFrame(
    audit_summary_records
)

final_audit_report.to_csv(
    REPORTS_DIR / "sppgan_final_audit_report.csv",
    index=False
)

with open(
    REPORTS_DIR / "sppgan_final_audit_report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        audit_summary_records,
        f,
        indent=4
    )

print(final_audit_report.to_string(index=False))
print("SECTION 27 STATUS: PASS")


                   audit  records
   Notebook 00 artifacts        0
   Notebook 01 artifacts        0
   Notebook 02 artifacts        0
   Notebook 03 artifacts        0
      Baseline artifacts        0
       SPP-GAN artifacts        0
Synthetic data artifacts        0
    Evaluation artifacts        0
      Ablation artifacts        0
     Trade-off artifacts        0
  Significance artifacts        0
   Publication artifacts        0
  Final project manifest        0
SECTION 27 STATUS: PASS


In [ ]:

# ============================================================
# SECTION 28 — GENERATE REPRODUCIBILITY REPORT
# ============================================================

reproducibility_report = {
    "project": "SPP-GAN",
    "notebook": NOTEBOOK_ID,
    "master_seed": MASTER_SEED,
    "repetition_seeds": REPETITION_SEEDS,
    "expected_repetitions": EXPECTED_REPETITIONS,
    "expected_datasets": EXPECTED_DATASETS,
    "expected_models": EXPECTED_MODELS,
    "expected_ablation_variants": EXPECTED_ABLATION_VARIANTS,
    "data_split": EXPECTED_SPLIT,
    "configuration_path": (
        str(configuration_path)
        if configuration_path else None
    ),
    "configuration_status": configuration_status,
    "python_version": sys.version,
    "platform": platform.platform(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "project_root": str(PROJECT_ROOT),
    "final_manifest_path": str(FINAL_MANIFEST_PATH),
    "generated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "reproducibility_limitations": [
        "Artifact presence does not prove successful experiment execution.",
        "Seed references do not prove that all random operations used those seeds.",
        "Exact reproducibility requires code, data, dependency, hardware and runtime records.",
        "Formal DP requires verified accountant and privacy-boundary evidence."
    ]
}

REPRODUCIBILITY_REPORT_PATH = (
    REPORTS_DIR / "sppgan_reproducibility_report.json"
)

with open(
    REPRODUCIBILITY_REPORT_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        reproducibility_report,
        f,
        indent=4
    )

print("Reproducibility report saved:")
print(REPRODUCIBILITY_REPORT_PATH)
print("SECTION 28 STATUS: PASS")


Reproducibility report saved:
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_22/reports/sppgan_reproducibility_report.json
SECTION 28 STATUS: PASS


In [ ]:

# ============================================================
# SECTION 29 — FINAL PASS / FAIL GATE
# ============================================================

gate_checks = []

def add_gate(name, passed, severity="FAIL"):
    gate_checks.append({
        "check": name,
        "passed": bool(passed),
        "severity": severity
    })

# Core project structure
add_gate(
    "Project root exists",
    PROJECT_ROOT.is_dir()
)

# Configuration is required for a fully reproducible run.
add_gate(
    "Master configuration found",
    configuration_path is not None
)

# Expected dataset and model coverage.
add_gate(
    "All expected datasets have artifacts",
    bool(dataset_coverage_audit["artifact_found"].all()),
    "WARNING"
)

add_gate(
    "All expected models have artifacts",
    bool(model_coverage_audit["artifact_found"].all()),
    "WARNING"
)

# Seed schedule is checked independently of experiment completion.
add_gate(
    "Repetition seed formula consistent",
    bool(seed_audit["seed_matches_formula"].all())
)

# Integrity verification.
add_gate(
    "All manifest hashes valid",
    bool(final_project_manifest["hash_verified"].all())
)

# Previous publication outputs.
publication_manifest_path = (
    NOTEBOOK_21 / "manifests" /
    "notebook_21_publication_manifest.csv"
)

add_gate(
    "Notebook 21 publication manifest exists",
    publication_manifest_path.is_file()
)

# Evidence completeness is a warning, not a technical pass.
master_results_path = (
    PROJECT_ROOT / "results/notebooks/notebook_20/results" /
    "notebook_20_master_results.csv"
)

master_measurement_rows = 0

if master_results_path.is_file():
    try:
        master_measurement_rows = sum(
            len(chunk)
            for chunk in pd.read_csv(
                master_results_path,
                chunksize=5000
            )
        )
    except Exception:
        master_measurement_rows = -1

add_gate(
    "Master empirical measurements available",
    master_measurement_rows > 0,
    "WARNING"
)

# Formal DP evidence is not inferred from configuration flags.
formal_dp_fields = (
    privacy_coverage_audit.loc[
        privacy_coverage_audit["privacy_category"] == "formal_dp",
        "fields_found"
    ].iloc[0]
)

formal_dp_evidence_found = (
    bool(formal_dp_fields)
    and "achieved_epsilon" in formal_dp_fields.lower()
    and "accountant" in formal_dp_fields.lower()
)

add_gate(
    "Formal DP evidence fields identified",
    formal_dp_evidence_found,
    "WARNING"
)

final_gate_audit = pd.DataFrame(gate_checks)

technical_failures = int(
    (
        (~final_gate_audit["passed"])
        & (final_gate_audit["severity"] == "FAIL")
    ).sum()
)

warnings_count = int(
    (
        (~final_gate_audit["passed"])
        & (final_gate_audit["severity"] == "WARNING")
    ).sum()
)

if technical_failures > 0:
    FINAL_GATE_STATUS = "FAIL"
elif warnings_count > 0:
    FINAL_GATE_STATUS = "PASS_WITH_WARNINGS"
else:
    FINAL_GATE_STATUS = "PASS"

final_gate_audit.to_csv(
    VALIDATION_DIR / "final_pass_fail_gate.csv",
    index=False
)

print(final_gate_audit.to_string(index=False))
print("Technical failures:", technical_failures)
print("Warnings:", warnings_count)
print("FINAL GATE:", FINAL_GATE_STATUS)

print("SECTION 29 STATUS: PASS")


KeyError: 'hash_verified'

In [ ]:

# ============================================================
# SECTION 30 — COMPLETION SUMMARY
# ============================================================

completion_summary = {
    "notebook": NOTEBOOK_ID,
    "title": NOTEBOOK_NAME,
    "project_root": str(PROJECT_ROOT),
    "final_gate_status": FINAL_GATE_STATUS,
    "technical_failures": technical_failures,
    "warnings": warnings_count,

    "expected_datasets": len(EXPECTED_DATASETS),
    "expected_models": len(EXPECTED_MODELS),
    "expected_repetitions": EXPECTED_REPETITIONS,
    "expected_dataset_repetition_combinations": EXPECTED_EXPERIMENTS,

    "dataset_artifacts_found": int(
        dataset_coverage_audit["artifact_found"].sum()
    ),
    "model_artifacts_found": int(
        model_coverage_audit["artifact_found"].sum()
    ),
    "repetition_seed_checks_passed": int(
        seed_audit["seed_matches_formula"].sum()
    ),

    "synthetic_artifacts": len(synthetic_audit),
    "evaluation_artifacts": len(evaluation_audit),
    "ablation_artifacts": len(ablation_audit),
    "tradeoff_artifacts": len(tradeoff_audit),
    "significance_artifacts": len(significance_audit),
    "publication_artifacts": len(publication_audit),

    "master_measurement_rows": master_measurement_rows,
    "final_manifest_records": len(final_project_manifest),
    "manifest_hash_failures": int(
        (~final_project_manifest["hash_verified"]).sum()
    ),

    "configuration_status": configuration_status,
    "publication_manifest_exists": publication_manifest_path.is_file(),

    "final_manifest_path": str(FINAL_MANIFEST_PATH),
    "final_audit_report_path": str(
        REPORTS_DIR / "sppgan_final_audit_report.csv"
    ),
    "reproducibility_report_path": str(
        REPRODUCIBILITY_REPORT_PATH
    ),
    "final_gate_report_path": str(
        VALIDATION_DIR / "final_pass_fail_gate.csv"
    ),

    "research_evidence_complete": (
        master_measurement_rows > 0
    ),

    "timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat()
}

COMPLETION_PATH = (
    METADATA_DIR / "notebook_22_completion_summary.json"
)

with open(
    COMPLETION_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        completion_summary,
        f,
        indent=4,
        default=str
    )

# Final readable report
print()
print("=" * 80)
print("NOTEBOOK 22 — REPRODUCIBILITY, MANIFEST & FINAL AUDIT")
print("=" * 80)

print("Final gate:", FINAL_GATE_STATUS)
print("Technical failures:", technical_failures)
print("Warnings:", warnings_count)

print("-" * 80)

print("Expected datasets:", len(EXPECTED_DATASETS))
print("Expected models:", len(EXPECTED_MODELS))
print("Expected repetitions:", EXPECTED_REPETITIONS)

print("Dataset artifacts found:", completion_summary[
    "dataset_artifacts_found"
])

print("Model artifacts found:", completion_summary[
    "model_artifacts_found"
])

print("Synthetic artifacts:", len(synthetic_audit))
print("Evaluation artifacts:", len(evaluation_audit))
print("Ablation artifacts:", len(ablation_audit))
print("Trade-off artifacts:", len(tradeoff_audit))
print("Significance artifacts:", len(significance_audit))
print("Publication artifacts:", len(publication_audit))

print("-" * 80)

print("Master measurement rows:", master_measurement_rows)
print("Final manifest records:", len(final_project_manifest))
print("Manifest hash failures:", completion_summary[
    "manifest_hash_failures"
])

print("Configuration:", configuration_status)
print("Research evidence complete:", completion_summary[
    "research_evidence_complete"
])

print("-" * 80)

print("Final manifest:", FINAL_MANIFEST_PATH)
print("Audit report:", REPORTS_DIR)
print("Reproducibility report:", REPRODUCIBILITY_REPORT_PATH)
print("Completion summary:", COMPLETION_PATH)

print("=" * 80)

gc.collect()

print("SECTION 30 STATUS: PASS")


NameError: name 'FINAL_GATE_STATUS' is not defined